In [1]:
import sys
import rqdatac

print("Python：", sys.version)
print("当前 rqdatac：", rqdatac.__version__)

Python： 3.6.10 |Anaconda, Inc.| (default, Jan  7 2020, 21:14:29) 
[GCC 7.3.0]
当前 rqdatac： 3.4.2


In [3]:
import sys

!{sys.executable} -m pip install --user --upgrade --no-cache-dir -i https://pypi.org/simple "rqdatac==3.4.2"

     |████████████████████████████████| 582 kB 492 kB/s eta 0:00:01
  Attempting uninstall: rqdatac
    Found existing installation: rqdatac 3.0.15
    Uninstalling rqdatac-3.0.15:
      Successfully uninstalled rqdatac-3.0.15


# 最大异常日收益率（ABNRETD）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNRETD} = \max_{t} \left| r_{i,t} - \bar{r}_t \right|
$$

其中：

$$
r_{i,t} \text{ 为股票 } i \text{ 在交易日 } t \text{ 的收益率}
$$$$
\bar{r}_t \text{ 为交易日 } t \text{ 的全市场收益率}
$$$$
m \text{ 为交易日窗口，一般取为月度}
$$

## 因子说明

最大异常日收益率（ABNRETD）是每月内所有交易日中，个股收益率与市场收益率之差的绝对值的最大值，属于**极端日收益类因子**。极端收益（无论是大幅上涨还是大幅下跌）的股票，相比收益平稳的股票更能吸引投资者的关注。因子取值越高，表明该股票在当月曾出现过极端的日收益，投资者的关注度也就越高。**然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票**。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度收盘价

In [1]:
# 20日滑动窗口

import numpy as np
import pandas as pd

from rqdatac import all_instruments, get_price, get_trading_dates


def _get_ids(stock_ids=None):
    """默认返回米筐 CS 股票全集。"""
    if stock_ids is not None:
        return list(dict.fromkeys(stock_ids))

    return (
        all_instruments("CS")
        .order_book_id
        .dropna()
        .drop_duplicates()
        .tolist()
    )


def _prepare_dates(start_date, end_date, lookback_days):
    """
    返回：
    target_dates：最终需要输出因子的交易日；
    price_dates：为计算滚动收益而额外向前展开后的价格日期。
    """
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(days=max(lookback_days * 3, 90)),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < lookback_days:
        raise ValueError("历史交易日不足，无法满足滚动窗口。")

    price_dates = calendar[first_loc - lookback_days:last_loc + 1]
    return target_dates, price_dates


def _get_batch_close_volume(batch, price_dates):
    """
    米筐多标的查询不能使用 skip_suspended=True；
    因此下载成交量，并用 volume > 0 排除当日停牌股票。
    """
    raw = get_price(
        order_book_ids=batch,
        start_date=price_dates[0],
        end_date=price_dates[-1],
        frequency="1d",
        fields=["close", "volume"],
        adjust_type="pre",
        skip_suspended=False,
        expect_df=True,
    )

    if raw is None or raw.empty:
        return None, None

    if isinstance(raw.index, pd.MultiIndex):
        close = raw["close"].unstack("order_book_id")
        volume = raw["volume"].unstack("order_book_id")
    else:
        close = raw[["close"]].rename(columns={"close": batch[0]})
        volume = raw[["volume"]].rename(columns={"volume": batch[0]})

    close.index = pd.to_datetime(close.index)
    volume.index = pd.to_datetime(volume.index)

    return (
        close.reindex(index=price_dates, columns=batch),
        volume.reindex(index=price_dates, columns=batch),
    )


def _get_returns(close, volume):
    """仅保留当日实际有成交的股票收益率。"""
    returns = close.pct_change(fill_method=None).iloc[1:]
    traded_today = volume.iloc[1:].gt(0)
    return returns.where(traded_today)


def _stack_factor(factor_wide, factor_name):
    """日期 × 股票宽表转为 date、order_book_id 两层索引。"""
    panel = factor_wide.stack(dropna=True).rename(factor_name).to_frame()
    panel.index = panel.index.set_names(["date", "order_book_id"])
    return panel

def calculate_abnretd_daily(
    start_date,
    end_date,
    window=20,
    candidate_ids=None,
    market_ids=None,
    batch_size=500,
    min_market_stocks=500,
):
    """
    输出日频、多标的 ABNRETD。

    返回
    ----
    DataFrame
        MultiIndex: date, order_book_id
        Column: ABNRETD
    """
    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=window,
    )

    market_ids = _get_ids(market_ids)
    candidate_ids = _get_ids(candidate_ids)

    return_dates = price_dates[1:]

    # 第一遍：计算全市场每日等权收益率 r_bar_t。
    return_sum = pd.Series(0.0, index=return_dates)
    return_count = pd.Series(0, index=return_dates, dtype="int64")

    for start in range(0, len(market_ids), batch_size):
        batch = market_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        return_sum += returns.sum(axis=1, skipna=True)
        return_count += returns.notna().sum(axis=1).astype("int64")

    market_returns = (
        return_sum / return_count.replace(0, np.nan)
    ).where(return_count >= min_market_stocks)

    # 第二遍：逐批计算个股的滚动最大异常收益。
    factor_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        # 每日异常收益 |r_i,t - r_bar_t|
        abnormal_returns = returns.sub(market_returns, axis=0).abs()

        # 每个交易日均使用过去 window 个交易日滚动计算。
        factor_wide = abnormal_returns.rolling(
            window=window,
            min_periods=window,
        ).max()

        factor_wide = factor_wide.reindex(target_dates)
        factor_parts.append(_stack_factor(factor_wide, "ABNRETD"))

    if not factor_parts:
        return pd.DataFrame(columns=["ABNRETD"])

    return pd.concat(factor_parts).sort_index()

abnretd_panel = calculate_abnretd_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    window=20,
)

abnretd_panel.head()

ABNRETD
date       order_book_id          
2020-01-02 000001.XSHE    0.023944
           000002.XSHE    0.036445
           000004.XSHE    0.070498
           000005.XSHE    0.016283
           000006.XSHE    0.020324

In [2]:
from ABNRETD_数据检查 import check_abnretd_output

summary, by_date, suspicious = check_abnretd_output(
    abnretd_panel,
    extreme_threshold=0.30,  # 超过 30% 列为待核查，不自动判错
)

print(summary.to_string(index=False))
print(by_date.tail())
print(suspicious.head(20))

# 它只检查：
# - 是否有重复的 date-stock、缺失值、inf、负值；
# - 总体均值、标准差、1%/99% 分位数、最大最小值；
# - 每日股票覆盖数和每日横截面分布；
# - 大于 30% 的异常值清单，供核对复权、停复牌和错误价格。
# 合理性上，ABNRETD 应始终大于等于 0；0.05 表示过去窗口内最大异常日收益为 5%。超过 30% 不一定错误，但建议逐个检查。该轻量脚本已用合成数据验证。

                 指标            数值                        判断
                观测数  6.760655e+06                          
               交易日数  1.455000e+03                  样本期交易日数量
                股票数  5.374000e+03                 样本期覆盖股票数量
      重复 date-stock  0.000000e+00                      应为 0
                缺失值  0.000000e+00              最终因子面板通常应为 0
 非有限值（inf/-inf/NaN）  0.000000e+00                      应为 0
                 负值  0.000000e+00     应为 0：ABNRETD 是绝对值的最大值
               零值占比  0.000000e+00  通常接近 0；大量为 0 需核对收益率或市场收益
                 均值  5.864736e-02          日收益率单位，0.05 即 5%
                标准差  3.656208e-02              越大表示横截面离散度越高
                最小值  4.696529e-03                   应大于等于 0
              1%分位数  1.377449e-02                    检查低端分布
                中位数  4.895752e-02            典型股票的月内最大异常日收益
             99%分位数  1.957581e-01                    检查高端分布
                最大值  4.877768e+00                 建议核对特别大的值
         > 30% 的观测数  1.540000e+02       

In [3]:
from rqdatac import get_price
import pandas as pd

raw = get_price(
    "000670.XSHE",
    start_date="2022-08-01",
    end_date="2022-09-20",
    frequency="1d",
    fields=["close", "volume"],
    adjust_type="pre",
    skip_suspended=False,
    expect_df=True,
)

raw["daily_return"] = raw["close"].pct_change()
raw["previous_volume"] = raw["volume"].shift(1)

display(raw[
    (raw["daily_return"].abs() > 0.30)
    | ((raw["volume"] > 0) & (raw["previous_volume"] <= 0))
])

,,close,volume,daily_return,previous_volume
order_book_id,date,,,,
000670.XSHE,2022-08-22,13.24,91221334.0,4.884444,0.0


In [4]:
raw

close       volume  daily_return  previous_volume
order_book_id date                                                         
000670.XSHE   2022-08-01   2.25          0.0           NaN              NaN
              2022-08-02   2.25          0.0      0.000000              0.0
              2022-08-03   2.25          0.0      0.000000              0.0
              2022-08-04   2.25          0.0      0.000000              0.0
              2022-08-05   2.25          0.0      0.000000              0.0
              2022-08-08   2.25          0.0      0.000000              0.0
              2022-08-09   2.25          0.0      0.000000              0.0
              2022-08-10   2.25          0.0      0.000000              0.0
              2022-08-11   2.25          0.0      0.000000              0.0
              2022-08-12   2.25          0.0      0.000000              0.0
              2022-08-15   2.25          0.0      0.000000              0.0
              2022-08-16   2.25          0.0      0.000000              0.0
              2022-08-17   2.25          0.0      0.000000              0.0
              2022-08-18   2.25          0.0      0.000000              0.0
              2022-08-19   2.25          0.0      0.000000              0.0
              2022-08-22  13.24   91221334.0      4.884444              0.0
              2022-08-23  11.92  141914034.0     -0.099698       91221334.0
              2022-08-24  10.73    9938200.0     -0.099832      141914034.0
              2022-08-25  11.11  207376846.0      0.035415        9938200.0
              2022-08-26  10.00  160000449.0     -0.099910      207376846.0
              2022-08-29   9.25  118629976.0     -0.075000      160000449.0
              2022-08-30   9.37  116264729.0      0.012973      118629976.0
              2022-08-31   9.42  113489742.0      0.005336      116264729.0
              2022-09-01  10.36  182108547.0      0.099788      113489742.0
              2022-09-02  11.40  144682206.0      0.100386      182108547.0
              2022-09-05  10.43  207522220.0     -0.085088      144682206.0
              2022-09-06  10.05  140754235.0     -0.036433      207522220.0
              2022-09-07   9.76  132627355.0     -0.028856      140754235.0
              2022-09-08   9.13  106454945.0     -0.064549      132627355.0
              2022-09-09   9.10   73003177.0     -0.003286      106454945.0
              2022-09-13   9.32   73464588.0      0.024176       73003177.0
              2022-09-14   9.80  126377966.0      0.051502       73464588.0
              2022-09-15   8.82  109150171.0     -0.100000      126377966.0
              2022-09-16   8.26   83044660.0     -0.063492      109150171.0
              2022-09-19   7.82   66930239.0     -0.053269       83044660.0
              2022-09-20   8.06   59590182.0      0.030691       66930239.0

# CSSD 模型（横截面收益标准差模型）

**用于检验羊群效应的行为金融模型**

$$
\text{CSSD} = \sqrt{\frac{\sum_{i=1}^{N} (r_i - r_m)^2}{N - 1}}
$$

其中：

$$
r_i \text{ 为个股 } i \text{ 的收益率}
$$$$
r_m \text{ 为市场组合的收益率}
$$$$
N \text{ 为样本中的股票数量}
$$

以 CSSD 作为被解释变量，以两个虚拟变量 $ D_t^u $ 和 $ D_t^d $ 作为解释变量，进行如下时序回归：

$$
\text{CSSD}_t = \alpha + \beta_1 D_t^u + \beta_2 D_t^d + \varepsilon_t
$$

其中：

$$
D_t^u = \begin{cases}1, & \text{若市场收益 } r_{m,t} \text{ 高于给定的上涨阈值} \\0, & \text{否则}\end{cases}
$$$$
D_t^d = \begin{cases}1, & \text{若市场收益 } r_{m,t} \text{ 低于给定的下跌阈值} \\0, & \text{否则}\end{cases}
$$

## 模型说明

CSSD 模型对羊群效应的检验，主要依据时序回归中市场极端收益项（即 $ D_t^u $ 和 $ D_t^d $）的回归系数 $ \beta_1 $ 和 $ \beta_2 $ 的符号及显著性。当 $ \beta_1 $ 或 $ \beta_2 $ 显著小于 0 时，表明在市场大幅上涨或大幅下跌的极端行情下，横截面收益标准差（CSSD）显著下降，意味着个股收益趋于一致，投资者的买卖行为趋向同步，从而反映出市场中存在明显的羊群效应。

数据：日度收盘价、日度可交易股票数量

In [14]:
def calculate_cssd_beta_factor(
    start_date,
    end_date,
    factor_type=0,
    regression_window=252,
    upper_threshold=0.03,
    lower_threshold=-0.03,
    market_ids=None,
    batch_size=500,
    min_market_stocks=500,
    min_extreme_days=5,
):
    """
    计算日频滚动 CSSD 羊群效应因子。

    参数
    ----
    factor_type : int
        0：输出 beta1，即极端上涨日虚拟变量 D_up 的滚动回归系数；
        1：输出 beta2，即极端下跌日虚拟变量 D_down 的滚动回归系数。

    regression_window : int
        每日回归使用的历史交易日窗口，例如 252。

    upper_threshold, lower_threshold : float
        市场极端上涨／下跌阈值。例如 0.03 表示单日上涨超过 3%。

    返回
    ----
    DataFrame
        单层日期索引，列名为 CSSD_beta1 或 CSSD_beta2。
        每个 T 日的值仅使用截至 T 日的数据估计，可用于 T+1 日交易。
    """
    if factor_type not in (0, 1):
        raise ValueError("factor_type 只能为 0（beta1）或 1（beta2）。")

    if lower_threshold >= upper_threshold:
        raise ValueError("lower_threshold 必须小于 upper_threshold。")

    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        # 252 个 CSSD 观测值需要额外取 252 个交易日价格
        lookback_days=regression_window,
    )

    market_ids = _get_ids(market_ids)
    return_dates = price_dates[1:]

    # 汇总每日的收益和、收益平方和、有效股票数。
    return_sum = pd.Series(0.0, index=return_dates)
    return_sq_sum = pd.Series(0.0, index=return_dates)
    stock_count = pd.Series(0, index=return_dates, dtype="int64")

    for start in range(0, len(market_ids), batch_size):
        batch = market_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        return_sum += returns.sum(axis=1, skipna=True)
        return_sq_sum += returns.pow(2).sum(axis=1, skipna=True)
        stock_count += returns.notna().sum(axis=1).astype("int64")

    # r_m,t：全市场当日有效股票收益率等权平均。
    market_return = return_sum / stock_count.replace(0, np.nan)

    # CSSD_t^2 = [sum(r_i^2) - N * r_m^2] / (N - 1)
    squared_deviation_sum = (
        return_sq_sum - stock_count * market_return.pow(2)
    ).clip(lower=0)

    cssd = np.sqrt(
        squared_deviation_sum / (stock_count - 1).replace(0, np.nan)
    )

    state = pd.DataFrame({
        "market_return": market_return,
        "CSSD": cssd,
        "N": stock_count,
    })

    # 有效股票太少的交易日不参与回归。
    state.loc[
        state["N"] < min_market_stocks,
        ["market_return", "CSSD"],
    ] = np.nan

    state = state.dropna(subset=["market_return", "CSSD"]).copy()

    # D_up 对应 beta1；D_down 对应 beta2。
    state["D_up"] = (
        state["market_return"] > upper_threshold
    ).astype(float)

    state["D_down"] = (
        state["market_return"] < lower_threshold
    ).astype(float)

    beta_result = pd.DataFrame(
        np.nan,
        index=state.index,
        columns=["CSSD_beta1", "CSSD_beta2"],
    )

    # 每个交易日 T 都仅使用截至 T 的 regression_window 个交易日回归。
    for end_pos in range(regression_window - 1, len(state)):
        sample = state.iloc[
            end_pos - regression_window + 1:end_pos + 1
        ]

        # 若窗口中上涨或下跌极端日过少，系数不具可解释性。
        if (
            sample["D_up"].sum() < min_extreme_days
            or sample["D_down"].sum() < min_extreme_days
        ):
            continue

        y = sample["CSSD"].values
        X = np.column_stack([
            np.ones(len(sample)),
            sample["D_up"].values,
            sample["D_down"].values,
        ])

        # alpha、beta1、beta2
        coefficients = np.linalg.lstsq(X, y, rcond=None)[0]

        beta_result.iloc[end_pos, 0] = coefficients[1]
        beta_result.iloc[end_pos, 1] = coefficients[2]

    # 最终只输出用户要求区间内的日频因子。
    beta_result = beta_result.reindex(target_dates)

    if factor_type == 0:
        return beta_result[["CSSD_beta1"]]
    else:
        return beta_result[["CSSD_beta2"]]

In [15]:
cssd_beta1 = calculate_cssd_beta_factor(
    start_date="2020-01-01",
    end_date="2025-12-31",
    factor_type=0,
    regression_window=252,
    upper_threshold=0.03,
    lower_threshold=-0.03,
)

cssd_beta1.head()

,CSSD_beta1
2020-01-02,-0.000999
2020-01-03,-0.001018
2020-01-06,-0.001042
2020-01-07,-0.001053
2020-01-08,-0.001065


In [16]:
cssd_beta2 = calculate_cssd_beta_factor(
    start_date="2020-01-01",
    end_date="2025-12-31",
    factor_type=1,
    regression_window=252,
    upper_threshold=0.03,
    lower_threshold=-0.03,
)

cssd_beta2.head()

,CSSD_beta2
2020-01-02,0.008837
2020-01-03,0.008818
2020-01-06,0.008794
2020-01-07,0.008782
2020-01-08,0.008771


# CSAD 模型（横截面绝对偏差模型）

**用于检验羊群效应的行为金融模型**

利用个股收益 $r_i$ 和市场收益 $R_m$ 计算股票收益的横截面绝对偏差：

$$
\text{CSAD} = \frac{\sum_{i=1}^{N} |r_i - R_m|}{N}
$$

其中：

* $r_i$ 为个股 $i$ 的收益率；
* $R_m$ 为市场组合的收益率；
* $N$ 为样本中的股票数量。

以 CSAD 作为被解释变量，以市场收益的绝对值和市场收益的平方作为解释变量，进行如下时序回归：

$$
\text{CSAD}_t = \alpha + \beta_1 |R_{mt}| + \beta_2 R_{mt}^2 + \varepsilon_t
$$

## 模型说明

CSAD 模型对羊群效应的检验，主要依据时序回归中市场收益平方项（$R_{mt}^2$）前的回归系数 $\beta_2$ 的符号及显著性。当 $\beta_2$ 显著小于 0 时，表明在市场大幅上涨或大幅下跌的极端行情下，横截面绝对偏差（CSAD）显著下降，意味着个股收益趋于一致，投资者的买卖行为趋向同步，从而反映出市场中存在明显的羊群效应。当 $\beta_1$ 和 $\beta_2$ 均显著小于 0 时，说明市场羊群效应非常显著，投资者在极端行情下的从众行为尤为突出。

数据：日度收盘价、日度可交易股票数量

In [32]:
def calculate_csad_beta_factor(
    start_date,
    end_date,
    factor_type=0,
    regression_window=252,
    market_ids=None,
    batch_size=500,
    min_market_stocks=500,
):
    """
    日频滚动 CSAD 羊群效应因子。

    CSAD_t = mean_i(|r_i,t - R_m,t|)

    回归：
    CSAD_t = alpha + beta1 * |R_m,t| + beta2 * R_m,t^2 + epsilon_t

    参数
    ----
    factor_type : int
        0：输出 beta1；
        1：输出 beta2。

    返回
    ----
    DataFrame
        日期单层索引；
        列名为 CSAD_beta1 或 CSAD_beta2。
    """
    if factor_type not in (0, 1):
        raise ValueError("factor_type 只能为 0（beta1）或 1（beta2）。")

    if regression_window < 20:
        raise ValueError("regression_window 建议不小于 20。")

    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        # 需要一段历史 CSAD 序列，才能在第一个输出日进行滚动回归。
        lookback_days=regression_window,
    )

    market_ids = _get_ids(market_ids)
    return_dates = price_dates[1:]

    # 第一步：计算每日市场等权收益率 R_m,t。
    return_sum = pd.Series(0.0, index=return_dates)
    stock_count = pd.Series(0, index=return_dates, dtype="int64")

    for start in range(0, len(market_ids), batch_size):
        batch = market_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        return_sum += returns.sum(axis=1, skipna=True)
        stock_count += returns.notna().sum(axis=1).astype("int64")

    market_return = (
        return_sum / stock_count.replace(0, np.nan)
    ).where(stock_count >= min_market_stocks)

    # 第二步：计算 CSAD_t。
    absolute_deviation_sum = pd.Series(0.0, index=return_dates)
    absolute_deviation_count = pd.Series(
        0, index=return_dates, dtype="int64"
    )

    for start in range(0, len(market_ids), batch_size):
        batch = market_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        absolute_deviation = returns.sub(
            market_return,
            axis=0,
        ).abs()

        absolute_deviation_sum += absolute_deviation.sum(
            axis=1,
            skipna=True,
        )

        absolute_deviation_count += absolute_deviation.notna().sum(
            axis=1
        ).astype("int64")

    csad = (
        absolute_deviation_sum
        / absolute_deviation_count.replace(0, np.nan)
    )

    state = pd.DataFrame({
        "market_return": market_return,
        "CSAD": csad,
        "N": absolute_deviation_count,
    })

    state.loc[
        state["N"] < min_market_stocks,
        ["market_return", "CSAD"],
    ] = np.nan

    state = state.dropna(subset=["market_return", "CSAD"]).copy()

    # 回归解释变量。
    state["abs_market_return"] = state["market_return"].abs()
    state["market_return_sq"] = state["market_return"].pow(2)

    beta_result = pd.DataFrame(
        np.nan,
        index=state.index,
        columns=["CSAD_beta1", "CSAD_beta2"],
    )

    # 每日仅使用截至当日的历史窗口，避免未来函数。
    for end_pos in range(regression_window - 1, len(state)):
        sample = state.iloc[
            end_pos - regression_window + 1:end_pos + 1
        ]

        y = sample["CSAD"].values

        X = np.column_stack([
            np.ones(len(sample)),
            sample["abs_market_return"].values,
            sample["market_return_sq"].values,
        ])

        if np.linalg.matrix_rank(X) < X.shape[1]:
            continue

        # alpha、beta1、beta2
        coefficients = np.linalg.lstsq(X, y, rcond=None)[0]

        beta_result.iloc[end_pos, 0] = coefficients[1]
        beta_result.iloc[end_pos, 1] = coefficients[2]

    beta_result = beta_result.reindex(target_dates)

    if factor_type == 0:
        return beta_result[["CSAD_beta1"]]

    return beta_result[["CSAD_beta2"]]

In [33]:
csad_beta1 = calculate_csad_beta_factor(
    start_date="2020-01-01",
    end_date="2025-12-31",
    factor_type=0,
    regression_window=252,
)

csad_beta1.head()

,CSAD_beta1
2020-01-02,0.109864
2020-01-03,0.108927
2020-01-06,0.108656
2020-01-07,0.108464
2020-01-08,0.109180


In [34]:
csad_beta2 = calculate_csad_beta_factor(
    start_date="2020-01-01",
    end_date="2025-12-31",
    factor_type=1,
    regression_window=252,
)

csad_beta2.head()

,CSAD_beta2
2020-01-02,0.354029
2020-01-03,0.362598
2020-01-06,0.356873
2020-01-07,0.356285
2020-01-08,0.342649


# CSAD 模型（基于“小型”板块）

**用于构建羊群效应因子的行为金融模型**

① 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_t$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；

② 选取与 $S_t$ 相关系数最大的 9 只股票，连同 $S_t$ 一起构成一个包含 10 只股票的“小型”板块：

$$
S_i,\quad i = 1, 2, \dots, 10
$$

③ 计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - \bar{r}_t \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$\bar{r}_t$ 为板块内所有股票在交易日 $t$ 的平均收益率。

④ 对 CSAD 进行时序标准化，并计算过去 20 个交易日的均值，作为羊群效应因子：

$$
Z_{jt} = \frac{\text{CSAD}_{jt} - \mu_{\text{CSAD}_j}}{\sigma_{\text{CSAD}_j}}
$$$$
f_{jt} = -\frac{1}{20} \sum_{k=1}^{20} Z_{j,t-k+1}
$$

其中，$j$ 表示第 $j$ 个“小型”板块，$\mu_{\text{CSAD}_j}$ 和 $\sigma_{\text{CSAD}_j}$ 分别为该板块 CSAD 在样本期内的均值和标准差，$Z_{jt}$ 为标准化后的 CSAD，$f_{jt}$ 为最终的羊群效应因子。

## 因子逻辑说明

CSAD 利用股票收益在横截面上的分散度来衡量市场羊群效应的强弱。为使因子对个股具备选股能力，该模型将与目标个股走势最为相近的股票组成“小型”板块，借助该小板块上的羊群效应强弱来近似反映该股票近期的关注度变化。

因子值 $f_{jt}$ 越大（由于负号处理，对应近期 CSAD 显著偏低），表明板块近期的羊群效应越强，板块内个股收益趋于一致，该板块受市场关注的程度提升越明显，未来收益预期也越高。

数据：日度收盘价

In [39]:
def _download_returns_wide(order_book_ids, price_dates, batch_size=500):
    """
    分批下载收盘价与成交量，返回“日期 × 股票”的日收益率宽表。

    停牌或无成交日的收益率会保留为 NaN。
    """
    return_parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(
            batch=batch,
            price_dates=price_dates,
        )

        if close is None:
            continue

        return_parts.append(_get_returns(close, volume))

    if not return_parts:
        raise RuntimeError("未取得有效价格数据。")

    returns = pd.concat(return_parts, axis=1)

    return (
        returns
        .reindex(index=price_dates[1:], columns=order_book_ids)
        .sort_index()
    )

def calculate_small_board_csad(
    start_date,
    end_date,
    candidate_ids=None,
    peer_universe_ids=None,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    zscore_window=120,
    smooth_window=20,
    min_group_active=10,
    batch_size=500,
):
    """
    基于“小型”板块的 CSAD 羊群效应因子。

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：SMALL_BOARD_CSAD

    因子值越大：
        表示近期板块 CSAD 越低、股票走势越趋同、羊群效应越强。
    """
    if group_size < 2:
        raise ValueError("group_size 至少为 2。")

    if min_group_active > group_size:
        raise ValueError("min_group_active 不能大于 group_size。")

    if corr_window <= max_inactive_days:
        raise ValueError("corr_window 必须大于 max_inactive_days。")

    # 第一个输出日需要：
    # 1）120 日相关性历史；
    # 2）120 日 CSAD 标准化历史；
    # 3）20 日 Z 值平滑历史；
    # 4）预留一个月，确保存在上一个月末的分组。
    required_lookback = (
        corr_window + zscore_window + smooth_window + 25
    )

    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=required_lookback,
    )

    candidate_ids = _get_ids(candidate_ids)
    peer_universe_ids = _get_ids(peer_universe_ids)

    # 目标股票必须包含在找同伴的股票池中。
    peer_universe_ids = list(dict.fromkeys(
        peer_universe_ids + candidate_ids
    ))

    # 日期 × 股票 的收益率宽表。
    returns = _download_returns_wide(
        order_book_ids=peer_universe_ids,
        price_dates=price_dates,
        batch_size=batch_size,
    )

    work_dates = returns.index
    target_date_set = set(target_dates)

    factor_wide = pd.DataFrame(
        np.nan,
        index=target_dates,
        columns=candidate_ids,
    )

    # 找到每个月最后一个交易日。
    month_end_dates = pd.DatetimeIndex(
        pd.Series(
            work_dates,
            index=work_dates.to_period("M"),
        )
        .groupby(level=0)
        .max()
        .values
    )

    min_corr_observations = corr_window - max_inactive_days

    for rebalance_pos, rebalance_date in enumerate(month_end_dates):
        date_pos = work_dates.get_loc(rebalance_date)

        # 没有足够的 120 日收益率历史时，无法建立小型板块。
        if date_pos + 1 < corr_window:
            continue

        correlation_returns = returns.iloc[
            date_pos - corr_window + 1:date_pos + 1
        ]

        valid_count = correlation_returns.notna().sum(axis=0)

        eligible_ids = valid_count[
            valid_count >= min_corr_observations
        ].index.tolist()

        if len(eligible_ids) < group_size:
            continue

        # 只用近期活跃且窗口内缺失不超过 max_inactive_days 的股票计算相关性。
        corr_matrix = correlation_returns[
            eligible_ids
        ].corr(min_periods=min_corr_observations)

        current_candidates = [
            stock for stock in candidate_ids
            if (
                stock in eligible_ids
                and pd.notna(returns.loc[rebalance_date, stock])
            )
        ]

        # 本月末形成的板块，持续到下个月末前。
        if rebalance_pos < len(month_end_dates) - 1:
            next_rebalance_date = month_end_dates[rebalance_pos + 1]
            effective_dates = work_dates[
                (work_dates >= rebalance_date)
                & (work_dates < next_rebalance_date)
            ]
        else:
            effective_dates = work_dates[
                work_dates >= rebalance_date
            ]

        output_dates = [
            date for date in effective_dates
            if date in target_date_set
        ]

        if not output_dates:
            continue

        for stock in current_candidates:
            peer_corr = (
                corr_matrix[stock]
                .drop(index=stock, errors="ignore")
                .dropna()
                .sort_values(ascending=False)
            )

            peers = peer_corr.head(group_size - 1).index.tolist()

            if len(peers) < group_size - 1:
                continue

            small_board = [stock] + peers
            board_returns = returns[small_board]

            # 每日板块平均收益与 CSAD。
            board_mean_return = board_returns.mean(axis=1, skipna=True)
            active_count = board_returns.notna().sum(axis=1)

            csad = (
                board_returns.sub(board_mean_return, axis=0)
                .abs()
                .mean(axis=1, skipna=True)
            ).where(active_count >= min_group_active)

            # 使用截至当天的滚动均值与标准差标准化。
            rolling_mean = csad.rolling(
                window=zscore_window,
                min_periods=zscore_window,
            ).mean()

            rolling_std = csad.rolling(
                window=zscore_window,
                min_periods=zscore_window,
            ).std(ddof=0)

            zscore = (
                (csad - rolling_mean)
                / rolling_std.replace(0, np.nan)
            )

            # 加负号：CSAD 越低，因子越高，代表羊群越强。
            factor_series = -zscore.rolling(
                window=smooth_window,
                min_periods=smooth_window,
            ).mean()

            factor_wide.loc[
                output_dates, stock
            ] = factor_series.reindex(output_dates).values

    return _stack_factor(factor_wide, "SMALL_BOARD_CSAD")

In [40]:
from rqdatac import index_components

# 沪深300成分股

stock_pool = index_components(
    "000300.XSHG",
    date="2025-12-31",
)

small_board_csad_panel = calculate_small_board_csad(
    start_date="2024-01-01",
    end_date="2025-12-31",
    candidate_ids=stock_pool,
    peer_universe_ids=stock_pool,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    zscore_window=120,
    smooth_window=20,
)

small_board_csad_panel.head()

SMALL_BOARD_CSAD
date       order_book_id                  
2024-01-02 000002.XSHE            0.043948
           000858.XSHE           -0.049080
           601059.XSHG            0.158541
           000999.XSHE           -0.117612
           688981.XSHG            0.227340

> 当前函数中，candidate_ids=None 与 peer_universe_ids=None 就表示使用 _get_ids(None) 返回的全股票池。但全 A 版本会明显更慢：每个月末都要计算约 $N \times N$ 的相关系数矩阵。若当时有 5,000 只可用股票，就约有 2,500 万个相关系数；batch_size 只能减少下载行情时的请求压力，不能降低这一步的平方级计算量。
建议先按年运行，避免 Notebook 长时间占用内存.


```python
all_a_parts = []

for year in [2024, 2025]:
    factor_part = calculate_small_board_csad(
        start_date="{}-01-01".format(year),
        end_date="{}-12-31".format(year),
        candidate_ids=None,
        peer_universe_ids=None,
        corr_window=120,
        max_inactive_days=20,
        group_size=10,
        zscore_window=120,
        smooth_window=20,
        min_group_active=10,
        batch_size=500,
    )

    all_a_parts.append(factor_part)

small_board_csad_all_a = pd.concat(all_a_parts).sort_index()
```

这里每年单独运行仍然正确：函数会自动额外下载足够的历史交易日，用于相关性、CSAD 标准化和 20 日平滑窗口。

In [ ]:
# 计算量非常大

small_board_csad_all_a = calculate_small_board_csad(
    start_date="2025-01-01",
    end_date="2025-12-31",

    candidate_ids=None,       # 全 A 目标股票
    peer_universe_ids=None,   # 每只股票都在全 A 中寻找 9 个最高相关同伴

    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    zscore_window=120,
    smooth_window=20,
    min_group_active=10,
    batch_size=500,
)

# CSAD 模型（基于板块中地位）

**用于构建羊群效应因子的行为金融模型**

① 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_*$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；

② 选取与 $S_*$ 相关系数最大的 9 只股票，连同 $S_*$ 一起构成一个包含 10 只股票的“小型”板块：

$$
S_i, \quad i = 1, 2, \dots, 10
$$

③ 以 $S_*$ 为中心股票，计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - r_{*,t} \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$r_{*,t}$ 为中心股票 $S_*$ 在交易日 $t$ 的收益率。

④ 对 CSAD 进行时序标准化，并计算过去 20 个交易日的均值，作为羊群效应因子：

$$
Z_{jt} = \frac{\text{CSAD}_{jt} - \mu_{\text{CSAD}_j}}{\sigma_{\text{CSAD}_j}}
$$$$
f_{jt} = -\frac{1}{20} \sum_{k=1}^{20} Z_{j,t-k+1}
$$

其中，$j$ 表示第 $j$ 个“小型”板块，$\mu_{\text{CSAD}_j}$ 和 $\sigma_{\text{CSAD}_j}$ 分别为该板块 CSAD 在样本期内的均值和标准差，$Z_{jt}$ 为标准化后的 CSAD，$f_{jt}$ 为最终的羊群效应因子。

## 因子逻辑说明

CSAD 利用股票收益在横截面上的分散度来衡量市场羊群效应的强弱。为使因子对个股具备选股能力，该模型将与目标个股走势最为相近的股票组成“小型”板块，借助该小板块上的羊群效应强弱来近似反映该股票近期的关注度变化。

与基于板块均值的 CSAD 模型不同，本模型以中心股票 $S_*$ 的收益为参照基准，计算板块内其他股票相对于该中心股票的偏离程度。因子值 $f_{jt}$ 越大（由于负号处理，对应近期 CSAD 显著偏低），表明中心股票收益越接近板块内其他股票的中位数水平，板块近期的羊群效应越强，该板块受市场关注的程度提升越明显，未来收益预期也越高。

数据：日度收盘价

In [51]:
def _calculate_center_board_csad(
    board_returns,
    center_stock,
    min_group_active=10,
):
    """
    以中心股票收益为参照计算板块 CSAD。

    CSAD_t = mean_i(|r_i,t - r_center,t|)

    中心股票自身的偏差为 0，因此在 10 只股票都有效时，
    分母自然就是原公式中的 10。
    """
    center_return = board_returns[center_stock]

    active_count = board_returns.notna().sum(axis=1)

    csad = (
        board_returns.sub(center_return, axis=0)
        .abs()
        .mean(axis=1, skipna=True)
    )

    return csad.where(active_count >= min_group_active)

def calculate_center_board_csad(
    start_date,
    end_date,
    candidate_ids=None,
    peer_universe_ids=None,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    zscore_window=120,
    smooth_window=20,
    min_group_active=10,
    batch_size=500,
):
    """
    基于“小型”板块的 CSAD 羊群效应因子。

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：SMALL_BOARD_CSAD

    因子值越大：
        表示近期板块 CSAD 越低、股票走势越趋同、羊群效应越强。
    """
    if group_size < 2:
        raise ValueError("group_size 至少为 2。")

    if min_group_active > group_size:
        raise ValueError("min_group_active 不能大于 group_size。")

    if corr_window <= max_inactive_days:
        raise ValueError("corr_window 必须大于 max_inactive_days。")

    # 第一个输出日需要：
    # 1）120 日相关性历史；
    # 2）120 日 CSAD 标准化历史；
    # 3）20 日 Z 值平滑历史；
    # 4）预留一个月，确保存在上一个月末的分组。
    required_lookback = (
        corr_window + zscore_window + smooth_window + 25
    )

    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=required_lookback,
    )

    candidate_ids = _get_ids(candidate_ids)
    peer_universe_ids = _get_ids(peer_universe_ids)

    # 目标股票必须包含在找同伴的股票池中。
    peer_universe_ids = list(dict.fromkeys(
        peer_universe_ids + candidate_ids
    ))

    # 日期 × 股票 的收益率宽表。
    returns = _download_returns_wide(
        order_book_ids=peer_universe_ids,
        price_dates=price_dates,
        batch_size=batch_size,
    )

    work_dates = returns.index
    target_date_set = set(target_dates)

    factor_wide = pd.DataFrame(
        np.nan,
        index=target_dates,
        columns=candidate_ids,
    )

    # 找到每个月最后一个交易日。
    month_end_dates = pd.DatetimeIndex(
        pd.Series(
            work_dates,
            index=work_dates.to_period("M"),
        )
        .groupby(level=0)
        .max()
        .values
    )

    min_corr_observations = corr_window - max_inactive_days

    for rebalance_pos, rebalance_date in enumerate(month_end_dates):
        date_pos = work_dates.get_loc(rebalance_date)

        # 没有足够的 120 日收益率历史时，无法建立小型板块。
        if date_pos + 1 < corr_window:
            continue

        correlation_returns = returns.iloc[
            date_pos - corr_window + 1:date_pos + 1
        ]

        valid_count = correlation_returns.notna().sum(axis=0)

        eligible_ids = valid_count[
            valid_count >= min_corr_observations
        ].index.tolist()

        if len(eligible_ids) < group_size:
            continue

        # 只用近期活跃且窗口内缺失不超过 max_inactive_days 的股票计算相关性。
        corr_matrix = correlation_returns[
            eligible_ids
        ].corr(min_periods=min_corr_observations)

        current_candidates = [
            stock for stock in candidate_ids
            if (
                stock in eligible_ids
                and pd.notna(returns.loc[rebalance_date, stock])
            )
        ]

        # 本月末形成的板块，持续到下个月末前。
        if rebalance_pos < len(month_end_dates) - 1:
            next_rebalance_date = month_end_dates[rebalance_pos + 1]
            effective_dates = work_dates[
                (work_dates >= rebalance_date)
                & (work_dates < next_rebalance_date)
            ]
        else:
            effective_dates = work_dates[
                work_dates >= rebalance_date
            ]

        output_dates = [
            date for date in effective_dates
            if date in target_date_set
        ]

        if not output_dates:
            continue

        for stock in current_candidates:
            peer_corr = (
                corr_matrix[stock]
                .drop(index=stock, errors="ignore")
                .dropna()
                .sort_values(ascending=False)
            )

            peers = peer_corr.head(group_size - 1).index.tolist()

            if len(peers) < group_size - 1:
                continue

            small_board = [stock] + peers
            board_returns = returns[small_board]

            # 以目标股票 stock 为中心收益基准计算 CSAD。
            csad = _calculate_center_board_csad(
                board_returns=board_returns,
                center_stock=stock,
                min_group_active=min_group_active,
            )

            # 使用截至当天的滚动均值与标准差标准化。
            rolling_mean = csad.rolling(
                window=zscore_window,
                min_periods=zscore_window,
            ).mean()

            rolling_std = csad.rolling(
                window=zscore_window,
                min_periods=zscore_window,
            ).std(ddof=0)

            zscore = (
                (csad - rolling_mean)
                / rolling_std.replace(0, np.nan)
            )

            # 加负号：CSAD 越低，因子越高，代表羊群越强。
            factor_series = -zscore.rolling(
                window=smooth_window,
                min_periods=smooth_window,
            ).mean()

            factor_wide.loc[
                output_dates, stock
            ] = factor_series.reindex(output_dates).values

    return _stack_factor(factor_wide, "CENTER_BOARD_CSAD")

In [52]:
# 沪深300成分股

stock_pool = index_components(
    "000300.XSHG",
    date="2025-12-31",
)

center_board_csad_panel = calculate_center_board_csad(
    start_date="2024-01-01",
    end_date="2025-12-31",
    candidate_ids=stock_pool,
    peer_universe_ids=stock_pool,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    zscore_window=120,
    smooth_window=20,
    min_group_active=10,
)

center_board_csad_panel.head()

# CENTER_BOARD_CSAD 越高，表示目标股票与其 9 个高相关同伴的近期收益偏离越低，即该股票越处于板块共同交易节奏的中心。

CENTER_BOARD_CSAD
date       order_book_id                   
2024-01-02 000002.XSHE             0.108740
           000858.XSHE             0.036800
           601059.XSHG            -0.024649
           000999.XSHE            -0.200170
           688981.XSHG             0.116177

# CSAD 模型（基于时序回归变形）

**用于构建羊群效应因子的行为金融模型**

* 每个月末，提取当前可交易股票过去 120 个交易日的涨跌幅数据，计算目标股票 $S_*$ 与其他股票（不活跃交易日不超过 20 天）之间的日涨跌幅相关系数；
  
* 选取与 $S_*$ 相关系数最大的 9 只股票，连同 $S_*$ 一起构成一个包含 10 只股票的"小型"板块：
  

$$
S_i, \quad i = 1, 2, \dots, 10
$$

* 以 $S_*$ 为中心股票，计算该板块过去 120 个交易日的横截面绝对偏差（CSAD）：

$$
\text{CSAD}_t = \frac{1}{10} \sum_{i=1}^{10} \left| r_{i,t} - r_{*,t} \right|
$$

其中，$r_{i,t}$ 为板块内第 $i$ 只股票在交易日 $t$ 的收益率，$r_{*,t}$ 为中心股票 $S_*$ 在交易日 $t$ 的收益率。

* 以 CSAD 波动率相对于个股波动率的比值变化作为羊群效应因子，分别计算短期比值和长期比值：

$$
\text{ratio}[t_1, t_2] = \frac{\sigma[t_1, t_2](\text{CSAD})}{\sigma[t_1, t_2](r_*)}
$$

其中，$\sigma[t_1, t_2](\text{CSAD})$ 为 CSAD 在时间段 $[t_1, t_2]$ 内的波动率（标准差），$\sigma[t_1, t_2](r_*)$ 为中心股票收益率在同一时间段内的波动率。

最终的羊群效应因子定义为近期比值与长期比值的比率：

$$
f_{jt} = \frac{\text{ratio}[t - 19, t]}{\text{ratio}[t - 119, t]}
$$

其中，分子 $\text{ratio}[t - 19, t]$ 为过去 20 个交易日的短期 CSAD 相对波动率比值，分母 $\text{ratio}[t - 119, t]$ 为过去 120 个交易日的长期 CSAD 相对波动率比值。

## 因子逻辑说明

标准的 CSAD 时序回归模型通过回归系数 $\beta_2$ 的显著性来检验羊群效应（$\beta_2$ 显著为负表明羊群效应存在），核心逻辑是：当羊群效应存在时，股票收益率的分散度（CSAD）随市场极端行情而减小的幅度，大于市场收益水平本身的变化幅度。

考虑到时序长度不足和滞后误差等因素的影响，本模型对标准时序回归方法进行了变形处理。通过比较 CSAD 波动率与中心股票波动率的比值在短期和长期的变化情况，避免了直接进行时序回归可能带来的估计偏差。当 $f_{jt} < 1$ 时，表明近期 CSAD 的相对波动率较长期水平有所下降，即板块内个股收益趋于一致，羊群效应增强，该板块受关注程度提升；反之，当 $f_{jt} > 1$ 时，表明羊群效应减弱。

* * *

In [5]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_price,
    get_trading_dates,
)


def _csad_ratio_ids(stock_ids):
    if stock_ids is None:
        return (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

    if isinstance(stock_ids, str):
        return [stock_ids]

    return list(dict.fromkeys(stock_ids))


def _csad_ratio_get_returns_wide(
    order_book_ids,
    price_dates,
    batch_size=500,
):
    """下载复权收盘价并生成 日收益率宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_price(
            order_book_ids=batch,
            start_date=price_dates[0],
            end_date=price_dates[-1],
            frequency="1d",
            fields=["close", "volume"],
            adjust_type="pre",
            skip_suspended=False,
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            close = raw["close"].unstack("order_book_id")
            volume = raw["volume"].unstack("order_book_id")
        else:
            close = raw[["close"]].rename(columns={"close": batch[0]})
            volume = raw[["volume"]].rename(columns={"volume": batch[0]})

        close.index = pd.to_datetime(close.index)
        volume.index = pd.to_datetime(volume.index)

        close = close.reindex(index=price_dates, columns=batch)
        volume = volume.reindex(index=price_dates, columns=batch)

        stock_return = close.pct_change(fill_method=None).iloc[1:]

        # 停牌日不将“价格未变”当作真实零收益
        stock_return = stock_return.where(volume.iloc[1:].gt(0))
        parts.append(stock_return)

    if not parts:
        raise RuntimeError("未取得有效价格数据。")

    returns = pd.concat(parts, axis=1)
    returns = returns.loc[:, ~returns.columns.duplicated()]

    return returns.reindex(
        index=price_dates[1:],
        columns=order_book_ids,
    )


def _csad_ratio_month_end_dates(dates):
    """取每个月的最后一个交易日。"""
    month_index = pd.Index(dates.to_period("M"))

    return dates[
        ~month_index.duplicated(keep="last")
    ]


def _csad_ratio_select_boards(
    returns,
    rebalance_date,
    candidate_ids,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
):
    """
    在月末为每个中心股票选择 group_size - 1 个最高相关股票。
    """
    history = returns.loc[:rebalance_date].tail(corr_window)
    min_observations = corr_window - max_inactive_days

    # 同时要求：过去窗口足够活跃，且月末当日可交易
    eligible = (
        history.notna().sum(axis=0) >= min_observations
    )
    eligible = eligible & history.loc[rebalance_date].notna()

    eligible_ids = eligible[eligible].index.tolist()

    if len(eligible_ids) < group_size:
        return {}

    corr_matrix = history[eligible_ids].corr(
        min_periods=min_observations
    )

    boards = {}

    for stock in candidate_ids:
        if stock not in corr_matrix.columns:
            continue

        correlations = corr_matrix[stock].dropna()
        correlations = correlations[
            correlations.index != stock
        ]

        peers = correlations.nlargest(group_size - 1).index.tolist()

        if len(peers) == group_size - 1:
            boards[stock] = [stock] + peers

    return boards


def _csad_ratio_for_one_board(
    board_returns,
    center_stock,
    short_window=20,
    long_window=120,
    min_group_active=10,
):
    """
    计算固定小型板块的：
    ratio_short / ratio_long
    """
    center_return = board_returns[center_stock]

    active_count = board_returns.notna().sum(axis=1)

    csad = (
        board_returns.sub(center_return, axis=0)
        .abs()
        .mean(axis=1, skipna=True)
        .where(active_count >= min_group_active)
    )

    short_ratio = (
        csad.rolling(
            short_window,
            min_periods=short_window,
        ).std(ddof=1)
        / center_return.rolling(
            short_window,
            min_periods=short_window,
        ).std(ddof=1)
    )

    long_ratio = (
        csad.rolling(
            long_window,
            min_periods=long_window,
        ).std(ddof=1)
        / center_return.rolling(
            long_window,
            min_periods=long_window,
        ).std(ddof=1)
    )

    factor = short_ratio / long_ratio

    return factor.replace([np.inf, -np.inf], np.nan)


def calculate_center_board_csad_ratio_daily(
    start_date,
    end_date,
    candidate_ids=None,
    peer_universe_ids=None,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    short_window=20,
    long_window=120,
    min_group_active=10,
    batch_size=500,
):
    """
    CSAD 模型（基于时序回归变形）

    每月末：
    1. 为每只中心股票选取 9 只最高相关股票；
    2. 形成中心股票 + 9 个同伴的小型板块；
    3. 计算 CSAD 相对中心股收益率的短期、长期波动率比值；
    4. 输出 short_ratio / long_ratio。

    输出：
        index  : [date, order_book_id]
        column : CSAD_TS_RATIO
    """
    if group_size < 2:
        raise ValueError("group_size 至少应为 2。")

    if min_group_active > group_size:
        raise ValueError(
            "min_group_active 不能大于 group_size。"
        )

    if short_window > long_window:
        raise ValueError(
            "short_window 不应大于 long_window。"
        )

    candidate_ids = _csad_ratio_ids(candidate_ids)
    peer_universe_ids = _csad_ratio_ids(peer_universe_ids)

    # 确保中心股票也在相关性候选池中
    all_ids = list(
        dict.fromkeys(candidate_ids + peer_universe_ids)
    )

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    # 预留相关性窗口、长期波动率窗口及上一个月末的空间
    required_history = max(corr_window, long_window) + 25

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max(required_history * 3, 365)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < required_history:
        raise ValueError("历史交易日不足，无法构建小型板块。")

    price_dates = calendar[
        first_loc - required_history:last_loc + 1
    ]

    returns = _csad_ratio_get_returns_wide(
        order_book_ids=all_ids,
        price_dates=price_dates,
        batch_size=batch_size,
    )

    # 找到开始日之前最近一个可用月末，
    # 使开始日可沿用该月末建立的小型板块。
    all_month_ends = _csad_ratio_month_end_dates(
        returns.index
    )

    valid_month_ends = []

    for date in all_month_ends:
        if returns.index.get_loc(date) >= corr_window - 1:
            valid_month_ends.append(date)

    valid_month_ends = pd.DatetimeIndex(valid_month_ends)
    valid_month_ends = valid_month_ends[
        valid_month_ends <= target_dates[-1]
    ]

    prior_rebalance_dates = valid_month_ends[
        valid_month_ends <= target_dates[0]
    ]

    if len(prior_rebalance_dates) == 0:
        raise ValueError(
            "开始日前缺少可用于建组的月末交易日。"
        )

    first_rebalance_date = prior_rebalance_dates[-1]
    rebalance_dates = valid_month_ends[
        valid_month_ends >= first_rebalance_date
    ]

    factor_wide = pd.DataFrame(
        index=target_dates,
        columns=candidate_ids,
        dtype=float,
    )

    # 每次月末建组，并将该组沿用至下一次月末建组前
    for pos, rebalance_date in enumerate(rebalance_dates):
        next_rebalance_date = (
            rebalance_dates[pos + 1]
            if pos + 1 < len(rebalance_dates)
            else None
        )

        if next_rebalance_date is None:
            segment_dates = target_dates[
                target_dates >= rebalance_date
            ]
        else:
            segment_dates = target_dates[
                (target_dates >= rebalance_date)
                & (target_dates < next_rebalance_date)
            ]

        if len(segment_dates) == 0:
            continue

        boards = _csad_ratio_select_boards(
            returns=returns,
            rebalance_date=rebalance_date,
            candidate_ids=candidate_ids,
            corr_window=corr_window,
            max_inactive_days=max_inactive_days,
            group_size=group_size,
        )

        # 需要取出本段之前的 long_window - 1 日，
        # 才能在本段首日计算长期波动率。
        segment_start_loc = returns.index.get_loc(
            segment_dates[0]
        )
        segment_end_loc = returns.index.get_loc(
            segment_dates[-1]
        )

        history_start_loc = max(
            0,
            segment_start_loc - long_window + 1,
        )

        calculation_dates = returns.index[
            history_start_loc:segment_end_loc + 1
        ]

        for stock, board in boards.items():
            board_returns = returns.loc[
                calculation_dates,
                board,
            ]

            factor = _csad_ratio_for_one_board(
                board_returns=board_returns,
                center_stock=stock,
                short_window=short_window,
                long_window=long_window,
                min_group_active=min_group_active,
            )

            factor_wide.loc[segment_dates, stock] = (
                factor.reindex(segment_dates)
            )

    result = (
        factor_wide
        .stack(dropna=True)
        .rename("CSAD_TS_RATIO")
        .to_frame()
    )
    result.index = result.index.set_names(
        ["date", "order_book_id"]
    )

    return result

In [6]:
from rqdatac import index_components

# 沪深300
hs300_ids = index_components(
    "000300.XSHG",
    date="2025-12-31",
)

csad_ts_ratio = calculate_center_board_csad_ratio_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    candidate_ids=hs300_ids,
    peer_universe_ids=hs300_ids,
    corr_window=120,
    max_inactive_days=20,
    group_size=10,
    short_window=20,
    long_window=120,
    min_group_active=10,
    batch_size=500,
)

csad_ts_ratio.head()

CSAD_TS_RATIO
date       order_book_id               
2020-01-02 001965.XSHE         1.033415
           300442.XSHE         0.934040
           600460.XSHG         0.838684
           601985.XSHG         1.150121
           000100.XSHE         1.170229

# 平均异常日收益率（ABNRETAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNRETAVG} = \frac{1}{m} \sum_{t}^{t-m+1} \left( r_{i,t} - \bar{r}_t \right)^2
$$

其中：

* $r_{i,t}$ 为股票 $i$ 在交易日 $t$ 的收益率；
* $\bar{r}_t$ 为交易日 $t$ 的全市场收益率；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

平均异常日收益率（ABNRETAVG）为每月日频收益率与市场收益率差值的均方，属于**极端日收益类因子**。极端收益（无论是大幅上涨还是大幅下跌）的股票，相比收益平稳的股票更能吸引投资者的关注。因子取值越高，表明该股票在当月曾出现过较为极端的日收益偏离，投资者的关注度也就越高。

然而，**投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为**。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度收盘价

In [27]:
def calculate_abnretavg_daily(
    start_date,
    end_date,
    window=20,
    candidate_ids=None,
    market_ids=None,
    batch_size=500,
    min_market_stocks=500,
):
    """
    计算日频滚动平均异常日收益率（ABNRETAVG）。

    ABNRETAVG_i,T =
        mean_{t=T-window+1,...,T}[(r_i,t - r_bar,t)^2]

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：ABNRETAVG
    """
    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=window,
    )

    market_ids = _get_ids(market_ids)
    candidate_ids = _get_ids(candidate_ids)

    return_dates = price_dates[1:]

    # 第一遍：计算全市场每日等权平均收益 r_bar,t。
    return_sum = pd.Series(0.0, index=return_dates)
    return_count = pd.Series(0, index=return_dates, dtype="int64")

    for start in range(0, len(market_ids), batch_size):
        batch = market_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        return_sum += returns.sum(axis=1, skipna=True)
        return_count += returns.notna().sum(axis=1).astype("int64")

    market_returns = (
        return_sum / return_count.replace(0, np.nan)
    ).where(return_count >= min_market_stocks)

    # 第二遍：计算每只股票的滚动平均异常收益平方。
    factor_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        close, volume = _get_batch_close_volume(batch, price_dates)
        if close is None:
            continue

        returns = _get_returns(close, volume)

        abnormal_return_squared = (
            returns.sub(market_returns, axis=0)
            .pow(2)
        )

        # 每日以过去 window 个交易日为窗口。
        factor_wide = abnormal_return_squared.rolling(
            window=window,
            min_periods=window,
        ).mean()

        factor_wide = factor_wide.reindex(target_dates)
        factor_parts.append(
            _stack_factor(factor_wide, "ABNRETAVG")
        )

    if not factor_parts:
        return pd.DataFrame(columns=["ABNRETAVG"])

    return pd.concat(factor_parts).sort_index()

In [28]:
abnretavg_panel = calculate_abnretavg_daily(
    start_date="2025-01-01",
    end_date="2025-12-31",
    window=20,
)

abnretavg_panel.head()

ABNRETAVG
date       order_book_id           
2025-01-02 000001.XSHE     0.000286
           000002.XSHE     0.000243
           000004.XSHE     0.002996
           000006.XSHE     0.000796
           000007.XSHE     0.000390

# 每日最大异常成交量（ABNVOLD）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNVOLD} = \max_{t} \left( \frac{VOL_{i,t}}{\overline{VOL}_{i,t}} \right)
$$

其中：

* $VOL_{i,t}$ 为股票 $i$ 在交易日 $t$ 的成交量；
* $\overline{VOL}_{i,t}$ 为股票 $i$ 在交易日 $t$ 滚动 1 年的平均成交量；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

每日最大异常成交量（ABNVOLD）为日频成交量除以过去一年平均成交量的最大值，属于**极端交易量类因子**。异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票在当月曾出现过异常放量，投资者的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：成交量

In [30]:
def calculate_abnvold_daily(
    start_date,
    end_date,
    short_window=20,
    long_window=250,
    candidate_ids=None,
    batch_size=500,
):
    """
    计算日频滚动每日最大异常成交量（ABNVOLD）。

    AVOL_i,t = VOL_i,t / mean(VOL_i,t-long_window ... VOL_i,t-1)

    ABNVOLD_i,T = max(AVOL_i,t), t 属于过去 short_window 个交易日

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：ABNVOLD
    """
    # 第一个输出日 T：
    # 其短窗口最早日为 T-(short_window-1)，
    # 该日还需要再向前 long_window 天来估计平均成交量。
    required_lookback = long_window + short_window - 1

    target_dates, volume_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=required_lookback,
    )

    candidate_ids = _get_ids(candidate_ids)
    factor_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        # 复用已有辅助函数；这里只使用其中的 volume。
        _, volume = _get_batch_close_volume(batch, volume_dates)

        if volume is None:
            continue

        # 不使用当天成交量作为当天的“正常成交量”基准。
        average_volume = (
            volume.rolling(
                window=long_window,
                min_periods=long_window,
            )
            .mean()
            .shift(1)
        )

        # 停牌或无成交日不计入有效异常成交量观测。
        abnormal_volume = (
            volume.div(average_volume)
            .where(volume.gt(0))
        )

        # 每个 T 日取过去 short_window 日的最大异常成交量。
        factor_wide = abnormal_volume.rolling(
            window=short_window,
            min_periods=short_window,
        ).max()

        factor_wide = factor_wide.reindex(target_dates)

        factor_parts.append(
            _stack_factor(factor_wide, "ABNVOLD")
        )

    if not factor_parts:
        return pd.DataFrame(columns=["ABNVOLD"])

    return pd.concat(factor_parts).sort_index()

In [31]:
abnvold_panel = calculate_abnvold_daily(
    start_date="2025-01-01",
    end_date="2025-12-31",
    short_window=20,  # 月度最大值窗口
    long_window=250,  # 一年平均成交量窗口
)

abnvold_panel.head()

ABNVOLD
date       order_book_id          
2025-01-02 000001.XSHE    1.572230
           000002.XSHE    1.453369
           000004.XSHE    3.053573
           000006.XSHE    1.818985
           000007.XSHE    3.617851

# 每日平均异常成交（ABNVOLAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{ABNVOLAVG} = \frac{1}{m} \sum_{t=1}^{m} \left( \frac{VOL_{i,t}}{\overline{VOL}_{i,t}} \right)
$$

其中：

* $VOL_{i,t}$ 为股票 $i$ 在交易日 $t$ 的成交量；
* $\overline{VOL}_{i,t}$ 为股票 $i$ 在交易日 $t$ 滚动 1 年的平均成交量；
* $m$ 为交易日窗口，一般取为月度。

## 因子说明

每日平均异常成交（ABNVOLAVG）为日频成交量除以过去一年平均成交量的均值，属于**极端交易量类因子**。异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票在当月平均异常成交水平越高，投资者的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度成交量

In [42]:
def calculate_abnvolavg_daily(
    start_date,
    end_date,
    short_window=20,
    long_window=250,
    candidate_ids=None,
    batch_size=500,
):
    """
    计算日频滚动每日平均异常成交量（ABNVOLAVG）。

    abnormal_volume_i,t =
        VOL_i,t / mean(VOL_i,t-long_window ... VOL_i,t-1)

    ABNVOLAVG_i,T =
        mean(abnormal_volume_i,t)，
        t 属于过去 short_window 个交易日

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：ABNVOLAVG
    """
    required_lookback = long_window + short_window - 1

    target_dates, volume_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=required_lookback,
    )

    candidate_ids = _get_ids(candidate_ids)
    factor_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        # 复用公共函数；本因子只使用成交量。
        _, volume = _get_batch_close_volume(batch, volume_dates)

        if volume is None:
            continue

        # 基准成交量只使用当天之前的历史成交量。
        average_volume = (
            volume.rolling(
                window=long_window,
                min_periods=long_window,
            )
            .mean()
            .shift(1)
        )

        # 无成交／停牌日不作为有效异常成交量观测。
        abnormal_volume = (
            volume.div(average_volume)
            .where(volume.gt(0))
        )

        # 每日取过去 short_window 日的平均异常成交量。
        factor_wide = abnormal_volume.rolling(
            window=short_window,
            min_periods=short_window,
        ).mean()

        factor_wide = factor_wide.reindex(target_dates)

        factor_parts.append(
            _stack_factor(factor_wide, "ABNVOLAVG")
        )

    if not factor_parts:
        return pd.DataFrame(columns=["ABNVOLAVG"])

    return pd.concat(factor_parts).sort_index()

In [43]:
abnvolavg_panel = calculate_abnvolavg_daily(
    start_date="2025-01-01",
    end_date="2025-12-31",
    short_window=20,
    long_window=250,
)

abnvolavg_panel.head()

ABNVOLAVG
date       order_book_id           
2025-01-02 000001.XSHE     0.876301
           000002.XSHE     0.659450
           000004.XSHE     1.323732
           000006.XSHE     1.039373
           000007.XSHE     1.498421

# 未实现盈利量（CGO）

**行为金融因子 —— 投资者参考点与盈亏状态**

当前未实现盈利量定义为当前价格相对于参考价格的相对偏离程度：

$$
\text{CGO}_t = \frac{P_t - RP_t}{RP_t}
$$

其中，参考价格 $RP_t$ 采用换手率加权的历史价格进行构造：

$$
RP_t = \frac{1}{k} \sum_{n=1}^{N} V_{t-n} \prod_{\tau=1}^{n-1} \left( 1 - V_{t-\tau} \right) P_{t-n}
$$

各变量定义如下：

* $P_t$ 为股票在 $t$ 时刻的价格；
* $P_{t-n}$ 为股票在 $t-n$ 时刻的价格；
* $V_t$ 为股票在 $t$ 时刻的换手率；
* $V_{t-n}$ 为股票在 $t-n$ 时刻的换手率；
* $\frac{1}{k}$ 为权重归一化参数，确保所有权重之和为 1；
* $N$ 为截断时间长度，通常可取 $N = 40$ 或 $60$ 个交易日。

## 因子说明

未实现盈利量（Capital Gain Overhang，CGO）衡量了当前投资者整体的盈亏水平，其核心逻辑在于通过比较当前价格与参考价格（即投资者的平均持仓成本）来判断市场参与者的盈利或亏损状态。

该因子通常与股票未来收益呈正相关关系：

* 当投资者处于**盈利状态**（$\text{CGO} > 0$）时，由于厌恶风险，他们更倾向于卖出股票以锁定利润，从而形成卖盘压力，导致股价被低估，未来预期收益上升；
* 当投资者处于**亏损状态**（$\text{CGO} < 0$）时，由于偏好风险（即“赌徒心态”或“处置效应”），他们更倾向于继续持有股票而非止损卖出，导致股价被高估，未来预期收益下降。

长期来看，上述现象在 A 股市场是成立的。但在短期内，浮盈股票面临的卖盘压力可能大于其被低估的幅度，从而出现浮盈股票未来表现反而更差的情况。

数据：日度收盘价、日度换手率

In [44]:
from rqdatac import get_turnover_rate


def _get_batch_turnover(batch, dates):
    """
    下载日换手率，返回：日期 × 股票 的宽表。
    字段 today 的单位通常是百分数，例如 0.519 表示 0.519%。
    """
    raw = get_turnover_rate(
        order_book_ids=batch,
        start_date=dates[0],
        end_date=dates[-1],
        fields="today",
        expect_df=True,
    )

    if raw is None or raw.empty:
        return None

    if isinstance(raw.index, pd.MultiIndex):
        turnover = raw["today"].unstack("order_book_id")
    else:
        turnover = raw[["today"]].rename(columns={"today": batch[0]})

    turnover.index = pd.to_datetime(turnover.index)

    return turnover.reindex(index=dates, columns=batch)

def calculate_cgo_daily(
    start_date,
    end_date,
    lookback_window=60,
    candidate_ids=None,
    turnover_is_percent=True,
    batch_size=500,
):
    """
    计算日频未实现盈利量（CGO）。

    RP_t = sum_n[w_t,n * P_t-n] / sum_n[w_t,n]

    w_t,n = V_t-n * product(1 - V_t-tau)
    其中 V 为换手率，后续换手越高，较早成交价格的权重越低。

    CGO_t = (P_t - RP_t) / RP_t

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：CGO
    """
    target_dates, price_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=lookback_window,
    )

    candidate_ids = _get_ids(candidate_ids)
    factor_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        # 前复权收盘价用于保证跨除权日的价格可比。
        close, volume = _get_batch_close_volume(batch, price_dates)
        turnover = _get_batch_turnover(batch, price_dates)

        if close is None or turnover is None:
            continue

        # CGO 公式中的 V 必须是 0～1 的比例。
        if turnover_is_percent:
            turnover = turnover / 100.0

        turnover = turnover.clip(lower=0.0, upper=1.0)

        # 停牌／缺失换手率视为未发生筹码交换。
        turnover = turnover.fillna(0.0)

        weight_sum = pd.DataFrame(
            0.0,
            index=price_dates,
            columns=batch,
        )

        weighted_price_sum = pd.DataFrame(
            0.0,
            index=price_dates,
            columns=batch,
        )

        # survival 表示：从历史成交日至今，筹码仍未换手的概率。
        survival = pd.DataFrame(
            1.0,
            index=price_dates,
            columns=batch,
        )

        for lag in range(1, lookback_window + 1):
            turnover_lag = turnover.shift(lag)
            price_lag = close.shift(lag)

            # w_t,n = V_t-n × Π(1 - V_t-tau)
            weight = turnover_lag * survival

            # 价格缺失时不将该观察计入参考价格。
            weight = weight.where(price_lag.notna(), 0.0)

            weighted_price_sum += weight * price_lag.fillna(0.0)
            weight_sum += weight

            # 为下一期更早的历史价格更新留存概率。
            survival *= (1.0 - turnover_lag)

        reference_price = (
            weighted_price_sum
            / weight_sum.replace(0.0, np.nan)
        )

        cgo = (
            (close - reference_price)
            / reference_price
        )

        # CGO 输出要求：
        # 1）当前日有实际成交；
        # 2）此前拥有完整 lookback_window 日价格历史。
        current_active = volume.gt(0)

        history_complete = (
            close.notna()
            .rolling(
                window=lookback_window,
                min_periods=lookback_window,
            )
            .sum()
            .shift(1)
            .eq(lookback_window)
        )

        cgo = cgo.where(current_active & history_complete)
        cgo = cgo.reindex(target_dates)

        factor_parts.append(_stack_factor(cgo, "CGO"))

    if not factor_parts:
        return pd.DataFrame(columns=["CGO"])

    return pd.concat(factor_parts).sort_index()

In [45]:
cgo_panel = calculate_cgo_daily(
    start_date="2025-01-01",
    end_date="2025-12-31",
    lookback_window=60,   # 可改为 40
    turnover_is_percent=True,
)

cgo_panel.head()

CGO
date       order_book_id          
2025-01-02 000001.XSHE   -0.021218
           000002.XSHE   -0.194660
           000004.XSHE   -0.167956
           000006.XSHE   -0.153405
           000007.XSHE   -0.032613

# 衰减加权月频成交量（ATTN）

**行为金融因子 —— 投资者注意力**

衰减加权月频成交量因子通过对过去 12 个月的成交量按月线性衰减加权求和得到，赋予近期成交量更高的权重：

$$
\text{ATTN}_t = \sum_{m=1}^{11} \frac{12 - m}{\sum_{k=1}^{11} k} \cdot VOL_{i, t-m}
$$

其中，分母为权重归一化常数：

$$
\sum_{k=1}^{11} k = \frac{11 \times 12}{2} = 66
$$

因此，因子可简化为：

$$
\text{ATTN}_t = \sum_{m=1}^{11} \frac{12 - m}{66} \cdot VOL_{i, t-m}
$$

各变量定义如下：

* $VOL_{i, t-m}$ 为股票 $i$ 在 $t-m$ 月的月频成交量（共取过去 11 个月，加上当月共覆盖 12 个月度窗口）；
* $m$ 为月份滞后阶数，从 1 到 11，取值越大，对应的成交量越早，权重越低；
* $\frac{12 - m}{66}$ 为线性衰减权重，权重随 $m$ 增大而递减（即近期权重高、远期权重低）。

## 因子说明

衰减加权月频成交量因子通过对股票历史成交量进行时间衰减加权得到，属于**极端交易量类因子**。

异常成交量是投资者注意力的常用代理指标之一。因子取值越高，表明该股票近期成交量持续活跃，投资者的关注度也就越高。

然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。通常情况下，关注度较高的股票会伴随更多的买入行为。但在 A 股市场中，由于**做多与做空机制不对称**，投资者非理性买入的倾向强于非理性卖出，导致高关注度股票因净买入力量而出现短期价格溢价，这种过度反应往往会在未来引发价格反转。

数据：日度成交量

In [47]:
def calculate_attn_daily(
    start_date,
    end_date,
    monthly_lags=11,
    candidate_ids=None,
    batch_size=500,
):
    """
    计算衰减加权月频成交量（ATTN），并映射为日频因子面板。

    当月因子使用此前已完成月份的月度成交量；
    因此每月首个交易日更新，月内保持不变。

    返回
    ----
    DataFrame
        MultiIndex：date、order_book_id
        Column：ATTN
    """
    if monthly_lags != 11:
        raise ValueError(
            "当前版本严格对应原公式：m=1,...,11，monthly_lags 必须为 11。"
        )

    # 预留约 14 个月交易日，确保首个输出月有完整的 11 个月历史。
    required_lookback = 350

    target_dates, volume_dates = _prepare_dates(
        start_date=start_date,
        end_date=end_date,
        lookback_days=required_lookback,
    )

    candidate_ids = _get_ids(candidate_ids)
    factor_parts = []

    # 原公式权重：11/66、10/66、...、1/66。
    weight_denominator = monthly_lags * (monthly_lags + 1) / 2.0

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        _, volume = _get_batch_close_volume(batch, volume_dates)

        if volume is None:
            continue

        month_index = volume.index.to_period("M")

        # 每月成交量：当月各交易日成交量之和。
        monthly_volume = volume.groupby(month_index).sum()

        # 用于判断该股票在该月是否确实存在交易。
        monthly_trade_days = (
            volume.gt(0)
            .groupby(month_index)
            .sum()
        )

        monthly_attn = pd.DataFrame(
            0.0,
            index=monthly_volume.index,
            columns=batch,
        )

        valid_history = pd.DataFrame(
            True,
            index=monthly_volume.index,
            columns=batch,
        )

        for lag in range(1, monthly_lags + 1):
            weight = (monthly_lags + 1 - lag) / weight_denominator

            monthly_attn += (
                weight * monthly_volume.shift(lag)
            )

            # 过去每一个所需月份均至少有一天实际成交。
            valid_history &= monthly_trade_days.shift(lag).gt(0)

        monthly_attn = monthly_attn.where(valid_history)

        # 将月频因子映射到每个交易日。
        daily_attn = monthly_attn.reindex(month_index)
        daily_attn.index = volume.index

        # 仅在当日实际交易时输出因子值。
        daily_attn = daily_attn.where(volume.gt(0))

        factor_wide = daily_attn.reindex(target_dates)

        factor_parts.append(
            _stack_factor(factor_wide, "ATTN")
        )

    if not factor_parts:
        return pd.DataFrame(columns=["ATTN"])

    return pd.concat(factor_parts).sort_index()

In [48]:
attn_panel = calculate_attn_daily(
    start_date="2025-01-01",
    end_date="2025-12-31",
)

attn_panel.head()

ATTN
date       order_book_id              
2025-01-02 000001.XSHE    2.804395e+09
           000002.XSHE    4.373357e+09
           000004.XSHE    3.014172e+08
           000006.XSHE    9.926508e+08
           000007.XSHE    1.752935e+08

# 平均换手率（TURNAVG）

**行为金融因子 —— 投资者注意力**

$$
\text{TURNAVG} = \frac{1}{m} \sum_{t=1}^{m} \text{TURN}_{i,t}
$$

其中：

* $\text{TURN}_{i,t}$ 为股票 $i$ 在交易日 $t$ 的换手率；
* $m$ 为交易日窗口，一般取为月度（即当月所有交易日）。

## 因子说明

平均换手率（TURNAVG）属于**股票交易活跃度类因子**，通过衡量股票在月度内的平均交易活跃度来反映投资者对该股票的关注程度。股票交易越活跃，说明市场参与者对该股票的买卖意愿越强，投资者对该股票的关注度也就越高。然而，投资者的注意力是有限的，他们更倾向于交易那些引起自己注意的股票。高关注度往往伴随着更多的非理性买入行为，在 A 股市场做多与做空不对称的背景下，这种非理性净买入会推动股价产生短期溢价，进而在未来引发价格反转。

数据：日度换手率

In [53]:
import numpy as np
import pandas as pd
from rqdatac import all_instruments, get_trading_dates, get_turnover_rate


def calculate_turnavg_daily(
    start_date,
    end_date,
    window=20,
    candidate_ids=None,
    batch_size=500,
):
    """
    平均换手率（TURNAVG）

    每个交易日使用截至当日的过去 window 个交易日，计算日换手率均值。
    输出：
        MultiIndex DataFrame
        index  : [date, order_book_id]
        column : TURNAVG
    """
    if window < 1:
        raise ValueError("window 必须为正整数。")

    # 未指定股票池时，使用全部股票
    if candidate_ids is None:
        candidate_ids = (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    else:
        candidate_ids = list(dict.fromkeys(candidate_ids))

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    # 额外取历史交易日，供滚动窗口使用
    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(days=max(window * 3, 90)),
            end_date=end_date,
        )
    )
    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    if first_loc < window - 1:
        raise ValueError("历史交易日不足，无法计算 TURNAVG。")

    fetch_dates = calendar[first_loc - window + 1:]

    # 分批下载日换手率，避免全市场一次性请求过大
    turnover_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        raw = get_turnover_rate(
            order_book_ids=batch,
            start_date=fetch_dates[0],
            end_date=fetch_dates[-1],
            fields="today",
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            turnover = raw["today"].unstack("order_book_id")
        else:
            # 单股票请求时的兼容处理
            turnover = raw[["today"]].rename(columns={"today": batch[0]})

        turnover.index = pd.to_datetime(turnover.index)
        turnover_parts.append(
            turnover.reindex(index=fetch_dates, columns=batch)
        )

    if not turnover_parts:
        raise RuntimeError("未取得有效换手率数据。")

    turnover_wide = pd.concat(turnover_parts, axis=1)
    turnover_wide = turnover_wide.loc[:, ~turnover_wide.columns.duplicated()]
    turnover_wide = turnover_wide.reindex(
        index=fetch_dates,
        columns=candidate_ids,
    )

    # 当日及过去 window 个交易日的平均换手率
    turnavg_wide = turnover_wide.rolling(
        window=window,
        min_periods=window,
    ).mean()

    turnavg_wide = turnavg_wide.reindex(target_dates)

    result = turnavg_wide.stack(dropna=True).rename("TURNAVG").to_frame()
    result.index = result.index.set_names(["date", "order_book_id"])

    return result

In [54]:
turnavg = calculate_turnavg_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    window=20,          # 约一个月的交易日
    candidate_ids=None, # None 表示全 A 股
    batch_size=500,
)

turnavg.head()

TURNAVG
date       order_book_id          
2020-01-02 000001.XSHE    0.390605
           000002.XSHE    0.925120
           000004.XSHE    1.802595
           000005.XSHE    0.747180
           000006.XSHE    0.649565

# 基于 PE 的分析师锚定效应偏差（CAF_EP）

**行为金融因子 —— 分析师预测锚定效应**

$$
\text{CAF_EP}_{i,t} = \frac{\text{FEP}_{i,t} - \text{IFEP}_t}{\text{IFEP}_t}
$$

其中，$\text{FEP}_{i,t}$ 为分析师在时间 $t$ 对股票 $i$ 的 PE 预测值所对应的盈利预测（即 PE 的倒数）：

$$
\text{FEP}_{i,t} = \frac{1}{\text{FPE}_{i,t}}
$$

各变量定义如下：

* $\text{FPE}_{i,t}$ 为分析师在时间 $t$ 对股票 $i$ 的 PE 指标的预测值；
* $\text{FEP}_{i,t}$ 为股票 $i$ 在时间 $t$ 的预期盈利水平（PE 的倒数），用于衡量估值水平的高低；
* $\text{IFEP}_t$ 为股票 $i$ 所属行业中所有成分股的 $\text{FPE}_{i,t}$ 的中位数（即行业中位数 PE 的倒数）。

## 因子说明

**锚定效应**指人们在做出判断时，会将最初接触到的信息（即“锚点”）作为参考依据，并在其基础上进行调整，但调整往往不充分，从而导致判断偏差。在分析师预测场景中，锚定效应表现为：分析师会以公司所在行业的估值水平（行业中位数 PE）作为定价的“锚点”。当某只股票的估值指标超出行业中位数时（即 $\text{CAF_EP}_{i,t} > 0$），分析师会倾向于做出**悲观预测**，将预测值向行业中位数靠拢；反之，当估值低于行业中位数时（即 $\text{CAF_EP}_{i,t} < 0$），分析师会倾向于做出**乐观预测**。财报公布后，实际财务指标将与分析师预测形成反向修正：被悲观预测的公司，其实际财务指标将高于预测值，股价表现更优；被乐观预测的公司，其实际财务指标将低于预测值，股价表现更差。因此，该因子可用于预测未来股价的走势方向。

数据：PE分析师预测值、行业分类数据、PE

In [12]:
import rqdatac

try:
    test = rqdatac.consensus.get_comp_indicators(
        order_book_ids=["000001.XSHE"],
        start_date="2025-01-01",
        end_date="2025-01-10",
        fields=["comp_con_pe_ftm"],
    )
    print(test.head())
except Exception as error:
    print(type(error).__name__)
    print(error)

PermissionDenied
permission denied: get_consensus_comp_indicators


In [1]:
import numpy as np
import pandas as pd
import rqdatac

from rqdatac import (
    all_instruments,
    get_trading_dates,
    get_instrument_industry,
)


def _get_consensus_fpe_wide(
    order_book_ids,
    dates,
    fpe_field="comp_con_pe_ftm",
    report_range=3,
    batch_size=500,
):
    """下载并整理为：日期 × 股票 的一致预期 PE 宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        # 新版 rqdatac 支持 report_range；
        # 旧版接口不支持时，自动降级为默认查询方式。
        try:
            raw = rqdatac.consensus.get_comp_indicators(
                order_book_ids=batch,
                start_date=dates[0],
                end_date=dates[-1],
                fields=[fpe_field],
                report_range=report_range,
            )
        except TypeError as error:
            if "report_range" not in str(error):
                raise

            raw = rqdatac.consensus.get_comp_indicators(
                order_book_ids=batch,
                start_date=dates[0],
                end_date=dates[-1],
                fields=[fpe_field],
            )

        if raw is None or raw.empty:
            continue

        temp = raw.reset_index()

        if "order_book_id" not in temp.columns or "date" not in temp.columns:
            raise ValueError(
                "一致预期数据返回格式异常：未找到 order_book_id 或 date 字段。"
            )

        temp["date"] = pd.to_datetime(temp["date"])
        temp = temp[["date", "order_book_id", fpe_field]]
        parts.append(temp)

    if not parts:
        raise RuntimeError(
            "未取得一致预期 PE 数据。请确认账户已开通一致预期数据权限，"
            "且 comp_con_pe_ftm 字段可用。"
        )

    fpe_long = pd.concat(parts, ignore_index=True)
    fpe_long = fpe_long.drop_duplicates(
        subset=["date", "order_book_id"],
        keep="last",
    )

    fpe_wide = fpe_long.pivot(
        index="date",
        columns="order_book_id",
        values=fpe_field,
    )

    # 负 PE、零 PE 没有正常的预期估值含义
    fpe_wide = fpe_wide.where(fpe_wide > 0)

    # 仅使用过去已经出现的预测值向前填充
    fpe_wide = (
        fpe_wide.reindex(index=dates, columns=order_book_ids)
        .ffill()
    )

    return fpe_wide

def _get_industry_code(
    order_book_ids,
    asof_date,
    industry_source="citics_2019",
    industry_level=1,
    batch_size=500,
):
    """获取指定日期的股票行业代码。"""
    parts = []
    industry_col = {
        1: "first_industry_code",
        2: "second_industry_code",
        3: "third_industry_code",
    }[industry_level]

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_instrument_industry(
            order_book_ids=batch,
            source=industry_source,
            level=industry_level,
            date=asof_date,
        )

        if raw is None or raw.empty:
            continue

        parts.append(raw[industry_col])

    if not parts:
        return pd.Series(index=order_book_ids, dtype=object)

    return (
        pd.concat(parts)
        .loc[lambda x: ~x.index.duplicated(keep="last")]
        .reindex(order_book_ids)
    )


def calculate_caf_ep_daily(
    start_date,
    end_date,
    candidate_ids=None,
    industry_universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_industry_stocks=5,
    initial_lookback_days=370,
    batch_size=500,
):
    """
    基于 PE 的分析师锚定效应偏差（CAF_EP）

    输出：
        index  : [date, order_book_id]
        column : CAF_EP

    参数：
        candidate_ids         输出因子股票池；None 为全 A 股。
        industry_universe_ids 计算行业 PE 中位数所用股票池；None 为全 A 股。
        industry_refresh      行业分类更新频率：'M'（默认）、'Q' 或 'D'。
                              'M' 在每月首个交易日更新，兼顾时点正确性与效率。
    """
    if industry_level not in (1, 2, 3):
        raise ValueError("industry_level 仅支持 1、2、3。")

    if industry_refresh not in ("M", "Q", "D"):
        raise ValueError("industry_refresh 仅支持 'M'、'Q'、'D'。")

    if candidate_ids is None:
        candidate_ids = (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    else:
        candidate_ids = list(dict.fromkeys(candidate_ids))

    if industry_universe_ids is None:
        industry_universe_ids = (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    else:
        industry_universe_ids = list(dict.fromkeys(industry_universe_ids))

    # 保证待输出的股票也位于其行业中位数的计算范围内
    calculation_ids = list(
        dict.fromkeys(industry_universe_ids + candidate_ids)
    )

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    all_dates = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(days=initial_lookback_days),
            end_date=end_date,
        )
    )
    target_dates = all_dates[
        (all_dates >= start_date) & (all_dates <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    # 1. 获取每日一致预期 FPE，并仅以前值填充
    fpe_wide_all = _get_consensus_fpe_wide(
        order_book_ids=calculation_ids,
        dates=all_dates,
        fpe_field="comp_con_pe_ftm",
        report_range=3,
        batch_size=batch_size,
    )
    fpe_wide = fpe_wide_all.reindex(target_dates)

    # 2. 生成行业分类更新时间点
    if industry_refresh == "D":
        refresh_mask = np.ones(len(target_dates), dtype=bool)
    else:
        periods = target_dates.to_period(industry_refresh)
        refresh_mask = ~periods.duplicated()

    refresh_locs = np.flatnonzero(refresh_mask)
    factor_wide = pd.DataFrame(
        index=target_dates,
        columns=candidate_ids,
        dtype=float,
    )

    # 3. 每个行业分类有效区间内，计算行业 FPE 中位数和 CAF_EP
    for pos, start_loc in enumerate(refresh_locs):
        end_loc = (
            refresh_locs[pos + 1]
            if pos + 1 < len(refresh_locs)
            else len(target_dates)
        )
        block_dates = target_dates[start_loc:end_loc]
        industry_date = block_dates[0]

        industry_code = _get_industry_code(
            order_book_ids=calculation_ids,
            asof_date=industry_date,
            industry_source=industry_source,
            industry_level=industry_level,
            batch_size=batch_size,
        )

        valid_ids = industry_code[industry_code.notna()].index.tolist()
        if not valid_ids:
            continue

        fpe_block = fpe_wide.loc[block_dates, valid_ids]
        group_key = industry_code.reindex(valid_ids)

        # 行业内每个交易日的 FPE 中位数与有效股票数
        industry_median_fpe = (
            fpe_block.T.groupby(group_key, sort=False).median().T
        )
        industry_count = (
            fpe_block.T.groupby(group_key, sort=False).count().T
        )

        # 行业内有效预期 PE 股票过少时，不产生因子值
        industry_median_fpe = industry_median_fpe.where(
            industry_count >= min_industry_stocks
        )

        # 将每只股票所在行业的中位数映射回个股
        reference_fpe = industry_median_fpe.reindex(
            columns=group_key.values
        )
        reference_fpe.columns = valid_ids

        # CAF_EP = 行业 FPE 中位数 / 个股 FPE - 1
        block_factor = (
            reference_fpe.reindex(columns=candidate_ids)
            .div(fpe_wide.loc[block_dates, candidate_ids])
            - 1.0
        )

        factor_wide.loc[block_dates, candidate_ids] = block_factor

    result = factor_wide.stack(dropna=True).rename("CAF_EP").to_frame()
    result.index = result.index.set_names(["date", "order_book_id"])

    return result

In [2]:
caf_ep = calculate_caf_ep_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    candidate_ids=None,
    industry_universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_industry_stocks=5,
    batch_size=500,
)

caf_ep.head()

PermissionDenied: permission denied: get_consensus_comp_indicators

## 数据获取（付费接口）

In [ ]:
# 分析师预期数据.py

# 基于异常收益的注意力溢出（SPILL）

**行为金融因子 —— 投资者注意力溢出效应**

首先，计算每月平均异常日收益率作为个股层面的注意力代理指标：

$$
\text{ATTN}_{i,t} = \frac{1}{m} \sum_{t}^{t-m+1} \left( r_{i,t} - \bar{r}_t \right)^2
$$

其中，$r_{i,t}$ 为股票 $i$ 在交易日 $t$ 的收益率，$\bar{r}_t$ 为交易日 $t$ 的全市场收益率，$m$ 为交易日窗口，一般取为月度。进一步，计算当期相近股票的注意力均值与个股注意力之差，即为注意力溢出程度：

$$
\text{SPILL}_{i,t} = \text{ATTN}_{i,t} - \overline{\text{ATTN}}_{g(i),t}
$$

其中，$\overline{\text{ATTN}}_{g(i),t}$ 为股票 $i$ 所属分组 $g(i)$ 内所有股票在 $t$ 期的注意力均值（不含股票 $i$ 自身）。

## 分组标准

相近股票的确定标准如下：

1. 先以中信一级行业作为分类标准，将全市场股票划分为不同行业；
2. 在每个行业内，按市值规模从大到小排序，以 **3:4:3** 的比例将股票分为**大市值、中市值、小市值**三组；
3. 同一行业同一市值组内的股票即被定义为“相近股票”。

## 因子逻辑说明

投资者注意力存在**溢出效应**：当某只股票受到高度关注时，投资者往往会同时注意到与其特征相似（同行业、同市值规模）的“邻居”股票，从而导致这些相近股票也获得一定的关注度提升，并享有注意力溢价。上述因子以异常收益（收益相对于市场的偏离程度）作为注意力的代理指标，通过计算个股注意力与其相近股票平均注意力之间的差值来衡量溢出程度。$\text{SPILL}_{i,t}$ 越大，表明个股当前受到的关注显著高于其同类股票，未来注意力向相近股票溢出的效应越显著，相近股票预期将取得正向超额收益。

数据：中信一级行业、日度市值、日度收盘价

In [59]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_price,
    get_factor,
    get_trading_dates,
    get_instrument_industry,
)


def _spill_get_all_a_ids():
    return (
        all_instruments("CS")["order_book_id"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )


def _spill_get_close_volume(batch, price_dates):
    raw = get_price(
        order_book_ids=batch,
        start_date=price_dates[0],
        end_date=price_dates[-1],
        frequency="1d",
        fields=["close", "volume"],
        adjust_type="pre",
        skip_suspended=False,
        expect_df=True,
    )

    if raw is None or raw.empty:
        return None, None

    if isinstance(raw.index, pd.MultiIndex):
        close = raw["close"].unstack("order_book_id")
        volume = raw["volume"].unstack("order_book_id")
    else:
        close = raw[["close"]].rename(columns={"close": batch[0]})
        volume = raw[["volume"]].rename(columns={"volume": batch[0]})

    close.index = pd.to_datetime(close.index)
    volume.index = pd.to_datetime(volume.index)

    return (
        close.reindex(index=price_dates, columns=batch),
        volume.reindex(index=price_dates, columns=batch),
    )


def _spill_get_market_cap_wide(order_book_ids, dates, batch_size=500):
    """下载市值因子，整理为 日期 × 股票 宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_factor(
            order_book_ids=batch,
            factor="market_cap_3",
            start_date=dates[0],
            end_date=dates[-1],
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            market_cap = raw["market_cap_3"].unstack("order_book_id")
        else:
            market_cap = raw[["market_cap_3"]].rename(
                columns={"market_cap_3": batch[0]}
            )

        market_cap.index = pd.to_datetime(market_cap.index)
        parts.append(
            market_cap.reindex(index=dates, columns=batch)
        )

    if not parts:
        raise RuntimeError("未取得 market_cap_3 市值数据。")

    market_cap_wide = pd.concat(parts, axis=1)
    market_cap_wide = market_cap_wide.loc[
        :, ~market_cap_wide.columns.duplicated()
    ]

    return market_cap_wide.reindex(
        index=dates,
        columns=order_book_ids,
    )


def _spill_get_industry_codes(
    order_book_ids,
    asof_date,
    industry_source="citics_2019",
    industry_level=1,
    batch_size=500,
):
    """获取指定时点的行业代码。"""
    field_map = {
        1: "first_industry_code",
        2: "second_industry_code",
        3: "third_industry_code",
    }
    industry_field = field_map[industry_level]

    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_instrument_industry(
            order_book_ids=batch,
            source=industry_source,
            level=industry_level,
            date=asof_date,
        )

        if raw is not None and not raw.empty:
            parts.append(raw[industry_field])

    if not parts:
        return pd.Series(index=order_book_ids, dtype=object)

    industry_code = pd.concat(parts)
    industry_code = industry_code[
        ~industry_code.index.duplicated(keep="last")
    ]

    return industry_code.reindex(order_book_ids)


def calculate_spill_daily(
    start_date,
    end_date,
    attention_window=20,
    candidate_ids=None,
    market_ids=None,
    peer_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_market_stocks=500,
    min_group_stocks=2,
    batch_size=500,
):
    """
    基于异常收益的注意力溢出（SPILL）

    输出：
        MultiIndex DataFrame
        index  : [date, order_book_id]
        column : SPILL

    参数：
        candidate_ids：输出因子值的股票池；None 为全 A 股。
        market_ids：计算市场平均收益的股票池；None 为全 A 股。
        peer_ids：计算行业-市值组均值的股票池；None 为全 A 股。
        industry_refresh：行业归属更新频率，支持 "M"、"Q"、"D"。
    """
    if attention_window < 1:
        raise ValueError("attention_window 必须为正整数。")

    if industry_level not in (1, 2, 3):
        raise ValueError("industry_level 仅支持 1、2、3。")

    if industry_refresh not in ("M", "Q", "D"):
        raise ValueError("industry_refresh 仅支持 'M'、'Q'、'D'。")

    # 仅在需要时下载全 A 股名单
    all_a_ids = None
    if (
        candidate_ids is None
        or market_ids is None
        or peer_ids is None
    ):
        all_a_ids = _spill_get_all_a_ids()

    candidate_ids = (
        all_a_ids
        if candidate_ids is None
        else list(dict.fromkeys(candidate_ids))
    )
    market_ids = (
        all_a_ids
        if market_ids is None
        else list(dict.fromkeys(market_ids))
    )
    peer_ids = (
        all_a_ids
        if peer_ids is None
        else list(dict.fromkeys(peer_ids))
    )

    # 目标股票也必须进入同组均值的计算范围
    peer_ids = list(dict.fromkeys(peer_ids + candidate_ids))
    price_ids = list(
        dict.fromkeys(candidate_ids + market_ids + peer_ids)
    )

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max((attention_window + 1) * 3, 90)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < attention_window:
        raise ValueError("历史交易日不足，无法计算异常收益滚动均值。")

    # 额外取一天价格，以获得第一个历史收益率
    price_dates = calendar[first_loc - attention_window:last_loc + 1]

    # 1. 下载价格，构造全市场日收益率
    return_parts = []
    market_return_sum = pd.Series(0.0, index=price_dates[1:])
    market_return_count = pd.Series(0, index=price_dates[1:])

    for start in range(0, len(price_ids), batch_size):
        batch = price_ids[start:start + batch_size]

        close, volume = _spill_get_close_volume(
            batch=batch,
            price_dates=price_dates,
        )

        if close is None:
            continue

        stock_return = close.pct_change(fill_method=None).iloc[1:]

        # 停牌日不使用“价格不变”的伪收益率
        stock_return = stock_return.where(volume.iloc[1:].gt(0))
        return_parts.append(stock_return)

        market_part = stock_return.reindex(columns=market_ids)
        market_return_sum = market_return_sum.add(
            market_part.sum(axis=1, skipna=True),
            fill_value=0.0,
        )
        market_return_count = market_return_count.add(
            market_part.notna().sum(axis=1),
            fill_value=0,
        )

    if not return_parts:
        raise RuntimeError("未取得有效价格数据。")

    returns_wide = pd.concat(return_parts, axis=1)
    returns_wide = returns_wide.loc[
        :, ~returns_wide.columns.duplicated()
    ]
    returns_wide = returns_wide.reindex(
        index=price_dates[1:],
        columns=price_ids,
    )

    market_return = (
        market_return_sum / market_return_count.replace(0, np.nan)
    ).where(market_return_count >= min_market_stocks)

    # 2. ATTN：过去 attention_window 日平均平方异常收益
    attention_wide = (
        returns_wide.sub(market_return, axis=0)
        .pow(2)
        .rolling(
            window=attention_window,
            min_periods=attention_window,
        )
        .mean()
        .reindex(target_dates)
    )

    # 3. 获取每日市值；行业归属按月初更新
    market_cap_wide = _spill_get_market_cap_wide(
        order_book_ids=peer_ids,
        dates=target_dates,
        batch_size=batch_size,
    )

    industry_wide = pd.DataFrame(
        index=target_dates,
        columns=peer_ids,
        dtype=object,
    )

    if industry_refresh == "D":
        refresh_mask = np.ones(len(target_dates), dtype=bool)
    else:
        periods = target_dates.to_period(industry_refresh)
        refresh_mask = ~periods.duplicated()

    refresh_locs = np.flatnonzero(refresh_mask)

    for pos, begin_loc in enumerate(refresh_locs):
        end_loc = (
            refresh_locs[pos + 1]
            if pos + 1 < len(refresh_locs)
            else len(target_dates)
        )
        block_dates = target_dates[begin_loc:end_loc]

        industry_code = _spill_get_industry_codes(
            order_book_ids=peer_ids,
            asof_date=block_dates[0],
            industry_source=industry_source,
            industry_level=industry_level,
            batch_size=batch_size,
        )

        industry_wide.loc[block_dates, peer_ids] = np.tile(
            industry_code.values,
            (len(block_dates), 1),
        )

    # 4. 每日行业内按市值 3:4:3 分组，计算剔除自身后的同组均值
    factor_wide = pd.DataFrame(
        index=target_dates,
        columns=candidate_ids,
        dtype=float,
    )

    for date in target_dates:
        day = pd.DataFrame({
            "attention": attention_wide.loc[date, peer_ids],
            "market_cap": market_cap_wide.loc[date, peer_ids],
            "industry": industry_wide.loc[date, peer_ids],
        })

        day = day.dropna(subset=["attention", "market_cap", "industry"])
        day = day[day["market_cap"] > 0]

        if day.empty:
            continue

        # 行业内按市值从大到小排序，并按 3:4:3 分为大、中、小市值组
        day["rank"] = day.groupby("industry")["market_cap"].rank(
            method="first",
            ascending=False,
        )
        day["industry_count"] = day.groupby("industry")[
            "market_cap"
        ].transform("count")

        large_cutoff = np.ceil(day["industry_count"] * 0.3)
        middle_cutoff = np.ceil(day["industry_count"] * 0.7)

        day["size_group"] = np.where(
            day["rank"] <= large_cutoff,
            "L",
            np.where(day["rank"] <= middle_cutoff, "M", "S"),
        )

        day["group_key"] = (
            day["industry"].astype(str)
            + "_"
            + day["size_group"]
        )

        group_sum = day.groupby("group_key")["attention"].transform("sum")
        group_count = day.groupby("group_key")["attention"].transform("count")

        # 剔除自身后的同组注意力均值
        peer_attention_mean = (
            (group_sum - day["attention"])
            / (group_count - 1).replace(0, np.nan)
        )

        day["SPILL"] = (
            day["attention"] - peer_attention_mean
        ).where(group_count >= min_group_stocks)

        output = day.loc[
            day.index.isin(candidate_ids),
            "SPILL",
        ]
        factor_wide.loc[date, output.index] = output

    result = factor_wide.stack(dropna=True).rename("SPILL").to_frame()
    result.index = result.index.set_names(["date", "order_book_id"])

    return result

In [60]:
spill = calculate_spill_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    attention_window=20,
    candidate_ids=None,
    market_ids=None,
    peer_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_market_stocks=500,
    min_group_stocks=2,
    batch_size=500,
)

spill.head()

SPILL
date       order_book_id          
2020-01-02 000001.XSHE    0.000046
           000002.XSHE   -0.000056
           000004.XSHE    0.000434
           000005.XSHE   -0.000094
           000006.XSHE   -0.000307

# 基于换手率的注意力溢出（SPILL）

**行为金融因子 —— 投资者注意力溢出效应**

首先，计算每月日度换手率均值作为个股层面的注意力代理指标：

$$
\text{ATTN}_{i,t} = \frac{1}{m} \sum_{t=1}^{t-m+1} \text{TURN}_{i,t}
$$

其中，$\text{TURN}_{i,t}$ 为股票 $i$ 在交易日 $t$ 的换手率，$m$ 为交易日窗口，一般取为月度。

进一步，计算当期相近股票的注意力均值与个股注意力之差，即为注意力溢出程度：

$$
\text{SPILL}_{i,t} = \text{ATTN}_{i,t} - \overline{\text{ATTN}}_{g(i),t}
$$

其中，$\overline{\text{ATTN}}_{g(i),t}$ 为股票 $i$ 所属分组 $g(i)$ 内所有股票在 $t$ 期的注意力均值（不含股票 $i$ 自身）。

## 分组标准

相近股票的确定标准如下：

1. 先以**中信一级行业**作为分类标准，将全市场股票划分为不同行业；
2. 在每个行业内，按市值规模从大到小排序，以 **3:4:3** 的比例将股票分为**大市值、中市值、小市值**三组；
3. 同一行业同一市值组内的股票即被定义为“相近股票”。

## 因子说明

投资者注意力存在溢出效应，当一只股票关注度较高时，投资者也会注意到与其相似的邻居股票，导致这些邻居股票的未来收益也会有不错的收益，享有注意力溢价；上述因子以日均换手率为注意力代理指标，计算注意力差值，注意力差值越大，表明未来注意力溢出效应越显著，股票预期会取得正向超额。

In [3]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_factor,
    get_instrument_industry,
    get_trading_dates,
    get_turnover_rate,
)


def _turn_spill_get_ids(stock_ids):
    if stock_ids is None:
        return (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

    if isinstance(stock_ids, str):
        return [stock_ids]

    return list(dict.fromkeys(stock_ids))


def _turn_spill_get_turnover_wide(
    order_book_ids,
    dates,
    batch_size=500,
):
    """下载日换手率，整理为 日期 × 股票 宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_turnover_rate(
            order_book_ids=batch,
            start_date=dates[0],
            end_date=dates[-1],
            fields="today",
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            turnover = raw["today"].unstack("order_book_id")
        else:
            turnover = raw[["today"]].rename(
                columns={"today": batch[0]}
            )

        turnover.index = pd.to_datetime(turnover.index)

        parts.append(
            turnover.reindex(index=dates, columns=batch)
        )

    if not parts:
        raise RuntimeError("未取得有效换手率数据。")

    turnover_wide = pd.concat(parts, axis=1)
    turnover_wide = turnover_wide.loc[
        :, ~turnover_wide.columns.duplicated()
    ]

    return turnover_wide.reindex(
        index=dates,
        columns=order_book_ids,
    )


def _turn_spill_get_market_cap_wide(
    order_book_ids,
    dates,
    batch_size=500,
):
    """下载总市值 market_cap_3，整理为 日期 × 股票 宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_factor(
            order_book_ids=batch,
            factor="market_cap_3",
            start_date=dates[0],
            end_date=dates[-1],
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            market_cap = raw["market_cap_3"].unstack(
                "order_book_id"
            )
        else:
            market_cap = raw[["market_cap_3"]].rename(
                columns={"market_cap_3": batch[0]}
            )

        market_cap.index = pd.to_datetime(market_cap.index)

        parts.append(
            market_cap.reindex(index=dates, columns=batch)
        )

    if not parts:
        raise RuntimeError("未取得有效市值数据。")

    market_cap_wide = pd.concat(parts, axis=1)
    market_cap_wide = market_cap_wide.loc[
        :, ~market_cap_wide.columns.duplicated()
    ]

    return market_cap_wide.reindex(
        index=dates,
        columns=order_book_ids,
    )


def _turn_spill_get_industry_codes(
    order_book_ids,
    asof_date,
    industry_source="citics_2019",
    industry_level=1,
    batch_size=500,
):
    """取得指定日期的行业代码。"""
    field_map = {
        1: "first_industry_code",
        2: "second_industry_code",
        3: "third_industry_code",
    }

    industry_field = field_map[industry_level]
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_instrument_industry(
            order_book_ids=batch,
            source=industry_source,
            level=industry_level,
            date=asof_date,
        )

        if raw is not None and not raw.empty:
            parts.append(raw[industry_field])

    if not parts:
        return pd.Series(index=order_book_ids, dtype=object)

    industry_codes = pd.concat(parts)
    industry_codes = industry_codes[
        ~industry_codes.index.duplicated(keep="last")
    ]

    return industry_codes.reindex(order_book_ids)


def calculate_turnover_spill_daily(
    start_date,
    end_date,
    attention_window=20,
    candidate_ids=None,
    peer_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_group_stocks=2,
    batch_size=500,
):
    """
    基于换手率的注意力溢出因子

    输出：
        index  : [date, order_book_id]
        column : SPILL_TURN

    注意：
        candidate_ids 是输出股票池；
        peer_ids 是计算同行业、同市值组均值的股票池。
        两者均为 None 时，使用全 A 股。
    """
    if attention_window < 1:
        raise ValueError("attention_window 必须为正整数。")

    if industry_level not in (1, 2, 3):
        raise ValueError("industry_level 仅支持 1、2、3。")

    if industry_refresh not in ("M", "Q", "D"):
        raise ValueError(
            "industry_refresh 仅支持 'M'、'Q'、'D'。"
        )

    candidate_ids = _turn_spill_get_ids(candidate_ids)
    peer_ids = _turn_spill_get_ids(peer_ids)

    # 保证待输出股票也属于同组比较范围
    peer_ids = list(dict.fromkeys(peer_ids + candidate_ids))

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max(attention_window * 3, 90)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < attention_window - 1:
        raise ValueError("历史交易日不足，无法计算平均换手率。")

    attention_dates = calendar[
        first_loc - attention_window + 1:last_loc + 1
    ]

    # 1. ATTN：过去 attention_window 个交易日的平均换手率
    turnover_wide = _turn_spill_get_turnover_wide(
        order_book_ids=peer_ids,
        dates=attention_dates,
        batch_size=batch_size,
    )

    attention_wide = (
        turnover_wide
        .rolling(
            window=attention_window,
            min_periods=attention_window,
        )
        .mean()
        .reindex(target_dates)
    )

    # 2. 每日市值；行业归属在月初、季初或每日更新
    market_cap_wide = _turn_spill_get_market_cap_wide(
        order_book_ids=peer_ids,
        dates=target_dates,
        batch_size=batch_size,
    )

    industry_wide = pd.DataFrame(
        index=target_dates,
        columns=peer_ids,
        dtype=object,
    )

    if industry_refresh == "D":
        refresh_mask = np.ones(len(target_dates), dtype=bool)
    else:
        periods = target_dates.to_period(industry_refresh)
        refresh_mask = ~periods.duplicated()

    refresh_locs = np.flatnonzero(refresh_mask)

    for pos, begin_loc in enumerate(refresh_locs):
        end_loc = (
            refresh_locs[pos + 1]
            if pos + 1 < len(refresh_locs)
            else len(target_dates)
        )
        block_dates = target_dates[begin_loc:end_loc]

        industry_codes = _turn_spill_get_industry_codes(
            order_book_ids=peer_ids,
            asof_date=block_dates[0],
            industry_source=industry_source,
            industry_level=industry_level,
            batch_size=batch_size,
        )

        industry_wide.loc[block_dates, peer_ids] = np.tile(
            industry_codes.values,
            (len(block_dates), 1),
        )

    # 3. 每日按 行业 × 市值 3:4:3 分组，计算剔除自身后的均值
    factor_wide = pd.DataFrame(
        index=target_dates,
        columns=candidate_ids,
        dtype=float,
    )

    for date in target_dates:
        day = pd.DataFrame({
            "attention": attention_wide.loc[date, peer_ids],
            "market_cap": market_cap_wide.loc[date, peer_ids],
            "industry": industry_wide.loc[date, peer_ids],
        })

        day = day.dropna(
            subset=["attention", "market_cap", "industry"]
        )
        day = day[day["market_cap"] > 0]

        if day.empty:
            continue

        # 行业内按市值由大至小排序
        day["rank"] = day.groupby("industry")[
            "market_cap"
        ].rank(
            method="first",
            ascending=False,
        )

        day["industry_count"] = day.groupby("industry")[
            "market_cap"
        ].transform("count")

        # 3:4:3 市值分组
        large_cutoff = np.ceil(
            day["industry_count"] * 0.3
        )
        middle_cutoff = np.ceil(
            day["industry_count"] * 0.7
        )

        day["size_group"] = np.where(
            day["rank"] <= large_cutoff,
            "L",
            np.where(day["rank"] <= middle_cutoff, "M", "S"),
        )

        day["group_key"] = (
            day["industry"].astype(str)
            + "_"
            + day["size_group"]
        )

        group_sum = day.groupby("group_key")[
            "attention"
        ].transform("sum")

        group_count = day.groupby("group_key")[
            "attention"
        ].transform("count")

        peer_attention_mean = (
            (group_sum - day["attention"])
            / (group_count - 1).replace(0, np.nan)
        )

        day["SPILL_TURN"] = (
            day["attention"] - peer_attention_mean
        ).where(group_count >= min_group_stocks)

        output = day.loc[
            day.index.isin(candidate_ids),
            "SPILL_TURN",
        ]

        factor_wide.loc[date, output.index] = output

    result = (
        factor_wide
        .stack(dropna=True)
        .rename("SPILL_TURN")
        .to_frame()
    )
    result.index = result.index.set_names(
        ["date", "order_book_id"]
    )

    return result

In [4]:
turnover_spill = calculate_turnover_spill_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    attention_window=20,
    candidate_ids=None,
    peer_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    min_group_stocks=2,
    batch_size=500,
)

turnover_spill.head()

# SPILL_TURN > 0 表示个股近期平均换手率高于同行业、同市值组的其他股票，即自身注意力相对更强；小于 0 则表示相对较弱。该因子不依赖基金或龙虎榜数据。

SPILL_TURN
date       order_book_id            
2020-01-02 000001.XSHE      0.201932
           000002.XSHE      0.061198
           000004.XSHE     -0.462329
           000005.XSHE     -0.875383
           000006.XSHE     -0.694905

# 前景价值（TK）

**行为金融因子 —— 投资者前景理论偏好**

① 以个股过去 $N$ 日收益率序列作为未来收益分布的预测。将负收益率从小到大排列为 $r_{-m}, \dots, r_{-1}$，正收益率从小到大排列为 $r_1, \dots, r_n$，加上零收益 $r_0$，每个收益出现的概率均为 $1/N$，其中 $N = m + n + 1$：

$$
\left( r_{-m}, \frac{1}{N}; \dots; r_{-1}, \frac{1}{N}; r_0, \frac{1}{N}; r_1, \frac{1}{N}; \dots; r_n, \frac{1}{N} \right)
$$

② 利用价值函数和概率权重函数计算前景价值 TK：

$$
\text{TK} = \sum_{i=-m}^{-1} v(r_i) \left[ w\left(\frac{-i+m+1}{N}\right) - w\left(\frac{-i+m}{N}\right) \right] + \sum_{i=1}^{n} v(r_i) \left[ w\left(\frac{n-i+1}{N}\right) - w\left(\frac{n-i}{N}\right) \right]
$$

更紧凑地，前景价值可表达为：

$$
\text{TK} = \sum_{i=-m}^{n} v(r_i) \cdot \pi_i
$$

其中 $\pi_i$ 为对应于收益率 $r_i$ 的概率权重。

③ 价值函数 $v(x)$ 和概率权重函数 $w^+(P)$、$w^-(P)$ 如下所示，函数参数取值为 $\alpha = 0.88$，$\lambda = 2.25$，$\gamma = 0.61$，$\delta = 0.69$：

$$
v(x) =\begin{cases}x^{\alpha}, & x \geq 0 \\-\lambda (-x)^{\alpha}, & x < 0\end{cases}
$$$$
w^+(P) = \frac{P^{\gamma}}{\left( P^{\gamma} + (1-P)^{\gamma} \right)^{1/\gamma}}
$$$$
w^-(P) = \frac{P^{\delta}}{\left( P^{\delta} + (1-P)^{\delta} \right)^{1/\delta}}
$$

其中，$w^+(P)$ 用于盈利概率的权重处理，$w^-(P)$ 用于亏损概率的权重处理。

## 因子说明

前景价值（TK）代表了投资者对每只个股的“综合打分”，其理论基础源于行为金融学中的**前景理论**（Prospect Theory）。根据前景理论，投资者在决策时并非基于客观的期望效用，而是基于对收益和损失的主观价值评估，同时对小概率事件存在过度权重、对大概率事件存在权重不足的倾向。TK 因子通过将个股的历史收益率分布代入价值函数和权重函数，量化了投资者对该股票的主观偏好程度。由于价值量 TK 高的股票在投资者眼中更具吸引力，投资者倾向于买入那些 TK 值较高的股票，导致这些股票被过度追捧、价格被高估；而 TK 值较低的股票则因缺乏关注而被低估。因此，股票的未来预期收益率与当前的前景价值 TK 呈**负相关**关系，TK 越高，未来收益越低；TK 越低，未来收益越高。

数据：日度收盘价

In [3]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_price,
    get_trading_dates,
)


def _tk_probability_weight(probability, parameter):
    """Tversky-Kahneman 概率权重函数。"""
    probability = np.asarray(probability, dtype=float)

    numerator = np.power(probability, parameter)
    denominator = np.power(
        np.power(probability, parameter)
        + np.power(1.0 - probability, parameter),
        1.0 / parameter,
    )

    return numerator / denominator


def _tk_value_from_return_window(
    return_window,
    alpha=0.88,
    loss_aversion=2.25,
    gamma=0.61,
    delta=0.69,
):
    """
    对一个“历史收益窗口 × 股票”的 ndarray 计算 TK。
    每一列是一只股票的过去 N 日收益率。
    """
    n_obs, n_stocks = return_window.shape
    result = np.full(n_stocks, np.nan)

    # 只有窗口内每天都有有效交易收益率的股票才计算
    valid = np.isfinite(return_window).all(axis=0)
    if not valid.any():
        return result

    values = return_window[:, valid]

    loss_count = (values < 0).sum(axis=0)
    gain_count = (values > 0).sum(axis=0)

    # 将损失、盈利收益分别由小到大排序。
    # np.inf 会排在最后，因此前 loss_count / gain_count 个即为有效值。
    loss_sorted = np.sort(
        np.where(values < 0, values, np.inf),
        axis=0,
    )
    gain_sorted = np.sort(
        np.where(values > 0, values, np.inf),
        axis=0,
    )

    tk_valid = np.zeros(values.shape[1])

    # 损失侧：
    # pi_i^- = w^-((i+m+1)/N) - w^-((i+m)/N)
    for count in np.unique(loss_count):
        if count == 0:
            continue

        columns = np.flatnonzero(loss_count == count)
        loss_values = loss_sorted[:count, columns]

        probabilities = np.arange(1, count + 1, dtype=float) / n_obs
        previous_probabilities = np.arange(0, count, dtype=float) / n_obs

        weights = (
            _tk_probability_weight(probabilities, delta)
            - _tk_probability_weight(previous_probabilities, delta)
        )

        loss_utility = -loss_aversion * np.power(
            -loss_values,
            alpha,
        )

        tk_valid[columns] += (
            loss_utility * weights[:, None]
        ).sum(axis=0)

    # 盈利侧：
    # pi_i^+ = w^+((n-i+1)/N) - w^+((n-i)/N)
    for count in np.unique(gain_count):
        if count == 0:
            continue

        columns = np.flatnonzero(gain_count == count)
        gain_values = gain_sorted[:count, columns]

        probabilities = (
            np.arange(count, 0, -1, dtype=float) / n_obs
        )
        previous_probabilities = (
            np.arange(count - 1, -1, -1, dtype=float) / n_obs
        )

        weights = (
            _tk_probability_weight(probabilities, gamma)
            - _tk_probability_weight(previous_probabilities, gamma)
        )

        gain_utility = np.power(gain_values, alpha)

        tk_valid[columns] += (
            gain_utility * weights[:, None]
        ).sum(axis=0)

    result[valid] = tk_valid
    return result


def _tk_get_close_volume(batch, price_dates):
    raw = get_price(
        order_book_ids=batch,
        start_date=price_dates[0],
        end_date=price_dates[-1],
        frequency="1d",
        fields=["close", "volume"],
        adjust_type="pre",
        skip_suspended=False,
        expect_df=True,
    )

    if raw is None or raw.empty:
        return None, None

    if isinstance(raw.index, pd.MultiIndex):
        close = raw["close"].unstack("order_book_id")
        volume = raw["volume"].unstack("order_book_id")
    else:
        close = raw[["close"]].rename(columns={"close": batch[0]})
        volume = raw[["volume"]].rename(columns={"volume": batch[0]})

    close.index = pd.to_datetime(close.index)
    volume.index = pd.to_datetime(volume.index)

    return (
        close.reindex(index=price_dates, columns=batch),
        volume.reindex(index=price_dates, columns=batch),
    )


def calculate_tk_daily(
    start_date,
    end_date,
    window=60,
    candidate_ids=None,
    alpha=0.88,
    loss_aversion=2.25,
    gamma=0.61,
    delta=0.69,
    batch_size=500,
):
    """
    前景价值因子（TK）

    输出：
        index  : [date, order_book_id]
        column : TK

    注意：
        window=60 表示使用过去 60 个交易日收益率；
        因子在当日收盘后完整可得，应于下一交易日使用。
    """
    if window < 2:
        raise ValueError("window 至少应为 2。")

    if candidate_ids is None:
        candidate_ids = (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    else:
        candidate_ids = list(dict.fromkeys(candidate_ids))

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max((window + 1) * 3, 120)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < window:
        raise ValueError("历史交易日不足，无法计算 TK。")

    # 需要 window 个收益率，因此要额外多取一个价格日
    price_dates = calendar[first_loc - window:last_loc + 1]
    result_parts = []

    for start in range(0, len(candidate_ids), batch_size):
        batch = candidate_ids[start:start + batch_size]

        close, volume = _tk_get_close_volume(
            batch=batch,
            price_dates=price_dates,
        )

        if close is None:
            continue

        stock_return = close.pct_change(fill_method=None).iloc[1:]

        # 停牌日不将“价格不变”误视为零收益
        stock_return = stock_return.where(volume.iloc[1:].gt(0))

        return_values = stock_return.values.astype(float)
        target_locs = stock_return.index.get_indexer(target_dates)

        tk_wide = pd.DataFrame(
            index=target_dates,
            columns=batch,
            dtype=float,
        )

        for output_loc, return_loc in enumerate(target_locs):
            history_returns = return_values[
                return_loc - window + 1:return_loc + 1,
                :
            ]

            tk_wide.iloc[output_loc] = _tk_value_from_return_window(
                return_window=history_returns,
                alpha=alpha,
                loss_aversion=loss_aversion,
                gamma=gamma,
                delta=delta,
            )

        part = tk_wide.stack(dropna=True).rename("TK").to_frame()
        result_parts.append(part)

    if not result_parts:
        empty_index = pd.MultiIndex.from_arrays(
            [[], []],
            names=["date", "order_book_id"],
        )
        return pd.DataFrame(index=empty_index, columns=["TK"])

    result = pd.concat(result_parts).sort_index()
    result.index = result.index.set_names(["date", "order_book_id"])

    return result

In [4]:
tk = calculate_tk_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    window=60,
    candidate_ids=None,  # 全 A 股
    batch_size=500,
)

tk.head()

TK
date       order_book_id          
2020-01-02 000001.XSHE   -0.012431
           000002.XSHE   -0.002185
           000005.XSHE   -0.016968
           000006.XSHE   -0.009527
           000007.XSHE   -0.008930

# 基于 PB 的前景价值 TK

**行为金融因子 —— 前景理论**

① 以个股过去 $N$ 期的估值变化率（PB 估值变化率）作为衡量公司价值的代理变量，构建未来价值变化率的概率分布。将负变化率从小到大排列为 $r_{pb,-m}, \dots, r_{pb,-1}$，正变化率从小到大排列为 $r_{pb,1}, \dots, r_{pb,n}$，加上零变化率 $r_{pb,0}$，每个变化率出现的概率均为 $1/N$，其中 $N = m + n + 1$：

$$
\left( r_{pb,-m}, \frac{1}{N}; \dots; r_{pb,-1}, \frac{1}{N}; r_{pb,0}, \frac{1}{N}; r_{pb,1}, \frac{1}{N}; \dots; r_{pb,n}, \frac{1}{N} \right)
$$

② 利用价值函数和概率权重函数计算前景价值 TK：

$$
\text{TK} = \sum_{i=-m}^{-1} v(r_{pb,i}) \left[ w^-\left(\frac{i+m+1}{N}\right) - w^-\left(\frac{i+m}{N}\right) \right] + \sum_{i=1}^{n} v(r_{pb,i}) \left[ w^+\left(\frac{n-i+1}{N}\right) - w^+\left(\frac{n-i}{N}\right) \right]
$$

③ 价值函数 $v(x)$ 和概率权重函数 $w^+(P)$、$w^-(P)$ 的参数取值与标准前景理论一致，取值为 $\alpha = 0.88$，$\lambda = 2.25$，$\gamma = 0.61$，$\delta = 0.69$：

$$
v(x) =\begin{cases}x^{\alpha}, & x \geq 0 \\-\lambda (-x)^{\alpha}, & x < 0\end{cases}
$$$$
w^+(P) = \frac{P^{\gamma}}{\left( P^{\gamma} + (1-P)^{\gamma} \right)^{1/\gamma}}
$$$$
w^-(P) = \frac{P^{\delta}}{\left( P^{\delta} + (1-P)^{\delta} \right)^{1/\delta}}
$$

其中，$w^+(P)$ 用于估值上升（盈利）概率的权重处理，$w^-(P)$ 用于估值下降（亏损）概率的权重处理。

## 因子说明

前景价值 TK 代表了投资者对每只个股估值水平的“综合打分”。与标准 TK 因子以历史收益率作为价值参考不同，**基于 PB 的前景价值 TK 采用 PB 估值变化率作为衡量公司价值的代理变量**。

其核心逻辑在于：投资者在决策时，会基于公司估值的历史变化形成参考点，高估值股票虽然当前价值量（TK）较高，但投资者倾向于认为其估值过高而选择卖出，导致卖出压力增大、股价被低估；相反，低估值股票因具备安全边际而受到投资者青睐，导致买入需求增加、股价被高估。

因此，在该框架下，**前景价值 TK 与股票未来预期收益呈正相关关系**：TK 值越高（对应高估值股票），未来收益反而可能更高（因超卖后的价格修复）；TK 值越低（对应低估值股票），未来收益反而可能更低（因超买后的价格回落）。

数据：PB估值

In [9]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_factor,
    get_trading_dates,
)


def _pb_tk_probability_weight(probability, parameter):
    """前景理论概率权重函数。"""
    probability = np.asarray(probability, dtype=float)

    numerator = np.power(probability, parameter)
    denominator = np.power(
        np.power(probability, parameter)
        + np.power(1.0 - probability, parameter),
        1.0 / parameter,
    )

    return numerator / denominator


def _pb_tk_value_from_window(
    value_changes,
    alpha=0.88,
    loss_aversion=2.25,
    gamma=0.61,
    delta=0.69,
):
    """
    对一个“过去 N 期 PB 变化率 × 股票”的数组计算 TK。
    每列对应一只股票。
    """
    n_obs, n_stocks = value_changes.shape
    result = np.full(n_stocks, np.nan)

    valid = np.isfinite(value_changes).all(axis=0)
    if not valid.any():
        return result

    values = value_changes[:, valid]

    loss_count = (values < 0).sum(axis=0)
    gain_count = (values > 0).sum(axis=0)

    # 损失、盈利变化率分别从小到大排序
    loss_sorted = np.sort(
        np.where(values < 0, values, np.inf),
        axis=0,
    )
    gain_sorted = np.sort(
        np.where(values > 0, values, np.inf),
        axis=0,
    )

    tk_value = np.zeros(values.shape[1])

    # 损失侧概率权重
    for count in np.unique(loss_count):
        if count == 0:
            continue

        columns = np.flatnonzero(loss_count == count)
        loss_values = loss_sorted[:count, columns]

        probability = (
            np.arange(1, count + 1, dtype=float) / n_obs
        )
        previous_probability = (
            np.arange(0, count, dtype=float) / n_obs
        )

        weights = (
            _pb_tk_probability_weight(probability, delta)
            - _pb_tk_probability_weight(
                previous_probability,
                delta,
            )
        )

        utility = -loss_aversion * np.power(
            -loss_values,
            alpha,
        )

        tk_value[columns] += (
            utility * weights[:, None]
        ).sum(axis=0)

    # 盈利侧概率权重
    for count in np.unique(gain_count):
        if count == 0:
            continue

        columns = np.flatnonzero(gain_count == count)
        gain_values = gain_sorted[:count, columns]

        probability = (
            np.arange(count, 0, -1, dtype=float) / n_obs
        )
        previous_probability = (
            np.arange(count - 1, -1, -1, dtype=float) / n_obs
        )

        weights = (
            _pb_tk_probability_weight(probability, gamma)
            - _pb_tk_probability_weight(
                previous_probability,
                gamma,
            )
        )

        utility = np.power(gain_values, alpha)

        tk_value[columns] += (
            utility * weights[:, None]
        ).sum(axis=0)

    result[valid] = tk_value

    return result


def _pb_tk_get_pb_wide(
    order_book_ids,
    dates,
    pb_field="pb_ratio_lf",
    batch_size=500,
):
    """下载 PB 因子并整理为 日期 × 股票 宽表。"""
    parts = []

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        raw = get_factor(
            order_book_ids=batch,
            factor=pb_field,
            start_date=dates[0],
            end_date=dates[-1],
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        if isinstance(raw.index, pd.MultiIndex):
            pb_wide = raw[pb_field].unstack("order_book_id")
        else:
            pb_wide = raw[[pb_field]].rename(
                columns={pb_field: batch[0]}
            )

        pb_wide.index = pd.to_datetime(pb_wide.index)

        parts.append(
            pb_wide.reindex(index=dates, columns=batch)
        )

    if not parts:
        raise RuntimeError("未取得有效 PB 因子数据。")

    result = pd.concat(parts, axis=1)
    result = result.loc[:, ~result.columns.duplicated()]

    return result.reindex(
        index=dates,
        columns=order_book_ids,
    )


def calculate_pb_tk_daily(
    start_date,
    end_date,
    window=60,
    candidate_ids=None,
    pb_field="pb_ratio_lf",
    alpha=0.88,
    loss_aversion=2.25,
    gamma=0.61,
    delta=0.69,
    batch_size=500,
):
    """
    基于 PB 估值变化率的前景价值（TK）

    估值变化率：
        r_pb,t = PB_t / PB_(t-1) - 1

    输出：
        index  : [date, order_book_id]
        column : TK_PB

    window=60：
        使用过去 60 个交易日的 PB 估值变化率构造前景价值。
    """
    if window < 2:
        raise ValueError("window 至少应为 2。")

    if candidate_ids is None:
        candidate_ids = (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    elif isinstance(candidate_ids, str):
        candidate_ids = [candidate_ids]
    else:
        candidate_ids = list(dict.fromkeys(candidate_ids))

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max((window + 1) * 3, 120)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < window:
        raise ValueError("历史交易日不足，无法计算 TK_PB。")

    # 计算 window 个 PB 变化率，需要额外一日 PB 水平值
    pb_dates = calendar[
        first_loc - window:last_loc + 1
    ]

    pb_wide = _pb_tk_get_pb_wide(
        order_book_ids=candidate_ids,
        dates=pb_dates,
        pb_field=pb_field,
        batch_size=batch_size,
    )

    # 非正 PB 通常对应负净资产或异常估值，不纳入前景价值分布
    pb_wide = pb_wide.where(pb_wide > 0)

    pb_change = pb_wide.pct_change(fill_method=None).iloc[1:]

    pb_change_values = pb_change.values.astype(float)
    target_locs = pb_change.index.get_indexer(target_dates)

    factor_wide = pd.DataFrame(
        index=target_dates,
        columns=candidate_ids,
        dtype=float,
    )

    for output_loc, change_loc in enumerate(target_locs):
        history_changes = pb_change_values[
            change_loc - window + 1:change_loc + 1,
            :
        ]

        factor_wide.iloc[output_loc] = _pb_tk_value_from_window(
            value_changes=history_changes,
            alpha=alpha,
            loss_aversion=loss_aversion,
            gamma=gamma,
            delta=delta,
        )

    result = (
        factor_wide
        .stack(dropna=True)
        .rename("TK_PB")
        .to_frame()
    )
    result.index = result.index.set_names(
        ["date", "order_book_id"]
    )

    return result

In [10]:
tk_pb = calculate_pb_tk_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    window=60,
    candidate_ids=None,
    pb_field="pb_ratio_lf",
    batch_size=500,
)

tk_pb.head()

# TK_PB 越高，表示近期 PB 估值变化分布在前景理论下越具吸引力；越低则代表近期估值下行及其损失厌恶效应更突出。代码仍按标准 TK 处理：使用过去 window 个实际 PB 变化率，实际出现的零变化率自然保留，但不额外插入人为零值。因子在 t 日收盘后完整可得，建议用于 t+1 日交易。

TK_PB
date       order_book_id          
2020-01-02 000001.XSHE   -0.027897
           000002.XSHE   -0.004396
           000004.XSHE   -0.008058
           000005.XSHE   -0.015810
           000006.XSHE   -0.011180

# 关注事件

**行为金融因子 —— 投资者注意力**

涨跌停事件数量和龙虎榜事件数量可作为投资者关注度的直接代理指标：

$$
\text{LimitEvent}_{i,t} = \sum_{d \in m} \left( \mathbb{1}_{\text{涨停},i,d} + \mathbb{1}_{\text{跌停},i,d} \right)
$$

其中：

* $\text{LimitEvent}_{i,t}$ 为股票 $i$ 在月度 $t$ 内发生涨停或跌停的总次数；
* $\mathbb{1}_{\text{涨停},i,d}$ 为指示函数，若股票 $i$ 在交易日 $d$ 涨停则取值为 1，否则为 0；
* $\mathbb{1}_{\text{跌停},i,d}$ 为指示函数，若股票 $i$ 在交易日 $d$ 跌停则取值为 1，否则为 0；
* $m$ 为月度交易日窗口。

$$
\text{DragonEvent}_{i,t} = \sum_{d \in m} \text{Dragon}_{i,d}
$$

其中：

* $\text{DragonEvent}_{i,t}$ 为股票 $i$ 在月度 $t$ 内登上龙虎榜的总次数；
* $\text{Dragon}_{i,d}$ 为股票 $i$ 在交易日 $d$ 登上龙虎榜的次数；
* $m$ 为月度交易日窗口。

## 应用方法

在实际投资组合构建中，可以每期买入上述涨跌停事件或龙虎榜事件数量排名前 50% 的股票构建投资组合，通过检验该组合的未来收益表现来评估关注事件类因子的选股能力。

## 因子说明

涨跌停股票和龙虎榜股票通常因其价格剧烈波动或受到监管机构公开披露而容易引起市场的广泛关注。这些特殊事件的发生，使得相关股票在短时间内暴露于大量投资者的视野中，成为市场关注的焦点。

因此，涨跌停事件数量和龙虎榜事件数量可以作为投资者关注度的有效代理指标。事件发生频率越高，表明该股票受市场关注的程度越高，后续可能引发更多的交易行为，并对股价产生相应的影响。

数据：日度涨跌停数据（日度收盘价、日度开盘价）、日度龙虎榜数据

In [2]:
import numpy as np
import pandas as pd
import rqdatac

from rqdatac import (
    all_instruments,
    get_price,
    get_trading_dates,
)


def _attention_event_get_ids(stock_ids):
    if stock_ids is None:
        return (
            all_instruments("CS")["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

    if isinstance(stock_ids, str):
        return [stock_ids]

    return list(dict.fromkeys(stock_ids))


def _attention_event_get_price_wide(
    order_book_ids,
    dates,
    limit_definition="close",
    batch_size=500,
):
    """下载涨跌停事件判断所需的日度行情宽表。"""
    fields = [
        "close",
        "volume",
        "limit_up",
        "limit_down",
    ]

    if limit_definition == "touch":
        fields.extend(["high", "low"])

    all_fields = {
        field: []
        for field in fields
    }

    for start in range(0, len(order_book_ids), batch_size):
        batch = order_book_ids[start:start + batch_size]

        # 涨跌停价是未复权价格，必须配合 adjust_type="none"
        raw = get_price(
            order_book_ids=batch,
            start_date=dates[0],
            end_date=dates[-1],
            frequency="1d",
            fields=fields,
            adjust_type="none",
            skip_suspended=False,
            expect_df=True,
        )

        if raw is None or raw.empty:
            continue

        for field in fields:
            if isinstance(raw.index, pd.MultiIndex):
                wide = raw[field].unstack("order_book_id")
            else:
                wide = raw[[field]].rename(
                    columns={field: batch[0]}
                )

            wide.index = pd.to_datetime(wide.index)

            all_fields[field].append(
                wide.reindex(index=dates, columns=batch)
            )

    result = {}

    for field in fields:
        if not all_fields[field]:
            raise RuntimeError(
                "未取得有效日度行情数据。"
            )

        wide = pd.concat(all_fields[field], axis=1)
        wide = wide.loc[:, ~wide.columns.duplicated()]

        result[field] = wide.reindex(
            index=dates,
            columns=order_book_ids,
        )

    return result


def _attention_event_get_dragon_daily(
    order_book_ids,
    dates,
):
    """
    构造龙虎榜事件次数：
    同一股票在同一日出现多条龙虎榜记录时，按多次事件计数。
    """
    if not hasattr(rqdatac, "get_abnormal_stocks"):
        raise RuntimeError(
            "当前 rqdatac 版本不支持 get_abnormal_stocks；"
            "请升级至包含龙虎榜日明细接口的版本。"
        )

    raw = rqdatac.get_abnormal_stocks(
        start_date=dates[0],
        end_date=dates[-1],
    )

    dragon_daily = pd.DataFrame(
        0.0,
        index=dates,
        columns=order_book_ids,
    )

    if raw is None or raw.empty:
        return dragon_daily

    temp = raw.reset_index()

    id_column = (
        "order_book_id"
        if "order_book_id" in temp.columns
        else "order_book_ids"
    )

    if id_column not in temp.columns or "date" not in temp.columns:
        raise ValueError("龙虎榜数据返回格式异常。")

    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp[temp[id_column].isin(order_book_ids)]

    if temp.empty:
        return dragon_daily

    daily_count = (
        temp.groupby(["date", id_column])
        .size()
        .unstack(id_column)
    )

    return (
        daily_count
        .reindex(index=dates, columns=order_book_ids)
        .fillna(0.0)
    )


def calculate_attention_events_daily(
    start_date,
    end_date,
    window=20,
    candidate_ids=None,
    factor_type="both",
    limit_definition="close",
    batch_size=500,
):
    """
    关注事件因子

    参数：
        factor_type:
            "limit"  - 只输出 LimitEvent
            "dragon" - 只输出 DragonEvent
            "both"   - 同时输出两个因子

        limit_definition:
            "close" - 收盘价封在涨停或跌停价，默认且更严格
            "touch" - 日内最高/最低价触及涨停或跌停价

    输出：
        MultiIndex DataFrame
        index  : [date, order_book_id]
        column : LimitEvent 和/或 DragonEvent
    """
    if factor_type not in ("limit", "dragon", "both"):
        raise ValueError(
            "factor_type 仅支持 'limit'、'dragon'、'both'。"
        )

    if limit_definition not in ("close", "touch"):
        raise ValueError(
            "limit_definition 仅支持 'close'、'touch'。"
        )

    if window < 1:
        raise ValueError("window 必须为正整数。")

    candidate_ids = _attention_event_get_ids(candidate_ids)

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date - pd.Timedelta(
                days=max(window * 3, 90)
            ),
            end_date=end_date,
        )
    )

    target_dates = calendar[
        (calendar >= start_date) & (calendar <= end_date)
    ]

    if len(target_dates) == 0:
        raise ValueError("所选区间内没有交易日。")

    first_loc = calendar.get_loc(target_dates[0])
    last_loc = calendar.get_loc(target_dates[-1])

    if first_loc < window - 1:
        raise ValueError("历史交易日不足，无法计算滚动事件次数。")

    event_dates = calendar[
        first_loc - window + 1:last_loc + 1
    ]

    price_data = _attention_event_get_price_wide(
        order_book_ids=candidate_ids,
        dates=event_dates,
        limit_definition=limit_definition,
        batch_size=batch_size,
    )

    close = price_data["close"]
    volume = price_data["volume"]
    limit_up = price_data["limit_up"]
    limit_down = price_data["limit_down"]

    # 仅在实际交易的日期判断事件
    active = close.notna() & volume.gt(0)

    tolerance = 1e-6

    if limit_definition == "close":
        up_hit = (
            (close - limit_up).abs()
            <= tolerance + limit_up.abs() * tolerance
        )
        down_hit = (
            (close - limit_down).abs()
            <= tolerance + limit_down.abs() * tolerance
        )
    else:
        high = price_data["high"]
        low = price_data["low"]

        up_hit = (
            high >= limit_up - tolerance - limit_up.abs() * tolerance
        )
        down_hit = (
            low <= limit_down + tolerance + limit_down.abs() * tolerance
        )

    valid_limit_data = (
        active
        & limit_up.notna()
        & limit_down.notna()
    )

    # 同日若同时发生涨停、跌停，计为 2 次
    daily_limit_event = (
        up_hit.astype(float) + down_hit.astype(float)
    ).where(valid_limit_data)

    limit_factor = (
        daily_limit_event
        .rolling(window=window, min_periods=window)
        .sum()
        .reindex(target_dates)
        .where(active.reindex(target_dates))
    )

    factor_parts = []

    if factor_type in ("limit", "both"):
        factor_parts.append(
            limit_factor
            .stack(dropna=True)
            .rename("LimitEvent")
        )

    if factor_type in ("dragon", "both"):
        dragon_daily = _attention_event_get_dragon_daily(
            order_book_ids=candidate_ids,
            dates=event_dates,
        )

        # 停牌日不产生龙虎榜事件；不将停牌日纳入滚动窗口
        dragon_daily = dragon_daily.where(active)

        dragon_factor = (
            dragon_daily
            .rolling(window=window, min_periods=window)
            .sum()
            .reindex(target_dates)
            .where(active.reindex(target_dates))
        )

        factor_parts.append(
            dragon_factor
            .stack(dropna=True)
            .rename("DragonEvent")
        )

    result = pd.concat(factor_parts, axis=1)
    result.index = result.index.set_names(
        ["date", "order_book_id"]
    )

    return result

In [3]:
attention_events = calculate_attention_events_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    window=20,
    candidate_ids=None,
    factor_type="both",
    limit_definition="close",
    batch_size=500,
)

attention_events.head()

LimitEvent  DragonEvent
date       order_book_id                         
2020-01-02 000001.XSHE           0.0          0.0
           000002.XSHE           0.0          0.0
           000004.XSHE           0.0          1.0
           000005.XSHE           0.0          0.0
           000006.XSHE           0.0          0.0

# LSV 模型

**用于捕捉基金“抱团”羊群效应的行为金融模型**

LSV 模型的核心指标计算如下：

$$
\text{LSV}_{i,t} = \left| p_{i,t} - p_t \right| - \mathbb{E}\left| p_{i,t} - p_t \right|
$$

其中，第二项为在无羊群效应假设下的期望绝对偏差，其计算公式为：

$$
\mathbb{E}\left| p_{i,t} - p_t \right| = \text{AF}(i,t) = \sum_{k=1}^{N_{i,t}} \binom{N_{i,t}}{k} p_t^k (1 - p_t)^{N_{i,t}-k} \left| \frac{k}{N_{i,t}} - p_t \right|
$$

各基础变量定义如下：

$$
p_{i,t} = \frac{B(i,t)}{B(i,t) + S(i,t)}
$$$$
p_t = \frac{\sum_{i=1}^{N} p_{i,t}}{N}
$$$$
N_{i,t} = B(i,t) + S(i,t)
$$

其中：

* $B(i,t)$ 为在区间 $t$ 内买入股票 $i$ 的基金数量（即报告期持仓相较上期增加的基金数）；
* $S(i,t)$ 为在区间 $t$ 内卖出股票 $i$ 的基金数量（即报告期持仓相较上期减少的基金数）；
* $N_{i,t}$ 为持仓发生变动的基金总数（买入基金与卖出基金之和）；
* $p_{i,t}$ 为股票 $i$ 在区间 $t$ 内买入基金的占比；
* $p_t$ 为全市场所有股票在区间 $t$ 内买入基金占比的均值（跨股票平均值）；
* $N$ 为全市场股票总数。

## 计算示例

以基金年报或半年报持仓数据为例，若在报告期内有 300 只基金对股票 $i$ 进行了加仓操作，50 只基金进行了减仓操作，则：

$$
B(i,t) = 300,\quad S(i,t) = 50,\quad N_{i,t} = 350
$$$$
p_{i,t} = \frac{300}{350} \approx 0.857
$$

## 因子说明

LSV 模型从基金持股变动数据出发，通过比较某只股票实际被买入的比例与全市场平均买入比例的偏差程度，来判断该股票是否被基金集中买入或集中卖出，从而有效捕捉基金“抱团”现象。

当 $\text{LSV}_{i,t} > 0$ 且数值较大时，表明股票 $i$ 在当期存在显著的基金集体买入或集体卖出行为，即基金“抱团”现象明显。基金抱团是股票出现羊群效应的潜在驱动因子之一，抱团行为可能导致股价偏离基本面价值，并在未来出现价格反转。

数据：基金持仓数据

In [4]:
import sys

!{sys.executable} -m pip install --user --upgrade --no-cache-dir -i https://pypi.org/simple "rqdatac==3.4.2" "rqdatac-fund==1.0.44"

     |████████████████████████████████| 582 kB 16 kB/s eta 0:00:01
Requirement already up-to-date: rqdatac-fund==1.0.44 in /home/rice/notebook/.packages/lib/python3.6/site-packages (1.0.44)
  Attempting uninstall: rqdatac
    Found existing installation: rqdatac 3.0.15
    Uninstalling rqdatac-3.0.15:
      Successfully uninstalled rqdatac-3.0.15


In [7]:
import numpy as np
import pandas as pd
from math import lgamma

from rqdatac import fund, get_trading_dates


def _lsv_adjustment_factor(n_funds, market_buy_ratio):
    """
    AF(i,t) = E[|K / N - p_t|]
    其中 K ~ Binomial(N, p_t)。
    """
    if n_funds <= 0:
        return np.nan

    if market_buy_ratio <= 0.0 or market_buy_ratio >= 1.0:
        return 0.0

    k = np.arange(n_funds + 1, dtype=float)

    # 以 log-PMF 计算，避免组合数溢出
    log_combination = np.array([
        lgamma(n_funds + 1)
        - lgamma(int(value) + 1)
        - lgamma(n_funds - int(value) + 1)
        for value in k
    ])

    log_probability = (
        log_combination
        + k * np.log(market_buy_ratio)
        + (n_funds - k) * np.log(1.0 - market_buy_ratio)
    )

    probability = np.exp(log_probability)

    return np.sum(
        probability
        * np.abs(k / n_funds - market_buy_ratio)
    )


def _lsv_get_fund_ids(
    fund_ids=None,
    fund_types=("Stock", "Hybrid"),
    asof_date=None,
):
    """默认选取股票型与混合型公募基金。"""
    if fund_ids is not None:
        if isinstance(fund_ids, str):
            return [fund_ids]
        return list(dict.fromkeys(fund_ids))

    fund_info = fund.all_instruments(date=asof_date)

    return (
        fund_info.loc[
            fund_info["fund_type"].isin(fund_types),
            "order_book_id",
        ]
        .dropna()
        .drop_duplicates()
        .tolist()
    )


def _lsv_download_holdings(
    fund_ids,
    fund_batch_size=100,
):
    """
    下载全部历史基金持仓。

    返回字段：
        fund_id, report_date, order_book_id, shares, release_date
    """
    parts = []

    for start in range(0, len(fund_ids), fund_batch_size):
        batch = fund_ids[start:start + fund_batch_size]

        raw = fund.get_holdings(
            order_book_ids=batch,
        )

        if raw is None or raw.empty:
            continue

        temp = raw.reset_index()

        required_columns = {
            "fund_id",
            "date",
            "order_book_id",
            "shares",
            "release_date",
        }

        missing_columns = required_columns - set(temp.columns)
        if missing_columns:
            raise ValueError(
                "基金持仓数据缺少字段：{}".format(
                    sorted(missing_columns)
                )
            )

        # 只保留 A 股股票持仓
        if "type" in temp.columns:
            temp = temp[temp["type"] == "Stock"]

        if "category" in temp.columns:
            temp = temp[
                temp["category"]
                .astype(str)
                .str.lower()
                .eq("ashare")
            ]

        temp["date"] = pd.to_datetime(temp["date"])
        temp["release_date"] = pd.to_datetime(
            temp["release_date"]
        )
        temp["shares"] = pd.to_numeric(
            temp["shares"],
            errors="coerce",
        )

        temp = temp.dropna(
            subset=[
                "fund_id",
                "date",
                "order_book_id",
                "shares",
                "release_date",
            ]
        )

        parts.append(
            temp[
                [
                    "fund_id",
                    "date",
                    "order_book_id",
                    "shares",
                    "release_date",
                ]
            ]
        )

    if not parts:
        raise RuntimeError("未取得有效基金持仓数据。")

    holdings = pd.concat(parts, ignore_index=True)

    # 同一基金、报告期、股票如有重复记录，合并持仓股数
    holdings = (
        holdings
        .groupby(
            ["fund_id", "date", "order_book_id"],
            as_index=False,
        )
        .agg({
            "shares": "sum",
            "release_date": "max",
        })
    )

    return holdings.rename(columns={"date": "report_date"})


def _lsv_report_factors(
    holdings,
    report_months=(6, 12),
    share_tolerance=1e-8,
):
    """
    基于相邻报告期持仓，生成每个报告期的 LSV 横截面结果。
    """
    holdings = holdings[
        holdings["report_date"].dt.month.isin(report_months)
    ].copy()

    report_dates = sorted(
        holdings["report_date"].drop_duplicates().tolist()
    )

    report_factor_parts = []

    for date_index in range(1, len(report_dates)):
        previous_date = report_dates[date_index - 1]
        current_date = report_dates[date_index]

        previous = holdings[
            holdings["report_date"] == previous_date
        ].copy()

        current = holdings[
            holdings["report_date"] == current_date
        ].copy()

        # 仅使用两个报告期均有披露持仓的基金，
        # 避免把“未披露”错误视作“清仓”。
        common_funds = np.intersect1d(
            previous["fund_id"].unique(),
            current["fund_id"].unique(),
        )

        if len(common_funds) == 0:
            continue

        previous = previous[
            previous["fund_id"].isin(common_funds)
        ]
        current = current[
            current["fund_id"].isin(common_funds)
        ]

        merged = pd.merge(
            current[
                ["fund_id", "order_book_id", "shares"]
            ].rename(columns={"shares": "shares_current"}),
            previous[
                ["fund_id", "order_book_id", "shares"]
            ].rename(columns={"shares": "shares_previous"}),
            on=["fund_id", "order_book_id"],
            how="outer",
        )

        merged["shares_current"] = (
            merged["shares_current"].fillna(0.0)
        )
        merged["shares_previous"] = (
            merged["shares_previous"].fillna(0.0)
        )

        change = (
            merged["shares_current"]
            - merged["shares_previous"]
        )

        buy_count = (
            merged.loc[
                change > share_tolerance
            ]
            .groupby("order_book_id")["fund_id"]
            .nunique()
            .rename("B")
        )

        sell_count = (
            merged.loc[
                change < -share_tolerance
            ]
            .groupby("order_book_id")["fund_id"]
            .nunique()
            .rename("S")
        )

        stock_changes = pd.concat(
            [buy_count, sell_count],
            axis=1,
        ).fillna(0.0)

        stock_changes["N"] = (
            stock_changes["B"] + stock_changes["S"]
        )
        stock_changes = stock_changes[
            stock_changes["N"] > 0
        ].copy()

        if stock_changes.empty:
            continue

        stock_changes["p_i"] = (
            stock_changes["B"] / stock_changes["N"]
        )

        # p_t 只对实际发生买卖变化的股票求横截面均值
        market_buy_ratio = stock_changes["p_i"].mean()

        stock_changes["p_t"] = market_buy_ratio

        af_map = {
            n_funds: _lsv_adjustment_factor(
                int(n_funds),
                market_buy_ratio,
            )
            for n_funds in stock_changes["N"].unique()
        }

        stock_changes["AF"] = stock_changes["N"].map(af_map)

        stock_changes["LSV"] = (
            (stock_changes["p_i"] - market_buy_ratio).abs()
            - stock_changes["AF"]
        )

        # 因子在该报告期最后一份有效持仓披露后才可使用
        available_date = current["release_date"].max()

        stock_changes = stock_changes.reset_index()
        stock_changes["report_date"] = current_date
        stock_changes["available_date"] = available_date

        report_factor_parts.append(stock_changes)

    if not report_factor_parts:
        raise RuntimeError("没有足够的相邻基金报告期用于计算 LSV。")

    return pd.concat(report_factor_parts, ignore_index=True)


def calculate_lsv_daily(
    start_date,
    end_date,
    candidate_ids=None,
    fund_ids=None,
    fund_types=("Stock", "Hybrid"),
    report_months=(6, 12),
    fund_batch_size=100,
    share_tolerance=1e-8,
):
    """
    基金抱团羊群效应因子（LSV）

    输出：
        index  : [date, order_book_id]
        column : LSV

    默认：
        - 使用股票型、混合型基金；
        - 使用半年报、年报，即 report_months=(6, 12)；
        - 使用实际 release_date 的下一交易日作为因子生效日。
    """
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    fund_ids = _lsv_get_fund_ids(
        fund_ids=fund_ids,
        fund_types=fund_types,
        asof_date=end_date,
    )

    holdings = _lsv_download_holdings(
        fund_ids=fund_ids,
        fund_batch_size=fund_batch_size,
    )

    report_factor = _lsv_report_factors(
        holdings=holdings,
        report_months=report_months,
        share_tolerance=share_tolerance,
    )

    # 使用披露日的下一交易日，避免使用同日盘后才公开的信息
    release_dates = pd.DatetimeIndex(
        report_factor["available_date"].dropna().unique()
    )

    signal_calendar = pd.DatetimeIndex(
        get_trading_dates(
            start_date=release_dates.min(),
            end_date=end_date,
        )
    )

    def next_trading_date(date):
        position = signal_calendar.searchsorted(
            pd.Timestamp(date),
            side="right",
        )

        if position >= len(signal_calendar):
            return pd.NaT

        return signal_calendar[position]

    report_factor["signal_date"] = report_factor[
        "available_date"
    ].map(next_trading_date)

    report_factor = report_factor.dropna(
        subset=["signal_date"]
    )

    if candidate_ids is None:
        candidate_ids = (
            report_factor["order_book_id"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )
    elif isinstance(candidate_ids, str):
        candidate_ids = [candidate_ids]
    else:
        candidate_ids = list(dict.fromkeys(candidate_ids))

    report_factor = report_factor[
        report_factor["order_book_id"].isin(candidate_ids)
    ].copy()

    # 同一天如有多个报告期快照，保留最新报告期
    report_factor = (
        report_factor
        .sort_values(["signal_date", "report_date"])
        .drop_duplicates(
            subset=["signal_date", "order_book_id"],
            keep="last",
        )
    )

    snapshot_wide = report_factor.pivot(
        index="signal_date",
        columns="order_book_id",
        values="LSV",
    )

    target_dates = pd.DatetimeIndex(
        get_trading_dates(
            start_date=start_date,
            end_date=end_date,
        )
    )

    all_dates = target_dates.union(
        snapshot_wide.index
    ).sort_values()

    # 非日频基金持仓因子：披露后前向填充为日频信号
    daily_wide = (
        snapshot_wide
        .reindex(index=all_dates, columns=candidate_ids)
        .ffill()
        .reindex(target_dates)
    )

    result = daily_wide.stack(dropna=True).rename("LSV").to_frame()
    result.index = result.index.set_names(
        ["date", "order_book_id"]
    )

    return result

In [2]:
import rqdatac
from rqdatac import fund

print("rqdatac:", rqdatac.__version__)
print("基金模块可用：", fund is not None)
print(
    "龙虎榜接口可用：",
    hasattr(rqdatac, "get_abnormal_stocks")
)
fund.all_instruments().head()

rqdatac: 3.4.2
基金模块可用： True
龙虎榜接口可用： True


PermissionDenied: permission denied: fund.all_instruments

In [8]:
lsv = calculate_lsv_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    candidate_ids=None,
    fund_types=("Stock", "Hybrid"),
    report_months=(6, 12),
    fund_batch_size=100,
)

lsv.head()

# 若要使用季报持仓，将 report_months 改为：
# report_months=(3, 6, 9, 12)

PermissionDenied: permission denied: fund.all_instruments

## 数据获取（付费接口）

In [ ]:
# 基金持仓数据.py

# 🚨基于异常收益的注意力捕捉

**行为金融因子 —— 投资者注意力**

对每个行业，分别统计每日行业内成分股中收益率绝对值高于 7% 的股票数量占比，作为市场关注程度的代理指标：

$$
\text{indus_abnd}_d = \frac{n_{\ge 7\%, d}}{N_d}
$$

其中，$n_{\ge 7\%, d}$ 为行业成分股中当日收益率绝对值达到或超过 7% 的股票数量，$N_d$ 为该行业当日交易股票总数。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{indus_abnd}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \epsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受投资者注意力影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\epsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而引发更多的买盘压力，导致股价被高估。

上述因子以异常收益占比（收益率绝对值超过 7% 的股票比例）作为市场关注度事件，其原因在于：极端日收益是衡量关注度的一个重要指标，出现过高或过低的收益都代表着该股票或行业受到了投资者的广泛关注。因此，该指标能够有效捕捉投资者注意力的集中程度，并进一步用于解释个股收益的横截面差异。

数据：日度收盘价、Fama-French因子（市值、无风险收益率、账面市值比（净资产/总市值））、行业分类数据

In [17]:
import numpy as np
import pandas as pd

from rqdatac import (
    all_instruments,
    get_factor,
    get_instrument_industry,
    get_price,
    get_trading_dates,
)


# ============================================================
# 1. 通用数据处理函数
# ============================================================

def _ac_chunks(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def _ac_to_wide(raw, order_book_ids, dates, value_col):
    """
    将米筐返回的长表 / MultiIndex 表转换为：
    index=date, columns=order_book_id 的宽表。
    """
    if raw is None or len(raw) == 0:
        return pd.DataFrame(index=dates, columns=order_book_ids, dtype=float)

    if isinstance(raw, pd.Series):
        raw = raw.to_frame(name=value_col)

    data = raw.copy()

    if isinstance(data.index, pd.MultiIndex) or data.index.name is not None:
        data = data.reset_index()

    if "order_book_id" not in data.columns:
        candidates = [
            col for col in data.columns
            if col.lower() in ("instrument", "symbol", "order_book_id")
        ]
        if not candidates:
            raise ValueError("无法在返回数据中识别 order_book_id 字段。")
        data = data.rename(columns={candidates[0]: "order_book_id"})

    if "date" not in data.columns:
        candidates = [
            col for col in data.columns
            if col.lower() in ("datetime", "trading_date", "date")
        ]
        if not candidates:
            raise ValueError("无法在返回数据中识别 date 字段。")
        data = data.rename(columns={candidates[0]: "date"})

    if value_col not in data.columns:
        non_key_cols = [
            col for col in data.columns
            if col not in ("date", "order_book_id")
        ]
        if len(non_key_cols) != 1:
            raise ValueError(
                "无法识别数值列 {}，当前列为：{}".format(
                    value_col, list(data.columns)
                )
            )
        data = data.rename(columns={non_key_cols[0]: value_col})

    data["date"] = pd.to_datetime(data["date"])

    wide = data.pivot_table(
        index="date",
        columns="order_book_id",
        values=value_col,
        aggfunc="last",
    )

    return wide.reindex(index=dates, columns=order_book_ids)


def _ac_get_price_wide(order_book_ids, dates, fields, batch_size):
    """分批下载日线数据，并转为宽表字典。"""
    result = {field: [] for field in fields}

    for ids in _ac_chunks(order_book_ids, batch_size):
        raw = get_price(
            order_book_ids=ids,
            start_date=dates[0],
            end_date=dates[-1],
            frequency="1d",
            fields=fields,
            adjust_type="pre",
            skip_suspended=False,
            expect_df=True,
        )

        for field in fields:
            result[field].append(
                _ac_to_wide(raw, ids, dates, field)
            )

    output = {}
    for field in fields:
        output[field] = (
            pd.concat(result[field], axis=1)
            .reindex(index=dates, columns=order_book_ids)
        )

    return output


def _ac_get_factor_wide(
    order_book_ids,
    dates,
    factor_name,
    batch_size,
):
    """分批下载 get_factor 数据。"""
    blocks = []

    for ids in _ac_chunks(order_book_ids, batch_size):
        raw = get_factor(
            order_book_ids=ids,
            factor=factor_name,
            start_date=dates[0],
            end_date=dates[-1],
            expect_df=True,
        )

        blocks.append(
            _ac_to_wide(raw, ids, dates, factor_name)
        )

    return (
        pd.concat(blocks, axis=1)
        .reindex(index=dates, columns=order_book_ids)
        .astype(float)
    )


def _ac_get_industry_codes(
    order_book_ids,
    date,
    industry_source,
    industry_level,
    batch_size,
):
    """
    获取某一日期的行业编码。
    返回 Series：
    index=order_book_id，value=行业代码。
    """
    blocks = []

    if industry_level != 1:
        raise ValueError("当前函数仅实现中信一级行业，请设置 industry_level=1。")

    industry_field = "first_industry_code"

    for ids in _ac_chunks(order_book_ids, batch_size):
        raw = get_instrument_industry(
            order_book_ids=ids,
            source=industry_source,
            level=industry_level,
            date=date,
        )

        if raw is None or len(raw) == 0:
            continue

        data = raw.copy()

        if isinstance(data.index, pd.MultiIndex) or data.index.name is not None:
            data = data.reset_index()

        if "order_book_id" not in data.columns:
            possible_cols = [
                col for col in data.columns
                if col.lower() in ("instrument", "symbol", "order_book_id")
            ]
            if not possible_cols:
                raise ValueError("行业数据中无法识别 order_book_id。")
            data = data.rename(columns={possible_cols[0]: "order_book_id"})

        if industry_field not in data.columns:
            raise ValueError(
                "行业数据中不存在字段 {}，当前列为：{}".format(
                    industry_field, list(data.columns)
                )
            )

        blocks.append(
            data.set_index("order_book_id")[industry_field]
        )

    if not blocks:
        return pd.Series(dtype=object)

    return pd.concat(blocks).groupby(level=0).last()


def _ac_first_dates_by_month(dates):
    """每个月的第一个交易日。"""
    month_index = dates.to_period("M")
    return dates[~month_index.duplicated()]


def _ac_first_dates_by_july(dates):
    """每年 7 月的第一个交易日。"""
    month_first_dates = _ac_first_dates_by_month(dates)
    return month_first_dates[month_first_dates.month == 7]


def _ac_value_weighted_return(returns, weights, mask, min_stocks):
    """计算一个组合的市值加权收益。"""
    valid = (
        mask
        & np.isfinite(returns)
        & np.isfinite(weights)
        & (weights > 0)
    )

    if valid.sum() < min_stocks:
        return np.nan

    selected_weights = weights[valid]
    selected_returns = returns[valid]

    return np.dot(selected_returns, selected_weights) / selected_weights.sum()


# ============================================================
# 2. 构造 MKT / SMB / HML / UMD
# ============================================================

def _ac_safe_size_value_masks(
    cap,
    btm,
    valid,
    size_breakpoint,
    btm_low,
    btm_high,
):
    """仅对有效股票做 Size / BTM 分组，避免 NaN 比较警告。"""
    n_stocks = len(cap)

    small = np.zeros(n_stocks, dtype=bool)
    big = np.zeros(n_stocks, dtype=bool)

    low = np.zeros(n_stocks, dtype=bool)
    neutral = np.zeros(n_stocks, dtype=bool)
    high = np.zeros(n_stocks, dtype=bool)

    valid_idx = np.flatnonzero(valid)

    cap_valid = cap[valid_idx]
    btm_valid = btm[valid_idx]

    small[valid_idx] = cap_valid <= size_breakpoint
    big[valid_idx] = cap_valid > size_breakpoint

    low[valid_idx] = btm_valid <= btm_low
    neutral[valid_idx] = (
        (btm_valid > btm_low)
        & (btm_valid <= btm_high)
    )
    high[valid_idx] = btm_valid > btm_high

    return small, big, low, neutral, high

def _ac_safe_momentum_masks(
    cumulative_return,
    valid,
    loser_breakpoint,
    winner_breakpoint,
):
    """仅对有效股票做动量分组，避免 NaN 比较警告。"""
    n_stocks = len(cumulative_return)

    losers = np.zeros(n_stocks, dtype=bool)
    winners = np.zeros(n_stocks, dtype=bool)

    valid_idx = np.flatnonzero(valid)
    valid_return = cumulative_return[valid_idx]

    losers[valid_idx] = valid_return <= loser_breakpoint
    winners[valid_idx] = valid_return >= winner_breakpoint

    return losers, winners

def _ac_build_ff4_controls(
    returns,
    lag_market_cap,
    lag_book_to_market,
    min_market_stocks=100,
    min_portfolio_stocks=10,
):
    """
    构造日频 MKT / SMB / HML / UMD。

    MKT:
        全市场前一日市值加权日收益，不减无风险利率。

    SMB / HML:
        每年 7 月首个交易日，按前一日市值和 B/M 分为：
        Small/Big × Low/Neutral/High，共六个组合；
        持有至下一年 7 月。

    UMD:
        每月首个交易日，以过去 11 个完整自然月的累计收益排序，
        跳过最近一个月，取前 30% 与后 30%，持有一个月。
    """
    dates = returns.index
    return_array = returns.values.astype(float)
    cap_array = lag_market_cap.values.astype(float)
    btm_array = lag_book_to_market.values.astype(float)

    n_dates, n_stocks = return_array.shape

    # --------------------------------------------------------
    # SMB / HML：每年 7 月形成一次 2 × 3 组合
    # --------------------------------------------------------
    size_label = np.full((n_dates, n_stocks), -1, dtype=np.int8)
    value_label = np.full((n_dates, n_stocks), -1, dtype=np.int8)

    annual_rebalance_dates = _ac_first_dates_by_july(dates)
    annual_positions = [dates.get_loc(date) for date in annual_rebalance_dates]

    for seq, position in enumerate(annual_positions):
        end_position = (
            annual_positions[seq + 1]
            if seq + 1 < len(annual_positions)
            else n_dates
        )

        cap = cap_array[position]
        btm = btm_array[position]

        valid = (
            np.isfinite(cap)
            & np.isfinite(btm)
            & (cap > 0)
            & (btm > 0)
        )

        if valid.sum() < max(min_market_stocks, 30):
            continue

        size_breakpoint = np.median(cap[valid])
        btm_low, btm_high = np.percentile(
            btm[valid],
            [30.0, 70.0],
        )

        small, big, low, neutral, high = _ac_safe_size_value_masks(
            cap=cap,
            btm=btm,
            valid=valid,
            size_breakpoint=size_breakpoint,
            btm_low=btm_low,
            btm_high=btm_high,
        )

        size_label[position:end_position, small] = 0
        size_label[position:end_position, big] = 1

        value_label[position:end_position, low] = 0
        value_label[position:end_position, neutral] = 1
        value_label[position:end_position, high] = 2

    # --------------------------------------------------------
    # UMD：每月形成一次，过去 11 个月、跳过最近 1 个月
    # --------------------------------------------------------
    momentum_label = np.full((n_dates, n_stocks), -1, dtype=np.int8)

    month_periods = dates.to_period("M")
    monthly_rebalance_dates = _ac_first_dates_by_month(dates)
    monthly_positions = [dates.get_loc(date) for date in monthly_rebalance_dates]

    for seq, position in enumerate(monthly_positions):
        end_position = (
            monthly_positions[seq + 1]
            if seq + 1 < len(monthly_positions)
            else n_dates
        )

        current_month = dates[position].to_period("M")

        # t-12 月至 t-2 月：共 11 个完整自然月
        begin_month = current_month - 12
        finish_month = current_month - 2

        history_mask = (
            (month_periods >= begin_month)
            & (month_periods <= finish_month)
        )

        history_returns = return_array[history_mask]

        if history_returns.shape[0] < 150:
            continue

        # 要求整个动量观察期内存在连续、可交易收益
        valid_history = np.isfinite(history_returns).all(axis=0)

        cumulative_return = np.full(n_stocks, np.nan)
        cumulative_return[valid_history] = (
            np.prod(
                1.0 + history_returns[:, valid_history],
                axis=0,
            ) - 1.0
        )

        formation_cap = cap_array[position]

        valid = (
            valid_history
            & np.isfinite(cumulative_return)
            & np.isfinite(formation_cap)
            & (formation_cap > 0)
        )

        if valid.sum() < max(min_market_stocks, 30):
            continue

        loser_breakpoint, winner_breakpoint = np.percentile(
            cumulative_return[valid],
            [30.0, 70.0],
        )

        losers, winners = _ac_safe_momentum_masks(
            cumulative_return=cumulative_return,
            valid=valid,
            loser_breakpoint=loser_breakpoint,
            winner_breakpoint=winner_breakpoint,
        )

        momentum_label[position:end_position, losers] = 0
        momentum_label[position:end_position, winners] = 1

    # --------------------------------------------------------
    # 每日组合收益
    # --------------------------------------------------------
    factor_values = np.full((n_dates, 4), np.nan)

    for t in range(n_dates):
        daily_return = return_array[t]
        daily_cap = cap_array[t]

        # MKT：全市场市值加权收益
        market_mask = (
            np.isfinite(daily_return)
            & np.isfinite(daily_cap)
            & (daily_cap > 0)
        )

        factor_values[t, 0] = _ac_value_weighted_return(
            daily_return,
            daily_cap,
            market_mask,
            min_market_stocks,
        )

        # SMB / HML：六个 Size × B/M 组合
        portfolio_returns = {}

        for size in (0, 1):
            for value in (0, 1, 2):
                mask = (
                    (size_label[t] == size)
                    & (value_label[t] == value)
                )

                portfolio_returns[(size, value)] = _ac_value_weighted_return(
                    daily_return,
                    daily_cap,
                    mask,
                    min_portfolio_stocks,
                )

        sl = portfolio_returns[(0, 0)]
        sn = portfolio_returns[(0, 1)]
        sh = portfolio_returns[(0, 2)]

        bl = portfolio_returns[(1, 0)]
        bn = portfolio_returns[(1, 1)]
        bh = portfolio_returns[(1, 2)]

        if np.isfinite([sl, sn, sh, bl, bn, bh]).all():
            factor_values[t, 1] = (
                (sl + sn + sh) / 3.0
                - (bl + bn + bh) / 3.0
            )

            factor_values[t, 2] = (
                (sh + bh) / 2.0
                - (sl + bl) / 2.0
            )

        # UMD：前 30% 动量组合 - 后 30% 动量组合
        winner_return = _ac_value_weighted_return(
            daily_return,
            daily_cap,
            momentum_label[t] == 1,
            min_portfolio_stocks,
        )

        loser_return = _ac_value_weighted_return(
            daily_return,
            daily_cap,
            momentum_label[t] == 0,
            min_portfolio_stocks,
        )

        if np.isfinite(winner_return) and np.isfinite(loser_return):
            factor_values[t, 3] = winner_return - loser_return

    return pd.DataFrame(
        factor_values,
        index=dates,
        columns=["MKT", "SMB", "HML", "UMD"],
    )


# ============================================================
# 3. 构造行业异常收益注意力，并进行滚动回归
# ============================================================

def calculate_indus_abnd_attention_daily(
    start_date,
    end_date,
    candidate_ids=None,
    universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    abnormal_threshold=0.07,
    regression_window=20,
    min_industry_stocks=5,
    min_market_stocks=100,
    min_portfolio_stocks=10,
    initial_lookback_calendar_days=900,
    price_batch_size=500,
    industry_batch_size=2000,
    return_controls=False,
):
    """
    基于异常收益的注意力捕捉因子。

    返回
    ----
    DataFrame
        两层 MultiIndex：
        date, order_book_id

        列：
        INDUS_ABND_BETA

        数值为滚动回归中行业注意力变量 beta 的绝对值。

    参数说明
    --------
    candidate_ids:
        最终需要输出因子的股票；None 表示全 A 股。

    universe_ids:
        构造行业注意力、MKT、SMB、HML、UMD 时使用的股票池；
        None 表示全 A 股。正式回测建议保持 None。

    industry_refresh:
        "M"：每月首个交易日更新行业归属，推荐；
        "D"：每日更新，最精确但请求量很大。

    return_controls:
        True 时，返回 (factor, controls)；
        controls 为日频 MKT / SMB / HML / UMD。
    """
    if regression_window <= 6:
        raise ValueError(
            "regression_window 必须大于 6，"
            "因为回归中包含截距、行业注意力和四个控制变量。"
        )

    if industry_refresh not in ("M", "D"):
        raise ValueError('industry_refresh 仅支持 "M" 或 "D"。')

    # --------------------------------------------------------
    # 股票池
    # --------------------------------------------------------
    all_stock_ids = list(
        all_instruments("CS")["order_book_id"].dropna().unique()
    )

    if universe_ids is None:
        universe_ids = all_stock_ids
    else:
        universe_ids = list(pd.Index(universe_ids).dropna().unique())

    if candidate_ids is None:
        candidate_ids = universe_ids
    else:
        candidate_ids = list(pd.Index(candidate_ids).dropna().unique())

    # 保证待输出股票也在因子构造股票池中
    total_ids = list(
        dict.fromkeys(universe_ids + candidate_ids)
    )

    # --------------------------------------------------------
    # 交易日与历史区间
    # --------------------------------------------------------
    target_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(start_date, end_date)
        )
    )

    if len(target_dates) == 0:
        raise ValueError("目标日期区间内没有交易日。")

    history_start = (
        pd.Timestamp(target_dates[0])
        - pd.Timedelta(days=initial_lookback_calendar_days)
    )

    data_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(history_start, end_date)
        )
    )

    if len(data_dates) < 300:
        raise ValueError("历史交易日不足，无法构造 UMD 与滚动回归。")

    # --------------------------------------------------------
    # 下载价格、成交量、市值、账面市值比
    # --------------------------------------------------------
    price_data = _ac_get_price_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        fields=["close", "volume"],
        batch_size=price_batch_size,
    )

    close = price_data["close"].astype(float)
    volume = price_data["volume"].astype(float)

    # 仅将当日成交量大于 0 的股票视为当日可交易股票
    returns = close.pct_change(fill_method=None).iloc[1:]
    returns = returns.where(volume.iloc[1:].gt(0))

    return_dates = returns.index

    market_cap = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="market_cap_3",
        batch_size=price_batch_size,
    )

    book_to_market = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="book_to_market_ratio_lf",
        batch_size=price_batch_size,
    )

    # 所有组合权重一律使用 t-1 市值，避免使用当日收盘后的信息
    lag_market_cap = market_cap.shift(1).reindex(return_dates)
    lag_book_to_market = book_to_market.shift(1).reindex(return_dates)

    # --------------------------------------------------------
    # 构造 Fama-French / UMD 日频控制变量
    # --------------------------------------------------------
    controls = _ac_build_ff4_controls(
        returns=returns,
        lag_market_cap=lag_market_cap,
        lag_book_to_market=lag_book_to_market,
        min_market_stocks=min_market_stocks,
        min_portfolio_stocks=min_portfolio_stocks,
    )

    # --------------------------------------------------------
    # 行业归属：月度或日度更新
    # --------------------------------------------------------
    n_dates = len(return_dates)
    n_stocks = len(total_ids)

    if industry_refresh == "M":
        industry_update_dates = _ac_first_dates_by_month(return_dates)
    else:
        industry_update_dates = return_dates

    industry_update_positions = [
        return_dates.get_loc(date)
        for date in industry_update_dates
    ]

    industry_labels = np.full(
        (n_dates, n_stocks),
        -1,
        dtype=np.int16,
    )

    code_to_label = {}
    total_id_to_position = {
        stock_id: pos
        for pos, stock_id in enumerate(total_ids)
    }

    for seq, update_date in enumerate(industry_update_dates):
        start_position = industry_update_positions[seq]
        end_position = (
            industry_update_positions[seq + 1]
            if seq + 1 < len(industry_update_positions)
            else n_dates
        )

        industry_code = _ac_get_industry_codes(
            order_book_ids=total_ids,
            date=update_date,
            industry_source=industry_source,
            industry_level=industry_level,
            batch_size=industry_batch_size,
        )

        labels_one_day = np.full(n_stocks, -1, dtype=np.int16)

        for stock_id, code in industry_code.items():
            if stock_id not in total_id_to_position:
                continue

            if pd.isnull(code):
                continue

            code = str(code)

            if code not in code_to_label:
                code_to_label[code] = len(code_to_label)

            labels_one_day[
                total_id_to_position[stock_id]
            ] = code_to_label[code]

        industry_labels[
            start_position:end_position
        ] = labels_one_day

    if not code_to_label:
        raise RuntimeError("未获得行业分类数据，请检查行业数据权限。")

    # --------------------------------------------------------
    # 计算每日各行业 indus_abnd
    # --------------------------------------------------------
    return_array = returns.values.astype(float)
    n_industries = len(code_to_label)

    industry_attention = np.full(
        (n_dates, n_industries),
        np.nan,
        dtype=float,
    )

    for t in range(n_dates):
        daily_returns = return_array[t]
        daily_industry = industry_labels[t]

        active = (
            np.isfinite(daily_returns)
            & (daily_industry >= 0)
        )

        for industry_id in np.unique(daily_industry[active]):
            member_mask = (
                active
                & (daily_industry == industry_id)
            )

            n_active = member_mask.sum()

            if n_active < min_industry_stocks:
                continue

            n_abnormal = (
                np.abs(daily_returns[member_mask])
                >= abnormal_threshold
            ).sum()

            industry_attention[t, industry_id] = (
                float(n_abnormal) / n_active
            )

    # --------------------------------------------------------
    # 对每个行业内的候选股票做滚动回归
    # r_i = alpha + beta * indus_abnd + rho * FF4 + error
    # --------------------------------------------------------
    candidate_positions = np.array(
        [total_id_to_position[stock_id] for stock_id in candidate_ids],
        dtype=int,
    )

    target_positions = [
        return_dates.get_loc(date)
        for date in target_dates
        if date in return_dates
    ]

    output = np.full(
        (len(target_positions), len(candidate_ids)),
        np.nan,
        dtype=float,
    )

    control_array = controls.values.astype(float)

    for output_row, t in enumerate(target_positions):
        left = t - regression_window + 1

        if left < 0:
            continue

        factor_window = control_array[left:t + 1]

        if not np.isfinite(factor_window).all():
            continue

        current_industry = industry_labels[t, candidate_positions]

        valid_industries = np.unique(
            current_industry[current_industry >= 0]
        )

        for industry_id in valid_industries:
            candidate_mask = (
                current_industry == industry_id
            )

            local_candidate_columns = np.where(
                candidate_mask
            )[0]

            stock_positions = candidate_positions[
                local_candidate_columns
            ]

            attention_window = industry_attention[
                left:t + 1,
                industry_id,
            ]

            if not np.isfinite(attention_window).all():
                continue

            x = np.column_stack(
                [
                    np.ones(regression_window),
                    attention_window,
                    factor_window,
                ]
            )

            if np.linalg.matrix_rank(x) < x.shape[1]:
                continue

            y = return_array[
                left:t + 1,
                :
            ][:, stock_positions]

            valid_stock_columns = np.isfinite(y).all(axis=0)

            if valid_stock_columns.sum() == 0:
                continue

            coefficients = np.linalg.lstsq(
                x,
                y[:, valid_stock_columns],
                rcond=None,
            )[0]

            # 第 2 行（位置 1）即 indus_abnd 的 beta
            output[
                output_row,
                local_candidate_columns[valid_stock_columns],
            ] = np.abs(coefficients[1])

    factor_wide = pd.DataFrame(
        output,
        index=return_dates[target_positions],
        columns=candidate_ids,
    )

    factor_wide.index.name = "date"
    factor = factor_wide.stack(dropna=False).to_frame(
        "INDUS_ABND_BETA"
    )
    factor.index.names = ["date", "order_book_id"]

    if return_controls:
        return factor, controls.reindex(factor_wide.index)

    return factor

In [25]:
# 计算量很大

indus_abnd_beta, ff4_controls = calculate_indus_abnd_attention_daily(
    start_date="2020-01-01",
    end_date="2025-12-31",
    candidate_ids=None,
    universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    abnormal_threshold=0.07,
    regression_window=20,
    min_industry_stocks=5,
    return_controls=True,
)

indus_abnd_beta.head()

INDUS_ABND_BETA
date       order_book_id                 
2020-01-02 000001.XSHE           0.057510
           000002.XSHE           0.659086
           000003.XSHE                NaN
           000004.XSHE           0.018611
           000005.XSHE           0.410603

In [26]:
# 检查四个控制变量是否已正常生成：
ff4_controls.head()

,MKT,SMB,HML,UMD
date,,,,
2020-01-02,0.014599,0.005582,-0.000562,-0.002880
2020-01-03,0.000879,0.007427,0.003047,-0.008041
2020-01-06,0.001076,0.007051,-0.000408,-0.010382
2020-01-07,0.008911,0.005863,-0.003441,0.003081
2020-01-08,-0.012180,-0.001107,-0.001000,-0.005001


In [19]:
# 建议你先用两年区间验证权限和字段：
test_factor, test_controls = calculate_indus_abnd_attention_daily(
    start_date="2023-01-01",
    end_date="2024-12-31",
    candidate_ids=None,
    universe_ids=None,
    return_controls=True,
)

In [20]:
test_controls.isna().mean()

MKT    0.0
SMB    0.0
HML    0.0
UMD    0.0
dtype: float64

In [21]:
test_controls

,MKT,SMB,HML,UMD
date,,,,
2023-01-03,0.012197,0.015413,-0.001356,0.003931
2023-01-04,0.002130,0.005411,0.010865,0.004113
2023-01-05,0.011771,-0.010137,-0.014342,-0.017173
2023-01-06,0.000905,-0.003968,-0.008790,-0.003969
2023-01-09,0.005853,-0.000118,-0.003523,-0.005009
...,...,...,...,...
2024-12-25,-0.003323,-0.025835,0.006177,0.017225
2024-12-26,0.002926,0.013655,-0.010305,-0.008924
2024-12-27,0.001009,0.008933,0.005765,-0.002413


In [22]:
test_factor["INDUS_ABND_BETA"].notna().mean()

0.8942263647481747

In [23]:
(
    test_factor["INDUS_ABND_BETA"]
    .notna()
    .groupby(level="date")
    .sum()
    .describe()
)

count     484.000000
mean     4985.311983
std        82.469442
min      4802.000000
25%      4909.750000
50%      5006.000000
75%      5056.250000
max      5098.000000
Name: INDUS_ABND_BETA, dtype: float64

# 🚨基于市场波动率的注意力捕捉

**行为金融因子 —— 投资者注意力**

对每个行业，分别统计每日行业内成分股收益率的横截面标准差，作为市场关注程度的代理指标：

$$
\text{indus_vol}_d = \text{std}\left( r_{i,d} \right), \quad i \in \text{行业内成分股}
$$

其中，$r_{i,d}$ 为行业成分股 $i$ 在交易日 $d$ 的收益率，$\text{indus_vol}_d$ 为行业内所有成分股当日收益率的横截面标准差，用于衡量该行业当日的收益分歧度和交易活跃程度。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{indus_vol}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受行业波动率变化影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{indus_vol}_d$ 为交易日 $d$ 行业内成分股收益率的横截面标准差；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而获取超额收益。

上述因子以行业内收益率的横截面标准差（即行业波动率）作为市场关注度事件，其原因在于：行业内成分股收益的分化程度反映了市场对该行业内部不同股票的分歧和关注。当行业波动率较高时，表明该行业内部收益差异较大，市场交易活跃，关注度较高；反之，则表明行业关注度较低。因此，该指标能够有效捕捉市场注意力的变化，并用于解释个股收益的横截面差异。

数据：日度收盘价、行业分类数据、Fama‑French 三因子

In [27]:
from rqdatac import get_yield_curve


def _ffcn_get_daily_rf(dates, tenor="1M", day_count=365.0):
    """
    将中债 1M 国债年化收益率转换为日化无风险收益率。
    """
    query_start = pd.Timestamp(dates[0]) - pd.Timedelta(days=10)

    curve = get_yield_curve(
        start_date=query_start,
        end_date=dates[-1],
        tenor=tenor,
    )

    if curve is None or len(curve) == 0:
        raise RuntimeError(
            "未获取到收益率曲线数据，请检查 get_yield_curve 权限。"
        )

    if isinstance(curve, pd.Series):
        annual_rate = curve.copy()
    else:
        curve = curve.copy()

        if tenor in curve.columns:
            annual_rate = curve[tenor]
        elif curve.shape[1] == 1:
            annual_rate = curve.iloc[:, 0]
        else:
            raise ValueError(
                "收益率曲线中未找到 {} 列，当前列为：{}".format(
                    tenor, list(curve.columns)
                )
            )

    annual_rate.index = pd.to_datetime(annual_rate.index)
    annual_rate = pd.to_numeric(annual_rate, errors="coerce")
    annual_rate = annual_rate.groupby(level=0).last().sort_index()

    # 非交易日没有国债曲线时，沿用最近可得到的收益率
    annual_rate = annual_rate.reindex(dates, method="ffill")

    if annual_rate.isna().any():
        raise RuntimeError(
            "无风险收益率存在无法前向填充的缺失值，请扩大历史下载区间。"
        )

    if (annual_rate <= -1.0).any():
        raise ValueError("收益率曲线数值异常，请检查是否以小数形式返回。")

    daily_rf = np.expm1(
        np.log1p(annual_rate.values.astype(float)) / day_count
    )

    return pd.Series(
        daily_rf,
        index=dates,
        name="RF",
    )


def _ffcn_main_board_mask(order_book_ids, breakpoint_universe):
    """
    原始 Fama-French 使用 NYSE 股票提供断点。
    A 股版本以主板股票提供断点；但全部 A 股均可进入组合。
    """
    if breakpoint_universe == "all_a":
        return np.ones(len(order_book_ids), dtype=bool)

    if breakpoint_universe != "main_board":
        raise ValueError(
            'breakpoint_universe 仅支持 "main_board" 或 "all_a"。'
        )

    instruments = all_instruments("CS")

    if "board_type" not in instruments.columns:
        raise RuntimeError(
            "all_instruments 返回结果中没有 board_type，"
            "无法构造主板断点样本。"
        )

    board_type = (
        instruments
        .drop_duplicates("order_book_id")
        .set_index("order_book_id")["board_type"]
    )

    return (
        board_type
        .reindex(order_book_ids)
        .eq("MainBoard")
        .fillna(False)
        .values
        .astype(bool)
    )


def _ffcn_first_dates_by_july(dates):
    month_first_dates = _ac_first_dates_by_month(dates)
    return month_first_dates[month_first_dates.month == 7]


def _ffcn_build_controls(
    raw_returns,
    lag_market_cap,
    lag_book_to_market_lyr,
    rf,
    breakpoint_mask,
    min_market_stocks=100,
    min_portfolio_stocks=10,
):
    """
    构造严格 A 股本土化的 MKT_RF / SMB / HML / UMD。

    SMB、HML：
        每年 7 月形成并持有一年；
        主板股票仅用于提供市值中位数与 B/M 30%、70% 断点；
        全部满足条件的 A 股进入六个组合。

    UMD：
        每月形成；
        使用 t-12 至 t-2 月共 11 个完整自然月收益；
        前 30% - 后 30%，持有一个月。
    """
    dates = raw_returns.index
    returns_array = raw_returns.values.astype(float)
    cap_array = lag_market_cap.values.astype(float)
    btm_array = lag_book_to_market_lyr.values.astype(float)
    rf_array = rf.reindex(dates).values.astype(float)

    n_dates, n_stocks = returns_array.shape

    # --------------------------------------------------------
    # 一、SMB / HML：每年 7 月构造 2 × 3 组合
    # --------------------------------------------------------
    size_labels = np.full((n_dates, n_stocks), -1, dtype=np.int8)
    value_labels = np.full((n_dates, n_stocks), -1, dtype=np.int8)

    annual_dates = _ffcn_first_dates_by_july(dates)
    annual_positions = [dates.get_loc(date) for date in annual_dates]

    for seq, position in enumerate(annual_positions):
        next_position = (
            annual_positions[seq + 1]
            if seq + 1 < len(annual_positions)
            else n_dates
        )

        formation_cap = cap_array[position]
        formation_btm = btm_array[position]

        eligible_all = (
            np.isfinite(formation_cap)
            & np.isfinite(formation_btm)
            & (formation_cap > 0)
            & (formation_btm > 0)
        )

        eligible_breakpoint = eligible_all & breakpoint_mask

        if eligible_breakpoint.sum() < max(min_market_stocks, 30):
            continue

        breakpoint_cap = formation_cap[eligible_breakpoint]
        breakpoint_btm = formation_btm[eligible_breakpoint]

        size_breakpoint = np.median(breakpoint_cap)
        btm_low, btm_high = np.percentile(
            breakpoint_btm,
            [30.0, 70.0],
        )

        # 只对已经验证有效的股票进行比较，避免 NaN 警告
        valid_idx = np.flatnonzero(eligible_all)

        cap_valid = formation_cap[valid_idx]
        btm_valid = formation_btm[valid_idx]

        small_idx = valid_idx[cap_valid <= size_breakpoint]
        big_idx = valid_idx[cap_valid > size_breakpoint]

        low_idx = valid_idx[btm_valid <= btm_low]
        neutral_idx = valid_idx[
            (btm_valid > btm_low) & (btm_valid <= btm_high)
        ]
        high_idx = valid_idx[btm_valid > btm_high]

        size_labels[position:next_position, small_idx] = 0
        size_labels[position:next_position, big_idx] = 1

        value_labels[position:next_position, low_idx] = 0
        value_labels[position:next_position, neutral_idx] = 1
        value_labels[position:next_position, high_idx] = 2

    # --------------------------------------------------------
    # 二、UMD：每月形成，跳过最近一个完整自然月
    # --------------------------------------------------------
    momentum_labels = np.full((n_dates, n_stocks), -1, dtype=np.int8)

    month_periods = dates.to_period("M")
    monthly_dates = _ac_first_dates_by_month(dates)
    monthly_positions = [dates.get_loc(date) for date in monthly_dates]

    for seq, position in enumerate(monthly_positions):
        next_position = (
            monthly_positions[seq + 1]
            if seq + 1 < len(monthly_positions)
            else n_dates
        )

        current_month = dates[position].to_period("M")

        # 当前月为 t，则用 t-12 至 t-2 的 11 个完整月；
        # t-1 月被跳过，避免短期反转污染动量。
        begin_month = current_month - 12
        end_month = current_month - 2

        history_mask = (
            (month_periods >= begin_month)
            & (month_periods <= end_month)
        )

        history_returns = returns_array[history_mask]

        if history_returns.shape[0] < 150:
            continue

        valid_history = np.isfinite(history_returns).all(axis=0)

        cumulative_return = np.full(n_stocks, np.nan)
        cumulative_return[valid_history] = (
            np.prod(
                1.0 + history_returns[:, valid_history],
                axis=0,
            ) - 1.0
        )

        formation_cap = cap_array[position]

        eligible = (
            valid_history
            & np.isfinite(cumulative_return)
            & np.isfinite(formation_cap)
            & (formation_cap > 0)
        )

        if eligible.sum() < max(min_market_stocks, 30):
            continue

        eligible_idx = np.flatnonzero(eligible)
        valid_momentum = cumulative_return[eligible_idx]

        loser_cut, winner_cut = np.percentile(
            valid_momentum,
            [30.0, 70.0],
        )

        loser_idx = eligible_idx[valid_momentum <= loser_cut]
        winner_idx = eligible_idx[valid_momentum >= winner_cut]

        momentum_labels[position:next_position, loser_idx] = 0
        momentum_labels[position:next_position, winner_idx] = 1

    # --------------------------------------------------------
    # 三、计算日频组合收益
    # --------------------------------------------------------
    factor_values = np.full((n_dates, 4), np.nan)

    for t in range(n_dates):
        daily_return = returns_array[t]
        daily_cap = cap_array[t]

        market_mask = (
            np.isfinite(daily_return)
            & np.isfinite(daily_cap)
            & (daily_cap > 0)
        )

        market_return = _ac_value_weighted_return(
            returns=daily_return,
            weights=daily_cap,
            mask=market_mask,
            min_stocks=min_market_stocks,
        )

        if np.isfinite(market_return) and np.isfinite(rf_array[t]):
            factor_values[t, 0] = market_return - rf_array[t]

        six_portfolios = {}

        for size in (0, 1):
            for value in (0, 1, 2):
                portfolio_mask = (
                    (size_labels[t] == size)
                    & (value_labels[t] == value)
                )

                six_portfolios[(size, value)] = (
                    _ac_value_weighted_return(
                        returns=daily_return,
                        weights=daily_cap,
                        mask=portfolio_mask,
                        min_stocks=min_portfolio_stocks,
                    )
                )

        sl = six_portfolios[(0, 0)]
        sn = six_portfolios[(0, 1)]
        sh = six_portfolios[(0, 2)]

        bl = six_portfolios[(1, 0)]
        bn = six_portfolios[(1, 1)]
        bh = six_portfolios[(1, 2)]

        if np.isfinite([sl, sn, sh, bl, bn, bh]).all():
            factor_values[t, 1] = (
                (sl + sn + sh) / 3.0
                - (bl + bn + bh) / 3.0
            )

            factor_values[t, 2] = (
                (sh + bh) / 2.0
                - (sl + bl) / 2.0
            )

        winner_return = _ac_value_weighted_return(
            returns=daily_return,
            weights=daily_cap,
            mask=(momentum_labels[t] == 1),
            min_stocks=min_portfolio_stocks,
        )

        loser_return = _ac_value_weighted_return(
            returns=daily_return,
            weights=daily_cap,
            mask=(momentum_labels[t] == 0),
            min_stocks=min_portfolio_stocks,
        )

        if np.isfinite(winner_return) and np.isfinite(loser_return):
            factor_values[t, 3] = winner_return - loser_return

    controls = pd.DataFrame(
        factor_values,
        index=dates,
        columns=["MKT_RF", "SMB", "HML", "UMD"],
    )

    controls["RF"] = rf.reindex(dates)

    return controls


def _ffcn_build_industry_labels(
    order_book_ids,
    dates,
    industry_source,
    industry_level,
    industry_refresh,
    industry_batch_size,
):
    """生成 日期 × 股票 的行业标签矩阵。"""
    n_dates = len(dates)
    n_stocks = len(order_book_ids)

    if industry_refresh == "M":
        update_dates = _ac_first_dates_by_month(dates)
    elif industry_refresh == "D":
        update_dates = dates
    else:
        raise ValueError('industry_refresh 仅支持 "M" 或 "D"。')

    update_positions = [dates.get_loc(date) for date in update_dates]

    labels = np.full(
        (n_dates, n_stocks),
        -1,
        dtype=np.int16,
    )

    id_to_position = {
        stock_id: position
        for position, stock_id in enumerate(order_book_ids)
    }

    code_to_label = {}

    for seq, update_date in enumerate(update_dates):
        start_position = update_positions[seq]
        end_position = (
            update_positions[seq + 1]
            if seq + 1 < len(update_positions)
            else n_dates
        )

        industry_codes = _ac_get_industry_codes(
            order_book_ids=order_book_ids,
            date=update_date,
            industry_source=industry_source,
            industry_level=industry_level,
            batch_size=industry_batch_size,
        )

        one_day_labels = np.full(
            n_stocks,
            -1,
            dtype=np.int16,
        )

        for stock_id, industry_code in industry_codes.items():
            if stock_id not in id_to_position:
                continue

            if pd.isnull(industry_code):
                continue

            industry_code = str(industry_code)

            if industry_code not in code_to_label:
                code_to_label[industry_code] = len(code_to_label)

            one_day_labels[
                id_to_position[stock_id]
            ] = code_to_label[industry_code]

        labels[start_position:end_position] = one_day_labels

    return labels, code_to_label


def calculate_indus_vol_attention_daily(
    start_date,
    end_date,
    candidate_ids=None,
    universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    regression_window=20,
    min_industry_stocks=5,
    industry_vol_ddof=1,
    min_market_stocks=100,
    min_portfolio_stocks=10,
    breakpoint_universe="main_board",
    initial_lookback_calendar_days=1100,
    price_batch_size=500,
    industry_batch_size=2000,
    return_controls=False,
):
    """
    基于市场波动率的注意力捕捉。

    回归：
    (r_i,d - RF_d)
      = alpha
      + beta * indus_vol_d
      + rho1 * MKT_RF_d
      + rho2 * SMB_d
      + rho3 * HML_d
      + rho4 * UMD_d
      + epsilon

    返回：
    两层索引 DataFrame：
    date, order_book_id

    列：
    INDUS_VOL_BETA
    """
    if regression_window <= 6:
        raise ValueError(
            "regression_window 必须大于 6。"
        )

    if industry_vol_ddof < 0:
        raise ValueError("industry_vol_ddof 必须非负。")

    instruments = all_instruments("CS")
    all_stock_ids = list(
        instruments["order_book_id"].dropna().unique()
    )

    if universe_ids is None:
        universe_ids = all_stock_ids
    else:
        universe_ids = list(
            pd.Index(universe_ids).dropna().unique()
        )

    if candidate_ids is None:
        candidate_ids = universe_ids
    else:
        candidate_ids = list(
            pd.Index(candidate_ids).dropna().unique()
        )

    total_ids = list(
        dict.fromkeys(universe_ids + candidate_ids)
    )

    target_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(start_date, end_date)
        )
    )

    if len(target_dates) == 0:
        raise ValueError("目标区间内没有交易日。")

    history_start = (
        target_dates[0]
        - pd.Timedelta(days=initial_lookback_calendar_days)
    )

    data_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(history_start, end_date)
        )
    )

    # 1. 日收益：仅当日成交量大于零时才视为活跃股票
    price_data = _ac_get_price_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        fields=["close", "volume"],
        batch_size=price_batch_size,
    )

    close = price_data["close"].astype(float)
    volume = price_data["volume"].astype(float)

    raw_returns = close.pct_change(fill_method=None).iloc[1:]
    raw_returns = raw_returns.where(volume.iloc[1:].gt(0))

    return_dates = raw_returns.index

    # 2. 所有组合权重均滞后一日，避免使用当日收盘后的市值
    market_cap = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="market_cap_3",
        batch_size=price_batch_size,
    )

    # LYR = 最近年度归母权益 / 总市值
    # 在每年 7 月形成时使用前一交易日数据。
    book_to_market_lyr = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="book_to_market_ratio_lyr",
        batch_size=price_batch_size,
    )

    lag_market_cap = market_cap.shift(1).reindex(return_dates)
    lag_book_to_market_lyr = (
        book_to_market_lyr.shift(1).reindex(return_dates)
    )

    # 3. 无风险利率与严格 FF / UMD 控制变量
    rf = _ffcn_get_daily_rf(return_dates, tenor="1M")

    breakpoint_mask = _ffcn_main_board_mask(
        total_ids,
        breakpoint_universe=breakpoint_universe,
    )

    controls = _ffcn_build_controls(
        raw_returns=raw_returns,
        lag_market_cap=lag_market_cap,
        lag_book_to_market_lyr=lag_book_to_market_lyr,
        rf=rf,
        breakpoint_mask=breakpoint_mask,
        min_market_stocks=min_market_stocks,
        min_portfolio_stocks=min_portfolio_stocks,
    )

    # 回归左侧使用超额收益
    excess_returns = raw_returns.sub(rf, axis=0)

    # 4. 行业标签
    industry_labels, code_to_label = _ffcn_build_industry_labels(
        order_book_ids=total_ids,
        dates=return_dates,
        industry_source=industry_source,
        industry_level=industry_level,
        industry_refresh=industry_refresh,
        industry_batch_size=industry_batch_size,
    )

    if not code_to_label:
        raise RuntimeError(
            "未获得行业数据，请检查行业分类接口权限。"
        )

    # 5. 每日行业横截面收益标准差 indus_vol
    raw_return_array = raw_returns.values.astype(float)
    n_dates = len(return_dates)
    n_industries = len(code_to_label)

    industry_vol = np.full(
        (n_dates, n_industries),
        np.nan,
        dtype=float,
    )

    minimum_required = max(
        min_industry_stocks,
        industry_vol_ddof + 1,
    )

    for t in range(n_dates):
        daily_returns = raw_return_array[t]
        daily_labels = industry_labels[t]

        active = (
            np.isfinite(daily_returns)
            & (daily_labels >= 0)
        )

        for industry_id in np.unique(daily_labels[active]):
            member_mask = (
                active
                & (daily_labels == industry_id)
            )

            values = daily_returns[member_mask]

            if len(values) < minimum_required:
                continue

            industry_vol[t, industry_id] = np.std(
                values,
                ddof=industry_vol_ddof,
            )

    # 6. 按行业分组做 20 日滚动回归
    excess_return_array = excess_returns.values.astype(float)

    control_array = controls[
        ["MKT_RF", "SMB", "HML", "UMD"]
    ].values.astype(float)

    id_to_position = {
        stock_id: position
        for position, stock_id in enumerate(total_ids)
    }

    candidate_positions = np.array(
        [id_to_position[stock_id] for stock_id in candidate_ids],
        dtype=int,
    )

    target_positions = [
        return_dates.get_loc(date)
        for date in target_dates
        if date in return_dates
    ]

    output = np.full(
        (len(target_positions), len(candidate_ids)),
        np.nan,
        dtype=float,
    )

    for output_row, t in enumerate(target_positions):
        left = t - regression_window + 1

        if left < 0:
            continue

        control_window = control_array[left:t + 1]

        if not np.isfinite(control_window).all():
            continue

        current_industry = industry_labels[
            t,
            candidate_positions,
        ]

        for industry_id in np.unique(
            current_industry[current_industry >= 0]
        ):
            local_columns = np.where(
                current_industry == industry_id
            )[0]

            stock_positions = candidate_positions[
                local_columns
            ]

            vol_window = industry_vol[
                left:t + 1,
                industry_id,
            ]

            if not np.isfinite(vol_window).all():
                continue

            x = np.column_stack(
                [
                    np.ones(regression_window),
                    vol_window,
                    control_window,
                ]
            )

            if np.linalg.matrix_rank(x) < x.shape[1]:
                continue

            y = excess_return_array[
                left:t + 1,
                :
            ][:, stock_positions]

            valid_stocks = np.isfinite(y).all(axis=0)

            if valid_stocks.sum() == 0:
                continue

            coefficients = np.linalg.lstsq(
                x,
                y[:, valid_stocks],
                rcond=None,
            )[0]

            # 第 2 行即行业波动率 indus_vol 的 beta
            output[
                output_row,
                local_columns[valid_stocks],
            ] = np.abs(coefficients[1])

    factor_wide = pd.DataFrame(
        output,
        index=return_dates[target_positions],
        columns=candidate_ids,
    )

    factor_wide.index.name = "date"

    factor = factor_wide.stack(dropna=False).to_frame(
        "INDUS_VOL_BETA"
    )
    factor.index.names = ["date", "order_book_id"]

    if return_controls:
        return factor, controls.reindex(factor_wide.index)

    return factor

In [28]:
indus_vol_beta, strict_controls = calculate_indus_vol_attention_daily(
    start_date="2023-01-01",
    end_date="2024-12-31",
    candidate_ids=None,
    universe_ids=None,
    industry_source="citics_2019",
    industry_level=1,
    industry_refresh="M",
    regression_window=20,
    min_industry_stocks=5,
    breakpoint_universe="main_board",
    return_controls=True,
)

indus_vol_beta.head()

INDUS_VOL_BETA
date       order_book_id                
2023-01-03 000001.XSHE          1.456297
           000002.XSHE          3.429513
           000003.XSHE               NaN
           000004.XSHE          0.923526
           000005.XSHE          0.864280

In [29]:
strict_controls.isna().mean()

MKT_RF    0.0
SMB       0.0
HML       0.0
UMD       0.0
RF        0.0
dtype: float64

In [30]:
indus_vol_beta["INDUS_VOL_BETA"].notna().mean()

0.8987610717859393

In [31]:
(
    indus_vol_beta["INDUS_VOL_BETA"]
    .notna()
    .groupby(level="date")
    .sum()
    .describe()
)

count     484.000000
mean     5010.592975
std        79.175079
min      4822.000000
25%      4932.250000
50%      5051.000000
75%      5068.000000
max      5098.000000
Name: INDUS_VOL_BETA, dtype: float64

# 🚨基于沪深总成交额的注意力捕捉

**行为金融因子 —— 投资者注意力**

首先，统计每日沪深两市的总成交额，作为市场关注程度的代理指标：

$$
\text{mkt_vol}_d = \text{Vol}_{\text{SH},d} + \text{Vol}_{\text{SZ},d}
$$

其中，$\text{Vol}_{\text{SH},d}$ 为上海证券交易所所有股票在交易日 $d$ 的总成交额，$\text{Vol}_{\text{SZ},d}$ 为深圳证券交易所所有股票在交易日 $d$ 的总成交额。

进一步，将上述市场关注代理指标与个股日收益进行月度时序回归，回归方程如下：

$$
r_{i,d} = \mu_{i,t} + \beta_{i,t} \cdot \text{mkt_vol}_d + \rho_{1,i,t} \text{Mkt}_d + \rho_{2,i,t} \text{SMB}_d + \rho_{3,i,t} \text{HML}_d + \rho_{4,i,t} \text{UMD}_d + \varepsilon_{i,t}
$$

其中，系数 $\beta_{i,t}$ 的绝对值 $|\beta_{i,t}|$ 即为股票 $i$ 对市场关注程度的敏感度，用于衡量该股票受全市场成交额变化影响的大小。

各变量定义如下：

* $r_{i,d}$ 为股票 $i$ 在交易日 $d$ 的收益率；
* $\text{mkt_vol}_d$ 为交易日 $d$ 的沪深两市总成交额；
* $\text{Mkt}_d$、$\text{SMB}_d$、$\text{HML}_d$ 分别为经典的 Fama‑French 三因子在 $d$ 日的收益；
* $\text{UMD}_d$ 为动量因子收益，即全市场过去 11 个月累积收益率排名前 30% 与后 30% 的股票组合在 $d$ 日的收益差；
* $\mu_{i,t}$ 为截距项，$\varepsilon_{i,t}$ 为残差项。

## 因子逻辑说明

不同股票对同一市场关注度事件的反映程度往往不同，可通过个股收益对市场关注度事件的敏感程度（即 $|\beta_{i,t}|$）来衡量股票对该事件的注意力捕捉效应。捕捉能力越强（敏感程度越高），表明该股票更能吸引市场注意，从而导致其股价波动性增大，买盘压力增加，股价被高估。

上述因子以沪深两市总成交额作为市场关注度事件，其原因在于：市场总成交额是反映全市场交易活跃度和投资者参与热情的最直接指标之一。成交额越高，说明市场整体关注度越高，资金流动越活跃，处于该市场环境下的个股受到的关注程度也随之提升。

数据：股票分类数据（上海交易所、深圳交易所）、日度成交额、日度收盘价、Fama‑French 三因子

In [32]:
def calculate_mkt_turnover_attention_daily(
    start_date,
    end_date,
    candidate_ids=None,
    universe_ids=None,
    regression_window=20,
    turnover_scale=1e8,
    min_market_stocks=100,
    min_portfolio_stocks=10,
    breakpoint_universe="main_board",
    initial_lookback_calendar_days=1100,
    price_batch_size=500,
    return_details=False,
):
    """
    基于沪深总成交额的注意力捕捉。

    回归：
    (r_i,d - RF_d)
      = alpha
      + beta * mkt_turnover_d
      + rho1 * MKT_RF_d
      + rho2 * SMB_d
      + rho3 * HML_d
      + rho4 * UMD_d
      + epsilon

    参数
    ----
    turnover_scale : float
        成交额缩放单位。默认 1e8，即以“亿元”为单位回归。

    return_details : bool
        True 时返回：
        factor, controls, market_turnover
    """
    if regression_window <= 6:
        raise ValueError(
            "regression_window 必须大于 6。"
        )

    if turnover_scale <= 0:
        raise ValueError("turnover_scale 必须为正数。")

    instruments = all_instruments("CS")
    all_stock_ids = list(
        instruments["order_book_id"].dropna().unique()
    )

    if universe_ids is None:
        universe_ids = all_stock_ids
    else:
        universe_ids = list(
            pd.Index(universe_ids).dropna().unique()
        )

    if candidate_ids is None:
        candidate_ids = universe_ids
    else:
        candidate_ids = list(
            pd.Index(candidate_ids).dropna().unique()
        )

    # 待输出股票必须也在市场数据下载范围内
    total_ids = list(
        dict.fromkeys(universe_ids + candidate_ids)
    )

    target_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(start_date, end_date)
        )
    )

    if len(target_dates) == 0:
        raise ValueError("目标区间内没有交易日。")

    history_start = (
        target_dates[0]
        - pd.Timedelta(days=initial_lookback_calendar_days)
    )

    data_dates = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(history_start, end_date)
        )
    )

    # --------------------------------------------------------
    # 1. 下载收益、成交额
    # --------------------------------------------------------
    price_data = _ac_get_price_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        fields=["close", "volume", "total_turnover"],
        batch_size=price_batch_size,
    )

    close = price_data["close"].astype(float)
    volume = price_data["volume"].astype(float)
    total_turnover = price_data["total_turnover"].astype(float)

    raw_returns = close.pct_change(fill_method=None).iloc[1:]
    raw_returns = raw_returns.where(volume.iloc[1:].gt(0))

    return_dates = raw_returns.index
    total_turnover = total_turnover.reindex(return_dates)

    # --------------------------------------------------------
    # 2. 严格计算沪市 + 深市总成交额
    # --------------------------------------------------------
    exchange = (
        instruments
        .drop_duplicates("order_book_id")
        .set_index("order_book_id")["exchange"]
        .reindex(total_ids)
    )

    sh_ids = exchange[exchange == "XSHG"].index.tolist()
    sz_ids = exchange[exchange == "XSHE"].index.tolist()

    if not sh_ids or not sz_ids:
        raise RuntimeError(
            "未识别到完整的沪市或深市股票列表，请检查股票池。"
        )

    sh_turnover = total_turnover[sh_ids].sum(
        axis=1,
        min_count=min_market_stocks,
    )

    sz_turnover = total_turnover[sz_ids].sum(
        axis=1,
        min_count=min_market_stocks,
    )

    # 原始市场总成交额，单位：元
    mkt_turnover_yuan = sh_turnover + sz_turnover

    # 回归用市场注意力变量，默认单位：亿元
    mkt_turnover = mkt_turnover_yuan / float(turnover_scale)

    # --------------------------------------------------------
    # 3. 构造严格 A 股版 FF / UMD 控制变量
    # --------------------------------------------------------
    market_cap = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="market_cap_3",
        batch_size=price_batch_size,
    )

    book_to_market_lyr = _ac_get_factor_wide(
        order_book_ids=total_ids,
        dates=data_dates,
        factor_name="book_to_market_ratio_lyr",
        batch_size=price_batch_size,
    )

    lag_market_cap = market_cap.shift(1).reindex(return_dates)
    lag_book_to_market_lyr = (
        book_to_market_lyr.shift(1).reindex(return_dates)
    )

    rf = _ffcn_get_daily_rf(return_dates, tenor="1M")

    breakpoint_mask = _ffcn_main_board_mask(
        order_book_ids=total_ids,
        breakpoint_universe=breakpoint_universe,
    )

    controls = _ffcn_build_controls(
        raw_returns=raw_returns,
        lag_market_cap=lag_market_cap,
        lag_book_to_market_lyr=lag_book_to_market_lyr,
        rf=rf,
        breakpoint_mask=breakpoint_mask,
        min_market_stocks=min_market_stocks,
        min_portfolio_stocks=min_portfolio_stocks,
    )

    excess_returns = raw_returns.sub(rf, axis=0)

    # --------------------------------------------------------
    # 4. 20 日滚动回归
    # --------------------------------------------------------
    id_to_position = {
        stock_id: position
        for position, stock_id in enumerate(total_ids)
    }

    candidate_positions = np.array(
        [id_to_position[stock_id] for stock_id in candidate_ids],
        dtype=int,
    )

    excess_return_array = excess_returns.values.astype(float)

    control_array = controls[
        ["MKT_RF", "SMB", "HML", "UMD"]
    ].values.astype(float)

    turnover_array = mkt_turnover.values.astype(float)

    target_positions = [
        return_dates.get_loc(date)
        for date in target_dates
        if date in return_dates
    ]

    output = np.full(
        (len(target_positions), len(candidate_ids)),
        np.nan,
        dtype=float,
    )

    for output_row, t in enumerate(target_positions):
        left = t - regression_window + 1

        if left < 0:
            continue

        turnover_window = turnover_array[left:t + 1]
        control_window = control_array[left:t + 1]

        if (
            (not np.isfinite(turnover_window).all())
            or (not np.isfinite(control_window).all())
        ):
            continue

        x = np.column_stack(
            [
                np.ones(regression_window),
                turnover_window,
                control_window,
            ]
        )

        if np.linalg.matrix_rank(x) < x.shape[1]:
            continue

        y = excess_return_array[
            left:t + 1,
            :
        ][:, candidate_positions]

        valid_stocks = np.isfinite(y).all(axis=0)

        if valid_stocks.sum() == 0:
            continue

        coefficients = np.linalg.lstsq(
            x,
            y[:, valid_stocks],
            rcond=None,
        )[0]

        # 第 2 行，即总成交额变量的 beta
        output[
            output_row,
            valid_stocks,
        ] = np.abs(coefficients[1])

    factor_wide = pd.DataFrame(
        output,
        index=return_dates[target_positions],
        columns=candidate_ids,
    )

    factor_wide.index.name = "date"

    factor = factor_wide.stack(dropna=False).to_frame(
        "MKT_TURNOVER_BETA"
    )
    factor.index.names = ["date", "order_book_id"]

    if return_details:
        market_turnover = pd.DataFrame(
            {
                "SH_TURNOVER_YUAN": sh_turnover,
                "SZ_TURNOVER_YUAN": sz_turnover,
                "MKT_TURNOVER_YUAN": mkt_turnover_yuan,
                "MKT_TURNOVER_100M": mkt_turnover,
            }
        )

        return (
            factor,
            controls.reindex(factor_wide.index),
            market_turnover.reindex(factor_wide.index),
        )

    return factor

In [33]:
mkt_turnover_beta, strict_controls, market_turnover = (
    calculate_mkt_turnover_attention_daily(
        start_date="2023-01-01",
        end_date="2024-12-31",
        candidate_ids=None,
        universe_ids=None,
        regression_window=20,
        turnover_scale=1e8,
        breakpoint_universe="main_board",
        return_details=True,
    )
)

In [34]:
market_turnover.describe()

,SH_TURNOVER_YUAN,SZ_TURNOVER_YUAN,MKT_TURNOVER_YUAN,MKT_TURNOVER_100M
count,4.840000e+02,4.840000e+02,4.840000e+02,484.000000
mean,4.066439e+11,5.568888e+11,9.635327e+11,9635.327211
std,1.613706e+11,2.448940e+11,4.029743e+11,4029.743050
min,2.073874e+11,2.663209e+11,4.773864e+11,4773.864104
25%,3.134733e+11,4.193959e+11,7.361495e+11,7361.495165
50%,3.623645e+11,4.924441e+11,8.482339e+11,8482.338653
75%,4.414617e+11,5.923641e+11,1.024400e+12,10243.996711
max,1.509538e+12,1.943808e+12,3.453346e+12,34533.457115


In [35]:
strict_controls.isna().mean()

MKT_RF    0.0
SMB       0.0
HML       0.0
UMD       0.0
RF        0.0
dtype: float64

In [36]:
mkt_turnover_beta["MKT_TURNOVER_BETA"].notna().mean()

0.89876329540822